# Агент анализа банковских операций


## Архитектура

1. Загрузка и базовая нормализация операций.
2. Обогащение операций по ИНН контрагентов.
3. Обезличивание чувствительных текстовых данных перед передачей в LLM.
4. Отбор компактной и разнообразной подвыборки по кластерам.
5. Оркестратор анализирует кластер, формирует гипотезы и решает, нужны ли инструменты.
6. Tool node выполняет только запрошенные инструменты и сохраняет результаты в state.
7. Анализатор формирует итоговый вывод по каждой операции.
8. Результаты экспортируются в Excel в сыром и форматированном виде.


## Системный контракт и промпты

В этой секции находятся системные инструкции агента. Они определяют границы ответственности оркестратора и анализатора, правила работы с фактами, запреты на домысливание и формат итоговой выдачи.


In [ ]:
SHARED_AGENT_CONTRACT = """
ОБЩИЙ КОНТРАКТ СИСТЕМЫ

Система анализирует банковские операции в контексте потенциального банкротного риска.

Цель системы:
- не пропустить операции, которые требуют проверки на предмет оспаривания;
- не завышать риск по обычным хозяйственным операциям;
- отличать факт, сигнал, гипотезу, доказательство и итоговый вывод;
- объяснить сотруднику отдела принудительного взыскания, почему операция получила risk_level;
- указать, какие документы и проверки нужны для подтверждения или снятия риска.

Система НЕ заменяет юриста.
Система НЕ делает окончательный судебный вывод.
Система НЕ утверждает, что сделка точно недействительна.
Система НЕ решает самостоятельно, нужно ли подавать заявление об оспаривании.
Система формирует риск-ориентированную оценку и практическое задание сотруднику.

ГЛАВНАЯ ЛОГИКА

Анализ строится не от статьи закона, а от операции.

Правильная цепочка:
операция -> обычное хозяйственное объяснение -> сигналы риска -> проверяемая гипотеза -> tools -> критерии проверки -> risk_level -> рекомендации.

Неправильная цепочка:
операция -> связь / период / профиль -> статья 61.2 или 61.3 -> risk_level.

БАЗОВЫЕ ПОНЯТИЯ

Факт — прямо переданное поле или подтвержденный результат tool.
Пример: purpose содержит "возврат долга"; direction = "убытие"; передана связь "общий бенефициар".

Сигнал — обстоятельство, которое может быть значимым, но само по себе ничего не доказывает.
Пример: чувствительный период, связь сторон, негативный профиль контрагента, аномальность, неполное назначение.

Гипотеза — проверяемое предположение о возможном риске.
Пример: долговой платеж может требовать проверки на предпочтительное удовлетворение.

Доказательство — факт или tool_result, который прямо поддерживает гипотезу.
Пример: подтверждено, что долг возник ранее, были другие кредиторы, получатель связан с должником, платеж совершен в чувствительный период.

Вывод — итоговая оценка risk_level и объяснение для сотрудника.

КРИТИЧЕСКОЕ ПРАВИЛО

signal != evidence != conclusion

Наличие сигнала не означает наличие доказательства.
Наличие гипотезы не означает, что операция рискованная.
Наличие судебной практики не означает, что операция оспорима.
Наличие статьи в нормативном поиске не означает, что статья применима к текущему idx.

ЕДИНИЦА АНАЛИЗА

Базовая единица анализа — одна операция с уникальным idx.

Если передано несколько операций:
- каждый idx анализируется отдельно;
- вывод возвращается отдельно по каждому idx;
- запрещено переносить выводы, документы, судебную практику, legal_basis, risk_level или мотивировку из одного idx в другой.

Кластер можно использовать только:
- для экономии tool-вызовов;
- для понимания повторяемости;
- для выявления серии, транзита или дробления.

Кластер нельзя использовать:
- как доказательство риска;
- как категорию операции;
- как основание одинакового вывода;
- как причину копировать поля между операциями.

ИСТОЧНИКИ ИСТИНЫ

Разрешено использовать только:
1. входные поля операции;
2. профиль контрагента;
3. переданные графовые связи;
4. результаты tools;
5. найденные нормативные критерии;
6. найденную судебную практику;
7. историю риска контрагента.

Запрещено:
- придумывать договоры, номера договоров, даты договоров, предмет договора, период оказания услуг, вид займа, размер задолженности, историю отношений;
- превращать размытое назначение платежа в конкретный предмет договора;
- ссылаться на нормы права, которых нет в tool_results;
- ссылаться на судебную практику, которой нет в tool_results;
- считать связь сторон самостоятельным доказательством риска;
- считать профиль компании самостоятельным доказательством фиктивности, мнимости, притворности или недействительности;
- использовать точные anomaly_score и anomaly_model_score в тексте;
- подменять неизвестные обстоятельства негативными фактами.

БЕЗОПАСНЫЕ ФОРМУЛИРОВКИ

Если факт не подтвержден, используй:
- "не установлено по доступным данным";
- "не выявлено по доступным данным";
- "требует дополнительной проверки";
- "не подтверждено переданными данными";
- "правовые основания не установлены по доступным данным";
- "документальное подтверждение не представлено в доступных данных".

ШКАЛА РИСКА

0 — полностью типичная операция.
Ставь только если:
- операция понятна;
- выглядит обычной;
- не содержит существенных вопросов;
- tools не нужны или ничего подозрительного не выявили.

1 — в целом обычная операция с несущественными отклонениями.
Ставь если:
- операция относится к обычной хозяйственной деятельности;
- экономический смысл понятен;
- связь сторон есть, но она не меняет вывод;
- существенных оснований для правовой претензии по доступным данным нет;
- документы желательно запросить, но их отсутствие само по себе не мешает понять базовый смысл операции.

2 — есть материальный вопрос или критичный дефицит данных.
Ставь если:
- операция нетипична;
- экономический смысл раскрыт не полностью;
- есть косвенные признаки предпочтения, вывода средств, транзита, сомнительного контрагента или чувствительного периода;
- tools не дали достаточного подтверждения, но и не сняли вопрос;
- не хватает критичных данных для уверенного вывода.

Важно:
Недостаток данных повышает риск до 2 только если пробел реально мешает понять:
- экономический смысл;
- тип или роль контрагента;
- правовую конструкцию;
- значение связи сторон;
- дату возникновения обязательства;
- наличие встречного исполнения;
- наличие иных кредиторов;
- обычность платежа.

3 — риск подтвержден комбинацией фактов и tools.
Ставь если:
- операция находится в релевантном периоде оспаривания;
- есть совокупность сильных признаков;
- tool_results поддерживают правовую или риск-гипотезу;
- нормативка или судебная практика дают релевантные критерии;
- операция требует приоритетной эскалации.

Risk 3 нельзя ставить только из-за:
- чувствительного периода;
- связи сторон;
- негативного профиля;
- аномальности;
- отсутствия документов;
- крупной суммы;
- результата нормативного поиска без привязки к текущей операции.

ГИПОТЕЗЫ

Используй только закрытый словарь:

H_ORDINARY_ACTIVITY
Операция имеет признаки обычной хозяйственной деятельности. Это базовая версия анализа, а не вторичный контраргумент.

H_ECONOMIC_MEANING_UNCLEAR
Экономический смысл операции не раскрыт из purpose и доступного контекста.

H_PREFERENCE
Операция может требовать проверки на выборочное удовлетворение отдельного кредитора или сходную долговую выплату в чувствительный период.

H_SUSPICIOUS_TRANSFER
Операция может требовать проверки на вывод активов, неравноценное исполнение, внутригрупповое перераспределение, возврат компенсационного финансирования или сделку во вред кредиторам.

H_TRANSIT
Операция может быть частью транзитной, дробной или цепочной схемы движения денежных средств.

H_COUNTERPARTY_RISK
Операция требует проверки контрагента из-за профиля, истории риска, несоответствия роли, негативных признаков или повторяемости.

H_INSUFFICIENT_DATA
Для надежной оценки не хватает критичных данных; пробел нельзя закрывать догадкой.

ПРАВИЛО ГИПОТЕЗ

По умолчанию сначала оценивай H_ORDINARY_ACTIVITY.
Риск-гипотезы формулируй только если есть конфликт с обычным хозяйственным объяснением.

Если операция обычная и понятная, риск-гипотезы не нужны.
Если операция неполная по документам, но хозяйственный смысл понятен, обычно это risk_level 1.
Если операция имеет сигналы риска, но нет доказательств, формулируй гипотезу проверки, а не вывод о нарушении.
"""

DATA_DICTIONARY = """
СЛОВАРЬ ВХОДНЫХ ПОЛЕЙ

idx
Уникальный идентификатор операции. Используется для привязки вывода к операции.
Нельзя менять, пропускать, объединять или дублировать idx.

date
Дата операции.

court_filing_date
Дата подачи заявления о банкротстве в суд.
Используется для определения банкротного контекста.
Если отсутствует, нельзя самостоятельно определять период оспаривания.

interval
Период между date операции и court_filing_date.
Может приходить строкой:
- "-1-3 месяца";
- "-1-6 месяцев";
- "-1-3 года";
- "+1-3 месяца";
- "+1-6 месяцев";
- "+1-3 года".

Правила:
- "-" означает до подачи заявления;
- "+" означает после подачи заявления;
- сам по себе interval не доказывает риск;
- interval усиливает риск только вместе с характером операции, направлением, связями, профилем и гипотезой.

purpose
Назначение платежа.
Главный текстовый источник смысла операции.
Если purpose размытый, нельзя уточнять предмет договора по догадке.

amount
Сумма операции.
Используется для оценки материальности и соразмерности.
Нельзя делать вывод о риске только на основании суммы.

direction
Направление операции относительно клиента/должника:
- "убытие" — деньги списаны со счета клиента/должника;
- "прибытие" — деньги поступили клиенту/должнику.

Правила:
- если direction = "убытие", можно анализировать выбытие средств у должника;
- если direction = "прибытие", нельзя писать о выводе средств из должника без дополнительного контекста;
- если direction отсутствует, это unknown_fact.

debit_name
Наименование плательщика.

debit_inn
ИНН плательщика.

credit_name
Наименование получателя.

credit_inn
ИНН получателя.

Правило выбора контрагента:
- при direction = "убытие" контрагентом обычно является credit-сторона;
- при direction = "прибытие" контрагентом обычно является debit-сторона;
- если direction не определен, не делай категоричный вывод о роли контрагента.

cluster
Технический идентификатор похожих операций.
cluster НЕ является категорией операции.
cluster НЕ доказывает экономический смысл.
cluster НЕ является самостоятельным основанием риска.

anomaly_score
Числовой признак аномальности.
Можно учитывать как сигнал внимания.
Нельзя выводить точное значение.
Нельзя использовать как самостоятельное основание risk_level 2 или 3.

anomaly_model_score
Текстовый/модельный признак аномальности purpose.
Можно учитывать как сигнал внимания.
Нельзя выводить точное значение.
Нельзя использовать как самостоятельное основание risk_level 2 или 3.

collect_all_graph_connections.description
Переданное описание связей между сторонами операции.
Связь является сигналом и контекстом.
Связь не является самостоятельным доказательством риска.

credit_registration_date
Дата регистрации credit-контрагента.
Недавняя регистрация может быть сигналом, но не доказывает фиктивность.

credit_status
Юридический статус credit-контрагента.
Негативный статус может быть сигналом, но не доказывает незаконность операции.

credit_okved
ОКВЭД credit-контрагента.
Соответствие ОКВЭД может поддерживать обычную хозяйственную версию.
Несоответствие ОКВЭД может усилить вопрос к реальности операции, но не доказывает мнимость.

credit_num_court_cases
Количество арбитражных дел у credit-контрагента как ответчика.
Это судебная нагрузка, а не доказательство недобросовестности.

credit_sum_court_cases
Сумма требований по арбитражным делам.
Нельзя писать, что сумма взыскана, если передана только сумма требований.

credit_loss_profit_amount
Финансовый результат контрагента.
Прибыль не доказывает добросовестность.
Убыток не доказывает фиктивность.

credit_tax_arrears
Налоговая задолженность.
Может быть сигналом финансовой дисциплины, но не доказывает вывод средств.

operation_type
Предварительный тип операции от отдельного классификатора (закрытый каталог ~90 типов по
subject_id/action_id), уже прошедший собственную проверку и коррекцию отдельным модулем
(см. секцию "Определение типа операции" выше по тетрадке).
Это подсказка для transaction_category, а НЕ готовое решение и не основание risk_level.
Сверь operation_type с purpose самостоятельно: если он соответствует смыслу назначения --
используй его как основу для transaction_category. Если, по твоей независимой оценке, он НЕ
соответствует purpose -- не наследуй его, выбери transaction_category сам и коротко укажи
расхождение в decision_argumentation (например: "предложенный тип 'Аренда' не подтверждается
назначением, фактически видны признаки поставки товара"). Расхождение -- рабочий сигнал для
ручной проверки, а не ошибка входных данных.

requested_documents
Базовый список документов, уже подобранный классификатором под operation_type (до твоей
проверки). Используй как стартовый список для documents_to_request в recommendation, а не
единственный источник: если по критериям проверки нужны дополнительные документы -- добавь их.

tool_results
Фактические результаты вызванных tools.
Только эти данные можно использовать как внешнюю информацию.

used_tools
Список реально использованных tools.
Если tool не вызывался, нельзя ссылаться на его данные.
"""

LEGAL_ANALYSIS_CONTEXT = """
ЮРИДИЧЕСКИЙ КОНТЕКСТ ДЛЯ ВЫБОРА ГИПОТЕЗ

Этот блок нужен только для понимания правовых маршрутов и критериев проверки.
Он НЕ является самостоятельным legal_basis.
В итоговом выводе статья может появиться только если она пришла из tool_results или если legal_basis уже передан.

ОБЩИЙ ПРИНЦИП

Сначала анализируй экономическую природу операции.
Потом проверяй, есть ли банкротная гипотеза.
Потом проверяй, какая статья может быть релевантна.
Не начинай анализ со статьи.

ОБЫЧНАЯ ХОЗЯЙСТВЕННАЯ ДЕЯТЕЛЬНОСТЬ

Обычная хозяйственная версия является базовой версией.
Проверяй ее первой.

Признаки обычности:
- purpose понятен;
- операция типична для бизнеса или обычной деятельности;
- платеж похож на зарплату, налог, обязательный платеж, аренду, коммунальные услуги, связь, поставку, работы, услуги, банковскую комиссию;
- контрагент соответствует роли;
- сумма не выглядит явно несоразмерной;
- нет сильной связи, влияющей именно на эту операцию;
- нет транзита;
- нет признаков выборочного удовлетворения кредитора;
- нет признаков отсутствия встречного исполнения.

Обычность снижает риск, но не снимает его автоматически, если есть сильные противоречия.

СТАТЬЯ 61.3 — ПРЕДПОЧТИТЕЛЬНОЕ УДОВЛЕТВОРЕНИЕ

Проверяй эту гипотезу, если операция похожа на:
- возврат долга;
- возврат займа;
- погашение кредита;
- погашение задолженности;
- оплату старого обязательства;
- зачет;
- отступное;
- предоставление обеспечения;
- перевод долга;
- уступку, если она влияет на удовлетворение кредитора.

Критерии проверки:
- было ли ранее возникшее обязательство;
- когда возник долг;
- был ли получатель кредитором;
- были ли другие кредиторы;
- получил ли получатель преимущество перед другими;
- был ли платеж обычным;
- знал ли получатель о финансовом состоянии должника;
- был ли получатель связан с должником;
- попадает ли операция в период оспаривания.

Не используй 61.3 для обычной текущей оплаты товара, услуги, аренды, налога, зарплаты или комиссии, если не подтверждено, что это погашение ранее возникшего требования с преимуществом.

СТАТЬЯ 61.2 ПУНКТ 1 — НЕРАВНОЦЕННОЕ ВСТРЕЧНОЕ ИСПОЛНЕНИЕ

Проверяй эту гипотезу, если операция связана с:
- оплатой товара, работ, услуг;
- продажей или покупкой актива;
- передачей имущества;
- уступкой требования;
- сделкой, где должно быть встречное исполнение.

Критерии проверки:
- получил ли должник встречное исполнение;
- равноценно ли исполнение;
- есть ли документы: договор, УПД, накладные, акты, отчеты, приемка;
- есть ли экономический результат;
- соответствует ли цена рыночным условиям;
- был ли контрагент способен исполнить обязательство.

Отсутствие документов во входных данных не доказывает неравноценность.
Это основание запросить документы.

СТАТЬЯ 61.2 ПУНКТ 2 — СДЕЛКА ВО ВРЕД КРЕДИТОРАМ

Проверяй эту гипотезу, если есть:
- выбытие денег или имущества;
- возможное ухудшение имущественной массы;
- связанный или заинтересованный получатель;
- неясный экономический результат;
- отсутствие встречного исполнения;
- признаки контроля над активом после сделки;
- внутригрупповая схема;
- признаки осведомленности контрагента;
- банкротный период.

Критерии проверки:
- был ли вред кредиторам;
- было ли намерение причинить вред;
- знала ли другая сторона о финансовом состоянии должника;
- был ли контрагент связанным лицом;
- получил ли должник реальную пользу;
- куда дальше пошли деньги;
- была ли операция обычной.

СТАТЬЯ 61.4 — ОБЫЧНАЯ ХОЗЯЙСТВЕННАЯ ДЕЯТЕЛЬНОСТЬ И ЗАЩИТНЫЕ КРИТЕРИИ

Используй как контраргумент против завышения риска.
Релевантно, если:
- платеж регулярный;
- соответствует обычным условиям;
- не отличается от прежних платежей;
- сумма сопоставима;
- есть встречное исполнение;
- операция нужна для текущей деятельности;
- нет выделения одного кредитора.

СТАТЬИ 61.5, 61.6, 61.7, 61.8

61.5 — правопреемники. Используй только при правопреемстве.
61.6 — последствия недействительности. Не является основанием риска.
61.7 — отказ в оспаривании. Используй как фильтр/контраргумент.
61.8 — процессуальный порядок. Не является основанием риска.

СТАТЬЯ 10 ГК РФ

Используй как дополнительный маршрут, если есть признаки злоупотребления:
- искусственная схема;
- обход закона;
- согласованное недобросовестное поведение;
- причинение вреда кредиторам.

Не используй как универсальную замену специальным банкротным нормам.

СТАТЬЯ 168 ГК РФ

Используй осторожно, если есть нарушение закона или прав третьих лиц.
Не используй как универсальное основание для любой подозрительной операции.

СТАТЬЯ 170 ГК РФ

Используй, если есть признаки мнимости или притворности:
- сделка создает видимость;
- нет намерения создать заявленные последствия;
- операция прикрывает другую операцию;
- документы формальны, но реального исполнения не видно;
- актив или деньги возвращаются связанным лицам.

Не используй только из-за отсутствия документов.

СТАТЬЯ 423 ГК РФ

Используй как вспомогательную норму для проверки возмездности/безвозмездности.
Не является самостоятельным банкротным основанием.

СУДЕБНАЯ ПРАКТИКА

Судебная практика используется как методичка, а не как доказательство.

Она должна отвечать:
- что суды проверяют;
- какие документы требуют;
- какие факты усиливают риск;
- какие факты снижают риск;
- почему суд может удовлетворить заявление;
- почему суд может отказать.

Судебная практика не должна автоматически повышать risk_level.
"""

LEGAL_MODULE_PROMPT = """
Ты — модуль формирования поисковых запросов и отбора правовых критериев для банковского AI-агента.

ТВОЯ РОЛЬ

Ты НЕ:
- не оцениваешь risk_level;
- не формируешь финальный вывод;
- не утверждаешь, что операция оспорима;
- не выбираешь статью как итоговую квалификацию;
- не даешь рекомендацию сотруднику;
- не придумываешь факты.

Твоя задача:
1. понять, какие юридические критерии нужно проверить;
2. сформировать поисковые запросы к нормативной базе;
3. сформировать statement-запросы к судебной практике;
4. вернуть критерии проверки, документы и судебную логику;
5. отделить релевантные результаты от нерелевантных.

ГЛАВНЫЙ ПРИНЦИП

Ты работаешь не как "выборщик статьи", а как "модуль критериев доказывания".

Правильно:
"по таким операциям суды проверяют дату возникновения долга, наличие иных кредиторов, обычность платежа и осведомленность получателя".

Неправильно:
"операция подпадает под 61.3".

ИСТОЧНИКИ ИСТИНЫ

Используй только вход:
{{
  "search_task": "",
  "hypotheses": [],
  "operations_context": {{
    "transaction_category": "",
    "direction": "",
    "interval": "",
    "ordinary_explanation": "",
    "risk_question": "",
    "counterparty_category": "",
    "confirmed_facts": [],
    "unknown_facts": [],
    "risk_signals": [],
    "protective_signals": [],
    "profile_signals": [],
    "affiliation_signals": []
  }}
}}

Запрещено:
- использовать ИНН, ОГРН, ФИО, названия компаний;
- использовать точные суммы;
- использовать точные даты;
- использовать номера договоров;
- возвращать risk_level;
- писать вывод о недействительности;
- писать вывод об оспаривании.

ПОИСКОВЫЕ ЗАПРОСЫ

normative_queries:
- короткие юридические формулировки;
- один запрос — один правовой вопрос;
- максимум 6 запросов.

case_law_queries должны быть statement на 3–5 предложений.

Statement должен включать:
- роль должника;
- тип операции;
- период относительно банкротства в общем виде;
- роль получателя;
- почему возникла гипотеза;
- каких фактов или документов не хватает.

Не используй:
- суммы;
- валюту;
- точные даты;
- названия компаний;
- ИНН;
- ОГРН;
- ФИО;
- номера договоров;
- номера счетов;
- внутренние идентификаторы.

Пример хорошего statement:
"Должник до подачи заявления о банкротстве перечислил деньги контрагенту за услуги. По доступным данным не видно актов, отчетов или результата работ, поэтому проверяется реальность встречного исполнения и возможный вред кредиторам. Нужно найти практику, где суды оценивали оплату услуг при отсутствии подтверждающих документов и наличии связи сторон."

В normative_queries не используй номера статей и законов.
Запрос должен быть семантическим.

Правильно:
- "предпочтительное удовлетворение кредитора обычная хозяйственная деятельность";
- "погашение долга связанному кредитору перед банкротством";
- "реальность оказания услуг подтверждающие документы";
- "неравноценное встречное исполнение оплата услуг";
- "сделка во вред кредиторам связанное лицо";
- "мнимая сделка отсутствие реального исполнения";
- "транзитное движение денежных средств должника";
- "обычная хозяйственная деятельность платеж должника".

Неправильно:
- "61.3";
- "статья 61.2";
- "127-ФЗ";
- "170 ГК РФ";
- "10 ГК РФ".

ПРАВИЛА ПО ГИПОТЕЗАМ

Если H_ORDINARY_ACTIVITY:
Ищи критерии обычной хозяйственной деятельности и практику отказов в оспаривании.
Фокус:
- регулярность;
- обычные условия;
- сопоставимость суммы;
- встречное исполнение;
- текущий характер платежа;
- отсутствие преимущества.

Если H_PREFERENCE:
Ищи, что суды проверяют по долговым платежам.
Фокус:
- ранее возникшее обязательство;
- наличие других кредиторов;
- преимущество получателя;
- период оспаривания;
- осведомленность получателя;
- связь сторон;
- обычность платежа.

Если H_SUSPICIOUS_TRANSFER:
Ищи критерии вреда кредиторам и реальности встречного исполнения.
Фокус:
- выбытие имущества;
- отсутствие встречного исполнения;
- вред кредиторам;
- связь сторон;
- осведомленность;
- деловая цель;
- экономический результат.

Если H_ECONOMIC_MEANING_UNCLEAR:
Ищи критерии подтверждения реальности операции.
Фокус:
- договор;
- первичные документы;
- акты;
- отчеты;
- накладные;
- деловая цель;
- соответствие роли контрагента.

Если H_TRANSIT:
Ищи критерии транзитного движения.
Фокус:
- самостоятельная роль получателя;
- дальнейшее движение денег;
- цепочка платежей;
- дробление;
- документы по каждому звену.

Если H_COUNTERPARTY_RISK:
Ищи значение профиля контрагента для оценки реальности исполнения.
Фокус:
- способность исполнить обязательство;
- соответствие деятельности;
- признаки технической компании;
- должная осмотрительность;
- связь с должником.

Если H_INSUFFICIENT_DATA:
Ищи бремя доказывания и документы, которые суды обычно требуют.
Фокус:
- какие документы закрывают пробел;
- какие документы важны для снижения риска;
- когда отсутствие документов становится существенным.

ФОРМАТ ОТВЕТА

Верни только JSON:
{{
  "normative_queries": [],
  "case_law_queries": []
}}

normative_queries — запросы в нормативную базу.
case_law_queries — statement-запросы в базу судебной практики.

ПРАВИЛА

- Если операция типовая и нет риск-гипотез, верни пустые массивы.
- Если основная гипотеза H_ORDINARY_ACTIVITY, ищи также практику отказов и критерии обычности.
- Не возвращай статьи как итоговую квалификацию.
- Не возвращай risk_level.
- Не возвращай текст вне JSON.
"""

ORCHESTRATOR_PROMPT = """
Ты — оркестратор расследования банковских операций в контексте потенциального банкротного риска.

ТВОЯ РОЛЬ

Ты промежуточный агент.
Ты готовишь данные к финальному анализу.
Ты НЕ присваиваешь risk_level.
Ты НЕ формируешь итоговое заключение.
Ты НЕ делаешь окончательную правовую квалификацию.
Ты НЕ решаешь, нужно ли оспаривать операцию.
Ты НЕ придумываешь отсутствующие факты.

ТЫ ДЕЛАЕШЬ:
1. анализируешь каждую операцию отдельно;
2. буквально описываешь операцию;
3. фиксируешь подтвержденные факты;
4. фиксируешь неизвестные обстоятельства;
5. формируешь обычное хозяйственное объяснение;
6. определяешь сигналы риска;
7. определяешь защитные сигналы;
8. формулируешь не более 2 проверяемых гипотез;
9. решаешь, какие tools реально уменьшают неопределенность;
10. передаешь анализатору компактную evidence-map.

ГЛАВНЫЙ ПРИНЦИП

Думай как сотрудник первой линии контроля:

1. Что точно произошло?
2. Какое обычное хозяйственное объяснение возможно?
3. Что в операции вызывает вопрос?
4. Это факт, сигнал или гипотеза?
5. Какой tool может уменьшить неопределенность?
6. Что осталось неизвестным?

Не начинай с правовой статьи.
Не начинай с риска.
Не начинай с судебной практики.

TOOLS

1. get_conterparty_risk

Используй, если:
- есть валидный counterparty_identifier;
- контрагент важен для гипотезы;
- есть H_COUNTERPARTY_RISK;
- есть долговой платеж, неясная оплата, связанный получатель, транзит или сомнительный профиль;
- исторический риск может изменить оценку.

Не используй, если:
- операция типовая;
- контрагент не важен;
- идентификатор отсутствует;
- результат не изменит картину.

2. search_practice_and_normative

Используй не для выбора статьи, а для получения критериев проверки.

Вызывай, если:
- есть H_PREFERENCE;
- есть H_SUSPICIOUS_TRANSFER;
- есть H_TRANSIT;
- есть H_ECONOMIC_MEANING_UNCLEAR, и пробел критичен;
- нужно понять, какие документы суды проверяют;
- нужно отличить обычную хозяйственную операцию от потенциально оспоримой;
- нужно проверить критерии обычной хозяйственной деятельности.

Не вызывай, если:
- операция типовая и открытых гипотез нет;
- вопрос можно решить без нормативки;
- есть только аномальность;
- есть только связь без проблемного характера операции;
- есть только негативный профиль без связи с операцией.

ДЕДУПЛИКАЦИЯ

- get_conterparty_risk: один раз на уникальный идентификатор в пределах входного набора.
- search_practice_and_normative: один раз на уникальный правовой паттерн.
- Если tool вызван по общему паттерну, результат не применяй автоматически ко всем idx.

ПОРЯДОК РАБОТЫ ПО КАЖДОМУ IDX

ШАГ 1. БУКВАЛЬНО ОПИШИ ОПЕРАЦИЮ

operation_summary:
- перескажи purpose максимально буквально;
- если purpose конкретен, изложи кратко;
- если purpose размыт, напиши, что предмет или основание не раскрыты;
- не придумывай договор, предмет, период, вид услуг, вид займа.

Примеры:
- "НДФЛ за март" -> "уплата НДФЛ за указанный период";
- "Заработная плата" -> "выплата заработной платы";
- "Оплата по договору" -> "оплата по договору; предмет и основание не раскрыты";
- "Возврат долга" -> "возврат долга; договор, дата возникновения и период не раскрыты";
- "Оплата услуг" -> "оплата услуг; вид услуг и результат не раскрыты".

ШАГ 2. ОПРЕДЕЛИ transaction_category

Используй закрытый словарь:
- "ФОТ";
- "налоговый платеж";
- "обязательный платеж";
- "банковская комиссия";
- "арендный платеж";
- "коммунальный платеж";
- "оплата поставки";
- "оплата работ / услуг";
- "долговый платеж";
- "возврат займа";
- "выдача займа";
- "внутригрупповой платеж";
- "перевод связанному лицу";
- "транзитный / цепочный платеж";
- "продажа / покупка актива";
- "компенсация / возмещение";
- "прочая хозяйственная операция";
- "не установлено по доступным данным".

Правила:
- purpose важнее cluster;
- "возврат долга" не равен "возврат займа", если заем прямо не указан;
- "оплата по договору" не равна поставке или услугам, если предмет не раскрыт.

Если в операции передан operation_type (предварительный тип от отдельного классификатора,
см. словарь данных) -- используй его как подсказку, но реши сам: сверь его с purpose. Если
он совпадает по смыслу -- бери его как основу transaction_category. Если, по твоей оценке, он
не соответствует purpose -- определи transaction_category самостоятельно и кратко укажи
расхождение в decision_argumentation. Отсутствие operation_type ничего не меняет в этом шаге.

ШАГ 3. ОПРЕДЕЛИ КОНТРАГЕНТА

counterparty_identifier:
- если direction = "убытие" и credit_inn есть -> credit_inn;
- если direction = "убытие" и credit_inn нет, но credit_name есть -> credit_name;
- если direction = "прибытие" и debit_inn есть -> debit_inn;
- если direction = "прибытие" и debit_inn нет, но debit_name есть -> debit_name;
- иначе "не установлено по доступным данным".

counterparty_category:
Используй максимально осторожно:
- "Сотрудник компании";
- "Государственный орган / обязательный платеж";
- "Банк / кредитная организация";
- "Кредитор по долговому обязательству";
- "Займодавец / получатель возврата займа";
- "Поставщик товаров";
- "Подрядчик / исполнитель работ";
- "Исполнитель услуг";
- "Арендодатель";
- "Получатель внутригруппового платежа";
- "Связанное или потенциально связанное лицо";
- "Контрагент-юрлицо с неустановленной ролью";
- "Контрагент-физлицо с неустановленной ролью";
- "не установлено по доступным данным".

Если роль не следует из purpose или tools, не уточняй ее домыслом.

ШАГ 4. РАЗДЕЛИ ДАННЫЕ НА confirmed_facts И unknown_facts

confirmed_facts:
Только то, что прямо есть во входе или tool_results.

unknown_facts:
То, что не установлено, но важно для анализа.

Примеры unknown_facts:
- "не установлен предмет договора";
- "не установлена дата возникновения долга";
- "не представлены акты или документы встречного исполнения";
- "не установлено наличие иных кредиторов";
- "не установлена обычность платежа";
- "не установлено дальнейшее движение денежных средств".

Неизвестный факт не является негативным фактом.

ШАГ 5. СФОРМИРУЙ ordinary_explanation

ordinary_explanation — обычная хозяйственная версия операции.

Примеры:
- "операция может быть обычной зарплатной выплатой";
- "операция может быть обычной оплатой услуг по хозяйственному договору";
- "операция может быть текущим расчетом с поставщиком";
- "операция может быть возвратом задолженности, но дата возникновения обязательства не установлена";
- "обычное хозяйственное объяснение не установлено по доступным данным".

Это поле нужно для внутреннего анализа.
Оно помогает не начинать сразу с риска.

ШАГ 6. ОПРЕДЕЛИ risk_signals

risk_signals — только сигналы, не выводы.

Примеры:
- "чувствительный период";
- "убытие средств";
- "размытое назначение платежа";
- "долговой характер платежа";
- "связь сторон";
- "несоответствие ОКВЭД характеру операции";
- "негативный статус контрагента";
- "налоговая задолженность контрагента";
- "судебная нагрузка контрагента";
- "нет данных о встречном исполнении";
- "нет данных о дате возникновения обязательства".

Запрещено писать в risk_signals:
- "операция является подозрительной";
- "имеется предпочтение";
- "сделка недействительна";
- "применяется статья 61.2".

ШАГ 7. ОПРЕДЕЛИ protective_signals

protective_signals — факторы, которые поддерживают обычную версию или снижают риск.

Примеры:
- "purpose конкретен";
- "операция похожа на обычный обязательный платеж";
- "контрагент соответствует роли";
- "ОКВЭД соответствует характеру платежа";
- "связи не переданы";
- "связь слабая и не влияет на операцию";
- "операция похожа на регулярную хозяйственную";
- "платеж является поступлением, а не выбытием".

ШАГ 8. ОЦЕНИ СВЯЗИ

Если связи переданы, определи:
- strongest_connection;
- connection_set_summary;
- connection_strength;
- connection_relevance.

Сила связи:
- strong: общий бенефициар, общий руководитель, общий учредитель, родственник КДЛ, контроль через цепочку лиц;
- medium: бывший руководитель, бывший сотрудник, регулярные расчеты, общий адрес;
- weak: общий телефон, email, представитель, единичное техническое совпадение.

Правило:
Связь усиливает только уже возникшую гипотезу.
Если операция типовая, связь обычно остается фоновым сигналом.

ШАГ 9. СФОРМУЛИРУЙ ДО 2 ГИПОТЕЗ

Гипотезы формулируй только если есть вопрос, который может изменить итоговый вывод.

Формат гипотезы:
{{
  "code": "",
  "statement": "",
  "basis_fields": [],
  "evidence_for": [],
  "evidence_against": [],
  "evidence_gaps": [],
  "tool_to_check": "",
  "tool_reason": ""
}}

Правила:
- H_ORDINARY_ACTIVITY ставь, если обычная версия сильная и должна быть сохранена для анализатора;
- H_PREFERENCE ставь только для долговых платежей или погашения обязательств;
- H_SUSPICIOUS_TRANSFER ставь только если есть конфликт с обычным объяснением;
- H_COUNTERPARTY_RISK ставь только если профиль контрагента связан с характером операции;
- H_INSUFFICIENT_DATA ставь только если пробел критичен.

ШАГ 10. РЕШИ, НУЖНЫ ЛИ TOOLS

Если операция закрывается ordinary_explanation и существенных гипотез нет:
tools не вызывай.

Если есть материальная гипотеза:
- get_conterparty_risk — для контрагента;
- search_practice_and_normative — для критериев проверки и документов.

ШАГ 11. ПОДГОТОВЬ evidence-map

Не передавай анализатору преждевременную правовую квалификацию.
Не передавай статьи как вывод.
Не делай legal_context_summary как полуготовый вывод.

ФОРМАТ ОТВЕТА ОРКЕСТРАТОРА

Верни только JSON:
{{
  "operations": [
    {{
      "idx": 0,
      "operation_summary": "",
      "transaction_category": "",
      "direction": "",
      "counterparty_identifier": "",
      "counterparty_category": "",
      "confirmed_facts": [],
      "unknown_facts": [],
      "ordinary_explanation": "",
      "risk_signals": [],
      "protective_signals": [],
      "profile_signals": [],
      "affiliation_assessment": {{
        "strongest_connection": "",
        "connection_set_summary": "",
        "connection_strength": "",
        "connection_relevance": "",
        "limitation": ""
      }},
      "hypotheses": [],
      "documents_to_check": [],
      "used_tools": [],
      "ready_for_finalization": true,
      "stop_reason": ""
    }}
  ],
  "requested_tools": []
}}

ПРАВИЛА ФОРМАТА

- Верни только JSON.
- Количество operations равно количеству входных операций.
- Каждый idx встречается один раз.
- Не добавляй текст вне JSON.
- Не копируй поля между idx.
- Если данных нет, используй безопасные формулировки.
"""

ANALYZER_PROMPT = """
Ты — модуль финального анализа риска банковских операций.

ТВОЯ ЗАДАЧА

На основе:
- входных операций;
- результата оркестратора;
- tool_results;
- used_tools;
- графовых связей;
- профиля контрагента;
сформировать итоговый структурированный вывод по каждой операции.

Конечный пользователь — сотрудник отдела принудительного взыскания управления по работе с проблемными активами и финансовому оздоровлению юридических лиц.

Цель вывода:
- не пропустить подозрительную операцию;
- не превращать все операции с неполными данными в risk 2;
- объяснить, что сотруднику делать дальше;
- указать документы для проверки;
- дать предварительный правовой маршрут, если он реально вытекает из операции и tool_results.

ТЫ НЕ:
- не придумываешь факты;
- не делаешь окончательный судебный вывод;
- не утверждаешь, что сделка точно недействительна;
- не используешь статьи без основания;
- не используешь судебную практику как доказательство риска само по себе;
- не копируешь выводы между idx.

ИСТОЧНИКИ ИСТИНЫ

Разрешено использовать:
- входные поля операции;
- orchestrator_result;
- tool_results;
- used_tools;
- collect_all_graph_connections.description;
- профиль контрагента.

Запрещено:
- знания модели вне переданных данных;
- выдуманные нормы права;
- выдуманная судебная практика;
- выдуманные договоры, даты, периоды, документы, история отношений;
- точные anomaly_score и anomaly_model_score;
- повышение риска только по связи;
- повышение риска только по периоду;
- повышение риска только по профилю;
- повышение риска только по отсутствию документов;
- повышение риска только по судебной практике без связи с текущей операцией.

ГЛАВНАЯ МЕТОДИКА

Для каждого idx сравни две версии:

1. ordinary_version
Обычное хозяйственное объяснение операции.

2. risk_version
Риск-гипотеза, если есть факты или сигналы, конфликтующие с ordinary_version.

Риск повышается не от количества сигналов, а от силы связи между:
- характером операции;
- периодом;
- направлением;
- контрагентом;
- связями;
- профилем;
- tool_results;
- судебными критериями;
- недостающими критичными документами.

Если risk_version не сильнее ordinary_version, risk_level не выше 1.

ПОРЯДОК АНАЛИЗА ПО КАЖДОМУ IDX

ШАГ 1. УСТАНОВИ БАЗОВЫЙ СМЫСЛ

Определи:
- что написано в purpose;
- direction;
- amount;
- transaction_category;
- counterparty_category;
- период относительно банкротства;
- ordinary_explanation из оркестратора;
- risk_signals;
- protective_signals.

ШАГ 2. ОЦЕНИ, ЯВЛЯЕТСЯ ЛИ ОПЕРАЦИЯ ОБЫЧНОЙ

Операция считается обычной, если:
- purpose понятен;
- операция похожа на хозяйственную функцию;
- контрагент соответствует роли;
- нет сильной связи, влияющей именно на операцию;
- нет признаков транзита;
- нет признаков погашения старого долга с преимуществом;
- нет признаков отсутствия встречного исполнения;
- профиль контрагента не противоречит операции.

Если операция обычная:
- risk_level 0 или 1;
- legal_basis = [];
- court_basis = [];
- recommendation короткая;
- challenge_criteria не должно вести к оспариванию.

ШАГ 3. ОЦЕНИ МАТЕРИАЛЬНОСТЬ ВОПРОСА

Материальный вопрос есть, если пробел или сигнал мешает понять:
- экономический смысл;
- роль контрагента;
- дату возникновения обязательства;
- наличие встречного исполнения;
- наличие иных кредиторов;
- обычность платежа;
- значение связи;
- дальнейшее движение денег;
- правовую конструкцию.

Если пробел не мешает понять базовый смысл операции, это risk_level 1, а не 2.

ШАГ 4. ОЦЕНИ СВЯЗИ

Заполни connections_basis.

Если связи нет:
strongest_connection = "не выявлено по доступным данным".

Если связь есть:
- выдели strongest_connection;
- опиши совокупность связей;
- оцени силу;
- объясни влияние именно на текущую операцию;
- укажи ограничение.

Связь не повышает риск сама по себе.
Связь повышает риск только если операция уже имеет проблемный характер:
- долговой платеж;
- неясное основание;
- выбытие средств;
- отсутствие встречного исполнения;
- транзит;
- связанный получатель;
- чувствительный период.

ШАГ 5. ОЦЕНИ ПРОФИЛЬ КОНТРАГЕНТА

Профиль не является доказательством.
Профиль влияет только если связан с операцией.

Примеры:
- ОКВЭД соответствует услугам -> protective_signal;
- ОКВЭД не соответствует услугам + purpose размыт -> risk_signal;
- налоговая задолженность при обычной оплате поставки -> слабый фоновый сигнал;
- негативный статус при оплате услуг без результата -> усиливающий сигнал.

ШАГ 6. ОПРЕДЕЛИ challenge_criteria

challenge_criteria — это карта проверки для сотрудника.
Это не итог "оспаривать".

Формат:
{{
  "potential_route": "",
  "criteria_matched": [],
  "criteria_missing": [],
  "documents_needed": [],
  "challenge_readiness": ""
}}

potential_route выбирай осторожно.

Варианты:
- "оснований для оспаривания по доступным данным не выявлено";
- "требуется документальная проверка хозяйственного основания";
- "проверка обычной хозяйственной деятельности";
- "проверка признаков предпочтительного удовлетворения";
- "проверка признаков неравноценного встречного исполнения";
- "проверка признаков сделки во вред кредиторам";
- "проверка признаков транзитного движения денежных средств";
- "проверка признаков мнимой или притворной сделки";
- "приоритетная правовая оценка потенциально оспоримой операции".

Если tool_results дают конкретную норму и критерии, можно указать статью:
- "статья 61.3 Закона о банкротстве — проверка предпочтительного удовлетворения";
- "статья 61.2 пункт 1 Закона о банкротстве — проверка неравноценного встречного исполнения";
- "статья 61.2 пункт 2 Закона о банкротстве — проверка сделки во вред кредиторам";
- "статья 170 ГК РФ — проверка мнимости или притворности".

Но статья должна быть результатом анализа, а не стартовой точкой.

ШАГ 7. ОПРЕДЕЛИ legal_qualification

legal_qualification — предварительная квалификация.

Если операция обычная:
"типовая хозяйственная операция; банкротная квалификация по доступным данным не требуется"

Если смысл понятен, но нужны документы:
"хозяйственная операция требует документального подтверждения"

Если долговой платеж без полного набора критериев:
"долговой платеж, требующий проверки даты возникновения обязательства, обычности исполнения и наличия иных кредиторов"

Если есть признаки предпочтения:
"проверка признаков предпочтительного удовлетворения отдельного кредитора"

Если есть признаки неравноценности:
"проверка реальности и равноценности встречного исполнения"

Если есть признаки вреда:
"проверка деловой цели, встречного исполнения и влияния на имущественную массу должника"

Если есть сильная совокупность и tool_results:
можно указать конкретную статью.

ШАГ 8. ОПРЕДЕЛИ legal_basis

legal_basis заполняй только если:
- tool_results или orchestrator_result содержат релевантную норму;
- норма применима к текущему idx;
- decision_argumentation объясняет, какие факты текущей операции соответствуют критериям нормы.

Если нормативка не вызывалась, не дала конкретной нормы или не относится к операции:
legal_basis = []

В противном случае:
legal_basis = ["Полное наименование нормы (закон, статья, пункт): Краткое описание: как норма применяется к данной операции (1-2 предложения)"
]
ШАГ 9. ОПРЕДЕЛИ court_basis

court_basis — это судебная логика, а не доказательство риска.

Заполняй только если:
- практика пришла из tool_results;
- практика относится к тому же типу операции;
- практика помогает понять документы, критерии или причины отказа/удовлетворения;
- практика применима к текущему idx.

court_basis должен объяснять:
- что суды проверяют;
- какие документы смотрят;
- какие факторы повышают риск;
- какие факторы снижают риск;
- почему суд может отказать;
- чем это похоже на текущую операцию.

Если практика нерелевантна:
court_basis = []

ШАГ 10. ОПРЕДЕЛИ risk_level

Используй шкалу из SHARED_AGENT_CONTRACT.

Дополнительная логика:

risk_level = 0
Если операция понятна, обычна, существенных вопросов нет.

risk_level = 1
Если операция в целом обычная, но:
- есть слабые отклонения;
- есть неполные документы;
- есть фоновая связь;
- есть фоновый профильный сигнал;
- нужны стандартные документы.

risk_level = 2
Если есть материальный вопрос:
- долговой платеж в чувствительный период;
- неясное основание и значимые сигналы;
- оплата услуг/работ/поставки с критичным пробелом по встречному исполнению;
- связь сторон влияет на проблемную операцию;
- профиль контрагента противоречит роли;
- tool_results не сняли вопрос;
- судебная логика показывает важные критерии, которые не подтверждены.

risk_level = 3
Если есть сильная совокупность:
- операция в периоде оспаривания;
- выбытие средств или ухудшение имущественной массы;
- сильная связь или осведомленность;
- долговой/транзитный/неравноценный/вредоносный характер;
- tool_results поддерживают гипотезу;
- отсутствуют или противоречат ключевые подтверждения;
- требуется приоритетная эскалация.

ШАГ 11. НАПИШИ decision_argumentation

3–6 предложений.
Обязательно:
1. Что установлено по операции.
2. Какая обычная версия возможна.
3. Что вызывает риск-вопрос.
4. Как учтены связи и профиль.
5. Что дали tools / практика, если были.
6. Почему выбран risk_level.

Не перечисляй все поля подряд.
Не пиши статьи без legal_basis.
Не делай вывод только из сигналов.

ШАГ 12. НАПИШИ risk_explanation

2–3 предложения.
Объясни:
- почему именно этот risk_level;
- какие факторы решающие;
- что является только сигналом;
- что может изменить риск.

ШАГ 13. СФОРМИРУЙ recommendation

Формат:
{{
  "summary": "",
  "documents_to_request": [],
  "verification_goal": "",
  "risk_change_conditions": ""
}}

Для risk 0:
- коротко;
- обычно без документов.

Для risk 1:
- запросить минимальный набор документов, если нужно.

Для risk 2:
- запросить документы по критериям проверки;
- указать, что может повысить или снизить риск.

Для risk 3:
- приоритетная правовая оценка;
- полный комплект документов;
- проверка критериев оспаривания.

Если передан requested_documents (базовый список документов от классификатора типа операции)
-- используй его как стартовый набор для documents_to_request, дополни специфичными для
risk_level документами, если требуется. Пустой requested_documents ничего не меняет в этом шаге.

ШАГ 14. used_tools

Включай только реально использованные tools, применимые к текущему idx.

ФОРМАТ ОТВЕТА

Верни только JSON:
{{
  "operations": [
    {{
      "idx": 0,
      "transaction_category": "",
      "amount": 0,
      "counterparty_category": "",
      "connections_basis": {{
        "strongest_connection": "",
        "connection_set_summary": "",
        "connection_strength": "",
        "influence_on_risk": "",
        "limitation": ""
      }},
      "challenge_criteria": {{
        "potential_route": "",
        "criteria_matched": [],
        "criteria_missing": [],
        "documents_needed": [],
        "challenge_readiness": ""
      }},
      "risk_level": 0,
      "legal_qualification": "",
      "legal_basis": [],
      "court_basis": [],
      "decision_argumentation": "",
      "risk_explanation": "",
      "recommendation": {{
        "summary": "",
        "documents_to_request": [],
        "verification_goal": "",
        "risk_change_conditions": ""
      }},
      "used_tools": []
    }}
  ]
}}

ТРЕБОВАНИЯ

- JSON валидный.
- Верхний уровень — объект с ключом operations.
- Количество operations равно количеству входных операций.
- Каждый idx один раз.
- Дополнительных полей нет.
- Пустых строк нет.
- Если данных нет, используй безопасную формулировку.
- Нельзя упоминать cluster в финальном тексте.
- Нельзя копировать тексты между idx.
- Нельзя выводить точные anomaly_score.
- Нельзя ссылаться на неиспользованные tools.
- Нельзя использовать court_basis как доказательство риска само по себе.

ПРИМЕР 1. ТИПОВАЯ ОПЕРАЦИЯ

Вход:
{{
  "idx": 17,
  "purpose": "Заработная плата за апрель 2024",
  "amount": 85000,
  "direction": "убытие",
  "interval": "-1-3 года",
  "credit_name": "Иванов Петр Сергеевич"
}}

Выход:
{{
  "operations": [
    {{
      "idx": 17,
      "transaction_category": "ФОТ",
      "amount": 85000,
      "counterparty_category": "Сотрудник компании",
      "connections_basis": {{
        "strongest_connection": "не выявлено по доступным данным",
        "connection_set_summary": "связи между сторонами операции не переданы",
        "connection_strength": "не установлено по доступным данным",
        "influence_on_risk": "связи не влияют на оценку риска",
        "limitation": "вывод ограничен доступными данными по операции"
      }},
      "challenge_criteria": {{
        "potential_route": "оснований для оспаривания по доступным данным не выявлено",
        "criteria_matched": ["назначение платежа указывает на заработную плату", "операция имеет обычный хозяйственный характер"],
        "criteria_missing": [],
        "documents_needed": [],
        "challenge_readiness": "по доступным данным оснований для оспаривания не выявлено"
      }},
      "risk_level": 0,
      "legal_qualification": "типовая хозяйственная операция; банкротная квалификация по доступным данным не требуется",
      "legal_basis": [],
      "court_basis": [],
      "decision_argumentation": "Операция является выплатой заработной платы за указанный период. Обычная версия операции сильная: назначение платежа понятно, контрагент соответствует роли работника, признаков вывода активов, предпочтения или транзита по доступным данным нет. Связи не переданы и не влияют на вывод. Поэтому операция не требует дополнительной правовой проверки.",
      "risk_explanation": "Risk_level 0 выбран, потому что операция понятна и выглядит типовой. Существенных риск-сигналов по текущему idx не выявлено.",
      "recommendation": {{
        "summary": "Дополнительные действия не требуются.",
        "documents_to_request": [],
        "verification_goal": "не требуется",
        "risk_change_conditions": "риск может измениться только при появлении данных, что выплата не связана с реальными трудовыми отношениями"
      }},
      "used_tools": []
    }}
  ]
}}

ПРИМЕР 2. НЕПОЛНАЯ, НО В ЦЕЛОМ ХОЗЯЙСТВЕННАЯ ОПЕРАЦИЯ

Вход:
{{
  "idx": 22,
  "purpose": "Оплата по договору за услуги",
  "amount": 300000,
  "direction": "убытие",
  "interval": "-1-6 месяцев",
  "credit_name": "ООО Сервис"
}}

Выход:
{{
  "operations": [
    {{
      "idx": 22,
      "transaction_category": "оплата работ / услуг",
      "amount": 300000,
      "counterparty_category": "Исполнитель услуг",
      "connections_basis": {{
        "strongest_connection": "не выявлено по доступным данным",
        "connection_set_summary": "связи между сторонами операции не переданы",
        "connection_strength": "не установлено по доступным данным",
        "influence_on_risk": "связи не влияют на оценку риска",
        "limitation": "вывод ограничен отсутствием данных о связях"
      }},
      "challenge_criteria": {{
        "potential_route": "требуется документальная проверка хозяйственного основания",
        "criteria_matched": ["назначение платежа указывает на оплату услуг"],
        "criteria_missing": ["не установлен предмет услуг", "не представлены документы, подтверждающие результат услуг"],
        "documents_needed": ["договор", "счет", "акт оказанных услуг", "отчет исполнителя или иной результат услуг"],
        "challenge_readiness": "не готово к правовой квалификации, требуется сбор документов"
      }},
      "risk_level": 1,
      "legal_qualification": "хозяйственная операция требует документального подтверждения",
      "legal_basis": [],
      "court_basis": [],
      "decision_argumentation": "Операция выглядит как оплата услуг по хозяйственному договору. Обычная версия сохраняется, поскольку назначение платежа указывает на хозяйственную оплату, а значимые связи, транзит или проблемный профиль по текущим данным не установлены. Риск-вопрос связан с тем, что предмет услуг и результат встречного исполнения не раскрыты. Этот пробел требует документов, но сам по себе не формирует материальный банкротный риск.",
      "risk_explanation": "Risk_level 1 выбран, потому что операция в целом имеет хозяйственное объяснение, но требует подтверждающих документов. Риск может повыситься, если документы не подтвердят реальность услуг или появятся данные о связи сторон, транзите или отсутствии встречного исполнения.",
      "recommendation": {{
        "summary": "Запросить базовые документы по основанию и результату услуг.",
        "documents_to_request": ["договор", "счет", "акт оказанных услуг", "отчет исполнителя или иной результат услуг"],
        "verification_goal": "подтвердить предмет услуг, основание платежа и реальность встречного исполнения",
        "risk_change_conditions": "риск повысится, если документы отсутствуют, противоречивы или не подтверждают результат услуг; риск снизится, если документы подтверждают обычность и реальность операции"
      }},
      "used_tools": []
    }}
  ]
}}

ПРИМЕР 3. ДОЛГОВОЙ ПЛАТЕЖ С МАТЕРИАЛЬНЫМ ВОПРОСОМ

Вход:
{{
  "idx": 31,
  "purpose": "Возврат долга",
  "amount": 950000,
  "direction": "убытие",
  "interval": "-1-3 месяца",
  "credit_name": "ООО ФинансГрупп",
  "collect_all_graph_connections.description": ["У получателя и должника общий бенефициар"]
}}

Правильный вывод:
- risk_level обычно 2, если нет полного подтверждения критериев;
- risk_level 3 только если tools или дополнительные данные подтверждают сильную совокупность;
- legal_qualification: проверка признаков предпочтительного удовлетворения;
- recommendation: запросить документы по долгу, другим кредиторам, обычности платежа и осведомленности.
"""

ORCHESTRATOR_SYSTEM_PROMPT = (
    SHARED_AGENT_CONTRACT
    + "\n\n"
    + DATA_DICTIONARY
    + "\n\n"
    + LEGAL_ANALYSIS_CONTEXT
    + "\n\n"
    + ORCHESTRATOR_PROMPT
)

# Правовой модуль не видит сырые поля операции: его вход — уже обезличенный
# operations_context от оркестратора. Поэтому SHARED_AGENT_CONTRACT и
# DATA_DICTIONARY ему не нужны (все его запреты и правила продублированы в
# LEGAL_MODULE_PROMPT). Это экономит ~9 тыс. символов на каждый вызов
# search_practice_and_normative без изменения поведения модуля.
LEGAL_MODULE_SYSTEM_PROMPT = (
    LEGAL_ANALYSIS_CONTEXT
    + "\n\n"
    + LEGAL_MODULE_PROMPT
)

ANALYZER_SYSTEM_PROMPT = (
    SHARED_AGENT_CONTRACT
    + "\n\n"
    + DATA_DICTIONARY
    + "\n\n"
    + LEGAL_ANALYSIS_CONTEXT
    + "\n\n"
    + ANALYZER_PROMPT
)


In [ ]:
# ============================================================
# Базовая настройка окружения
# ============================================================


from __future__ import annotations

import hashlib
import json
import logging
import math
import os
import pickle
import re
import sqlite3
import sys
import traceback
import warnings
from datetime import date, datetime
from pathlib import Path
from typing import Annotated, Any, Dict, List, Literal, Optional, TypedDict

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

warnings.filterwarnings("ignore")
np.random.seed(42)
load_dotenv()


# ============================================================
# Логирование
# ============================================================

LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)

LOG_FILE = LOG_DIR / f"agent_run_{datetime.now():%Y%m%d_%H%M%S}.log"

logger = logging.getLogger("bankruptcy_agent")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False

_log_format = logging.Formatter(
    fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(funcName)s:%(lineno)d | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

_console_handler = logging.StreamHandler(sys.stdout)
_console_handler.setLevel(logging.INFO)
_console_handler.setFormatter(_log_format)

_file_handler = logging.FileHandler(LOG_FILE, encoding="utf-8")
_file_handler.setLevel(logging.DEBUG)
_file_handler.setFormatter(_log_format)

logger.addHandler(_console_handler)
logger.addHandler(_file_handler)

logger.info("Логирование и окружение инициализированы. Файл логов: %s", LOG_FILE)


# ============================================================
# Конфигурация входных и выходных путей
# ============================================================

DATA_DIR = Path("data")
SPARK_DATA_DIR = Path("spark_data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

CONFIG = {
    "transactions_path": DATA_DIR / "aff_merge_1.xlsx",
    "spark_path": SPARK_DATA_DIR / "spark_inn_data.xlsx",
    "court_filing_date": "2025-09-11",
    "min_abs_amount_for_llm": 10_000,
    "random_state": 42,
    # --- Адаптивная стратегия анализа кластеров ---
    # Порог однородности кластера. None -> используется порог уверенности
    # распространения (PROPAGATION_CONFIDENCE_LOW): если средняя пара операций
    # кластера ниже него, перенос объяснений внутри кластера ненадежен.
    # Скорректируйте по распределению из cluster_density_diagnostics.xlsx.
    "cluster_homogeneity_threshold": None,
    # Доля "изолированных" операций (нет достаточно похожего соседа),
    # при превышении которой кластер считается разреженным.
    "max_isolated_share": 0.30,
    # Верхняя граница числа представителей компактного кластера (страховка
    # стоимости; фактический размер выборки определяется покрытием).
    "max_representatives_per_cluster": 8,
    # Максимум операций в одном обращении к агенту (нарезка разреженных кластеров).
    "max_operations_per_llm_batch": 20,
}

REQUIRED_TRANSACTION_COLUMNS = {
    "date",
    "amount",
    "purpose",
    "cluster",
    "anomaly_score",
    "anomaly_model_score",
    "debit_inn",
    "credit_inn",
}


## Загрузка и подготовка данных

В этой секции ноутбук приводит выписку к формату, пригодному для агентного анализа.

Принципиально важно не смешивать два уровня данных:
- исходная выписка и признаки DS-модели;
- обогащение контрагентов, графовые связи, судебные/налоговые признаки.

Все преобразования ниже делают минимально необходимую нормализацию: проверяют обязательные колонки, приводят даты, создают стабильный `idx`, обогащают данные по ИНН и обезличивают чувствительные значения в текстовых полях.


In [ ]:
def require_columns(df: pd.DataFrame, required: set[str], df_name: str = "df") -> None:
    """Проверяет наличие обязательных колонок и явно показывает, чего не хватает."""
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"{df_name}: отсутствуют обязательные колонки: {missing}")


def read_table(path: Path) -> pd.DataFrame:
    """Читает табличный файл по расширению. Единая точка расширения форматов."""
    readers = {
        ".xlsx": pd.read_excel,
        ".xls": pd.read_excel,
        ".csv": pd.read_csv,
        ".parquet": pd.read_parquet,
    }
    reader = readers.get(path.suffix.lower())
    if reader is None:
        raise ValueError(
            f"Неподдерживаемый формат файла: {path.suffix!r} ({path}). "
            f"Поддерживаются: {sorted(readers)}"
        )
    try:
        return reader(path)
    except Exception as exc:
        raise RuntimeError(f"Не удалось прочитать файл {path}: {exc}") from exc


def make_stable_operation_id(df: pd.DataFrame) -> pd.Series:
    """
    Создает стабильный технический идентификатор операции.

    Ключ строится по назначению, сумме и дате. Полностью одинаковые операции
    (два идентичных платежа в один день) получают суффикс порядкового номера,
    чтобы idx оставался уникальным — это требование контракта агента
    («каждый idx встречается один раз»).
    """
    purpose = df["purpose"].fillna("").astype(str).str.strip().str.lower()
    amount = pd.to_numeric(df["amount"], errors="coerce").round(2).fillna(0).astype(str)
    date_part = pd.to_datetime(df["date"], errors="coerce").astype(str)
    raw_key = purpose + "|" + amount + "|" + date_part

    # Порядковый номер внутри группы одинаковых ключей защищает от коллизий.
    dedup_suffix = raw_key.groupby(raw_key).cumcount().astype(str)
    raw_key = raw_key + "|" + dedup_suffix

    return raw_key.apply(lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest()[:16])


def load_transactions(path: Path, court_filing_date: str) -> pd.DataFrame:
    """
    Загружает основную таблицу операций и выполняет базовую нормализацию:

    1. проверка обязательных колонок;
    2. приведение дат и суммы к типам;
    3. создание уникального `idx` (или использование входного `txn_id`);
    4. диагностика качества данных в лог (пустые даты/суммы).
    """
    logger.info("Загрузка операций: %s", path)
    if not path.exists():
        raise FileNotFoundError(f"Файл с операциями не найден: {path}")

    df = read_table(path)
    require_columns(df, REQUIRED_TRANSACTION_COLUMNS, "transactions")

    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["court_filing_date"] = pd.to_datetime(court_filing_date)
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

    # Диагностика качества: не роняем пайплайн, но фиксируем проблемы явно.
    bad_dates = int(df["date"].isna().sum())
    bad_amounts = int(df["amount"].isna().sum())
    if bad_dates:
        logger.warning("Операций с нераспознанной датой: %d", bad_dates)
    if bad_amounts:
        logger.warning("Операций с нераспознанной суммой: %d", bad_amounts)

    # Если во входной таблице уже есть txn_id, используем его как бизнес-идентификатор.
    # Если нет — создаем стабильный хеш по назначению, сумме и дате.
    if "txn_id" in df.columns:
        df["idx"] = df["txn_id"].astype(str)
    else:
        df["idx"] = make_stable_operation_id(df)

    n_dup = int(df["idx"].duplicated().sum())
    if n_dup:
        raise ValueError(f"Обнаружены дубликаты idx: {n_dup}. Проверьте txn_id во входном файле.")

    logger.info("Операции загружены: %d строк, %d колонок", len(df), len(df.columns))
    return df


def load_spark_enrichment(path: Path) -> pd.DataFrame:
    """Загружает таблицу обогащения по ИНН контрагентов."""
    logger.info("Загрузка обогащения контрагентов: %s", path)
    if not path.exists():
        logger.warning("Файл обогащения не найден. Блок обогащения будет пропущен: %s", path)
        return pd.DataFrame()

    spark_df = read_table(path)
    if "inn" in spark_df.columns:
        spark_df["inn"] = spark_df["inn"].astype(str).str.strip()
    logger.info("Обогащение загружено: %d строк, %d колонок", len(spark_df), len(spark_df.columns))
    return spark_df


df = load_transactions(CONFIG["transactions_path"], CONFIG["court_filing_date"])
spark_df = load_spark_enrichment(CONFIG["spark_path"])

df.head()


In [ ]:
def enrich_by_inn(df: pd.DataFrame, spark_df: pd.DataFrame) -> pd.DataFrame:
    """
    Добавляет признаки контрагентов отдельно для дебетовой и кредитовой стороны.

    Важно: справочник предварительно дедуплицируется по ИНН. В прежней версии
    merge выполнялся без дедупликации, а результат присоединялся к исходному
    датафрейму через `join` по позиционному индексу. При дублях ИНН в
    справочнике merge порождал больше строк, чем в исходных данных, и join
    молча присоединял признаки к чужим операциям.
    """
    if spark_df.empty:
        logger.warning("Обогащение по ИНН пропущено: spark_df пустой.")
        return df.copy()

    result = df.copy()

    for side in ("debit", "credit"):
        inn_col = f"{side}_inn"
        ref = spark_df.add_prefix(f"{side}_")
        if inn_col not in result.columns or inn_col not in ref.columns:
            logger.warning("Обогащение стороны %s пропущено: нет колонки %s.", side, inn_col)
            continue

        n_dup = int(ref[inn_col].duplicated().sum())
        if n_dup:
            logger.warning(
                "Справочник контрагентов: %d дублей по %s. Оставляется первая запись.",
                n_dup, inn_col,
            )
            ref = ref.drop_duplicates(subset=inn_col, keep="first")

        # Приводим тип ключа, чтобы merge не терял совпадения из-за int/str.
        result[inn_col] = result[inn_col].astype(str).str.strip()
        ref[inn_col] = ref[inn_col].astype(str).str.strip()

        result = result.merge(ref, on=inn_col, how="left")

    logger.info("После обогащения: %d строк, %d колонок", len(result), len(result.columns))
    return result


df = enrich_by_inn(df, spark_df)
df.to_excel(OUTPUT_DIR / "transactions_enriched.xlsx", index=False)
logger.info("Операций после обогащения: %d", len(df))
df.head()


In [ ]:
def clean_fio(text: Any) -> str:
    """
    Маскирует ФИО
    """
    if pd.isna(text):
        return ""

    text = str(text)

    surname_endings = [
        "ов", "ев", "ин", "ын", "ский", "ая", "ой", "ий", "ич", "вич",
        "ова", "ева", "ина", "ына", "иха", "ава", "ица", "иная",
        "яя", "ькая", "цкая", "цкий", "ской", "цкой", "ни", "нова",
        "ук", "чук",
    ]

    surname_pattern = r"\b[А-ЯЁ][а-яё]*(?:" + "|".join(surname_endings) + r")\b"
    name_pattern = r"[А-ЯЁ][а-яё]+"
    initial_pattern = r"[А-ЯЁ]\."

    patterns = [
        rf"{surname_pattern}\s+{name_pattern}\s+{name_pattern}",
        rf"{surname_pattern}\s+[А-ЯЁ]\s*[А-ЯЁ]",
        rf"{surname_pattern}\s+{initial_pattern}\s*{initial_pattern}",
        rf"{initial_pattern}\s*{initial_pattern}\s*{surname_pattern}",
        rf"[А-ЯЁ]\s*[А-ЯЁ]\s+{surname_pattern}",
        rf"{surname_pattern}\s+{name_pattern}",
        rf"{name_pattern}\s+{surname_pattern}",
    ]

    for pattern in patterns:
        text = re.sub(pattern, "<ФИО>", text)

    return text


In [ ]:
def anonymize_transactions(df: pd.DataFrame) -> pd.DataFrame:
    """Обезличивает назначение платежа и ФИО сторон перед передачей данных в LLM."""
    result = df.copy()

    if "purpose" in result.columns:
        result["purpose"] = (
            result["purpose"]
            .fillna("")
            .astype(str)
            .str.replace(r"\d{20,22}", "<Номер Счета>", regex=True)
            .str.replace(r"\d{10,12}", "<ИНН>", regex=True)
            .apply(clean_fio)
        )

    for col in ("debit_name", "credit_name"):
        if col in result.columns:
            result[col] = result[col].apply(clean_fio)

    logger.info("Обезличивание завершено.")
    return result


df = anonymize_transactions(df)
df.head()


## Инициализация LLM-компонентов и инструментов


- `llm` — основная LLM для оркестратора и анализатора;
- `llm_helper` — LLM для формирования поисковых запросов в нормативную базу и судебную практику;
- `embeddings` — embedding-модель для FAISS/Chroma;
- `retriever` — ретривер нормативной базы, если он не собирается в ноутбуке;
- `practice_tool` — объект поиска судебной практики, если используется внешний индекс;
- `risk_db` — объект `RiskMemoryDB`, если используется историческая память риска.



In [ ]:
def require_runtime_object(name: str) -> Any:
    """Возвращает объект из globals() или дает понятную ошибку конфигурации."""
    value = globals().get(name)
    if value is None:
        raise RuntimeError(
            f"Не найден runtime-объект `{name}`. "
            "Определите его перед запуском этой секции ноутбука."
        )
    return value


## Схемы данных и состояние графа



In [ ]:
from typing import Annotated, Any, Dict, List, Literal, Optional, TypedDict

from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

# ============================================================
# 1. Исходные данные операции
# ============================================================
# Схема описывает минимальный контекст одной операции, который может быть
# передан оркестратору и анализатору. Поля не переименовываются: они связаны
# с промптами, подготовкой данных и экспортом результатов.
# ============================================================

class BankOperation(TypedDict, total=False):
    idx: int
    cluster: Optional[int]
    date: Optional[str]
    interval: Optional[str]
    direction: Optional[Literal["прибытие", "убытие"]]
    purpose: Optional[str]
    amount: Optional[float]
    anomaly_score: Optional[float]
    anomaly_model_score: Optional[float]
    debit_name: Optional[str]
    debit_inn: Optional[str]
    credit_name: Optional[str]
    credit_inn: Optional[str]
    credit_registration_date: Optional[str]
    credit_status: Optional[str]
    credit_okved: Optional[str]
    credit_num_court_cases: Optional[int]
    credit_sum_court_cases: Optional[float]
    credit_loss_profit_amount: Optional[float]
    credit_tax_arrears: Optional[float]
    collect_all_graph_connections: Optional[Dict[str, Any]]

# ============================================================
# 2. Гипотеза оркестратора
# ============================================================

class RiskHypothesis(TypedDict, total=False):
    code: Literal[
        "H_ECONOMIC_MEANING_UNCLEAR",
        "H_PREFERENCE",
        "H_SUSPICIOUS_TRANSFER",
        "H_TRANSIT",
        "H_COUNTERPARTY_RISK",
        "H_ORDINARY_ACTIVITY",
        "H_INSUFFICIENT_DATA",
    ]
    statement: str
    basis_fields: List[str]
    evidence_for: List[str]
    evidence_against: List[str]
    evidence_gaps: List[str]
    status: Literal["open", "checked", "rejected", "not_checkable"]
    tool_to_check: Optional[str]
    tool_reason: Optional[str]

# ============================================================
# 3. Запрос tool от оркестратора
# ============================================================

class ToolRequest(TypedDict, total=False):
    tool_call_id: str
    name: Literal[
        "get_conterparty_risk",
        "search_practice_and_normative",
    ]
    args: Dict[str, Any]
    reason: str
    hypothesis_codes: List[str]

# ============================================================
# 4. Результат выполнения tool
# ============================================================

class ToolExecutionResult(TypedDict, total=False):
    tool_call_id: Optional[str]
    name: str
    args: Dict[str, Any]
    success: bool
    result: Dict[str, Any]
    error: Optional[str]
    limitations: List[str]

# ============================================================
# 5. Used tools
# ============================================================

class UsedTool(TypedDict, total=False):
    name: str
    reason: str
    input_summary: str
    result_summary: str
    success: bool
    limitations: List[str]

# ============================================================
# 6. Результаты нормативки и судебной практики
# ============================================================

class NormativeResult(TypedDict, total=False):
    source_id: str
    law_name: Optional[str]
    article: Optional[str]
    point: Optional[str]
    subpoint: Optional[str]
    path: Optional[str]
    text_summary: str
    criteria_extracted: List[str]
    hypothesis: str
    relevance: Literal["высокая", "средняя", "низкая"]
    limitations: List[str]

class CaseLawResult(TypedDict, total=False):
    source_id: str
    case_number: Optional[str]
    court_level: Optional[str]
    case_summary: str
    court_position: str
    legal_logic: str
    matching_facts: List[str]
    missing_or_different_facts: List[str]
    increase_risk_factors: List[str]
    reduce_risk_factors: List[str]
    important_documents: List[str]
    missing_documents: List[str]
    document_role: Optional[str]
    agent_rule: Optional[str]
    hypothesis: str
    relevance: Literal["высокая", "средняя", "низкая"]
    limitations: List[str]

# ============================================================
# 7. Результат оркестратора по кластеру
# ============================================================

class CounterpartyProfile(TypedDict, total=False):
    counterparty_identifier: str
    counterparty_identifier_source: Literal[
        "credit_inn", "credit_name", "debit_inn", "debit_name", "unknown"
    ]
    counterparty_type: str
    counterparty_role: str
    counterparty_category: str
    profile_assessment: str
    affiliation_assessment: str

class OrchestratorClusterResult(TypedDict, total=False):
    """
    Всё, что оркестратор смог собрать по кластеру.
    Анализатор получает этот объект целиком.
    """
    cluster_summary: str
    operations_count: int
    common_transaction_category: str
    common_direction: Optional[Literal["прибытие", "убытие"]]
    
    counterparties: List[CounterpartyProfile]
    
    confirmed_facts: List[str]
    unknown_facts: List[str]
    data_quality_issues: List[str]
    
    hypotheses: List[RiskHypothesis]
    
    legal_context_summary: Optional[str]
    relevant_norms: List[NormativeResult]
    relevant_case_law: List[CaseLawResult]
    
    documents_to_check: List[str]
    
    history_context: Optional[Dict[str, Any]]
    
    used_tools: List[UsedTool]
    requested_tools: List[ToolRequest]
    
    ready_for_finalization: bool
    stop_reason: str

# ============================================================
# 8. Результат анализатора по кластеру
# ============================================================

class LegalBasisItem(TypedDict, total=False):
    law_name: Optional[str]
    article: Optional[str]
    point: Optional[str]
    subpoint: Optional[str]
    path: Optional[str]
    summary: str

class CourtBasisItem(TypedDict, total=False):
    case_law_summary: str
    court_checked_factors: List[str]
    increase_risk_factors: List[str]
    reduce_risk_factors: List[str]
    important_documents: List[str]
    similarity_to_current_operation: str
    limitations: str

class ConnectionsBasis(TypedDict, total=False):
    strongest_connection: str
    connection_set_summary: str
    connection_strength: str
    influence_on_risk: str
    limitation: str

class ChallengeCriteria(TypedDict, total=False):
    potential_route: str
    criteria_matched: List[str]
    criteria_missing: List[str]
    documents_needed: List[str]
    challenge_readiness: str

class Recommendation(TypedDict, total=False):
    summary: str
    documents_to_request: List[str]
    verification_goal: str
    risk_change_conditions: str

class OperationAnalysis(TypedDict, total=False):
    """Анализ одной операции внутри кластера"""
    idx: int
    transaction_category: str
    amount: Optional[float]
    counterparty_category: str
    connections_basis: ConnectionsBasis
    challenge_criteria: ChallengeCriteria
    risk_level: Literal[0, 1, 2, 3]
    legal_qualification: str
    legal_basis: List[LegalBasisItem]
    court_basis: List[CourtBasisItem]
    decision_argumentation: str
    risk_explanation: str
    recommendation: Recommendation
    used_tools: List[UsedTool]

class AnalyzerClusterResult(TypedDict, total=False):
    """
    Финальный анализ по всему кластеру.
    Содержит разбор каждой операции + общую оценку кластера.
    """
    cluster_summary: str
    operations: List[OperationAnalysis]
    overall_risk_assessment: str
    used_tools: List[UsedTool]

# ============================================================
# 9. AgentState
# ============================================================

class AgentState(TypedDict, total=False):
    messages: Annotated[List[BaseMessage], add_messages]
    
    # Исходные данные - весь кластер операций
    operations: List[BankOperation]
    
    # Лимит итераций
    remaining_steps: int
    
    # Tools
    tool_to_check: List[ToolRequest]  # Запрошенные tools от оркестратора
    tool_results: List[ToolExecutionResult]  # Результаты выполнения tools
    
    # Результаты оркестратора
    orchestrator_result: Optional[OrchestratorClusterResult]
    
    # Результаты анализатора
    analyzer_result: Optional[AnalyzerClusterResult]
    
    # Финальный JSON
    final_json: Optional[Dict[str, Any]]
    
    # Ошибки и предупреждения
    errors: List[str]
    warnings: List[str]

## Нормативная база и судебная практика

Блок ниже отвечает за подключение индексов нормативных документов и судебной практики

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS


def build_normative_retriever(embeddings: Any, index_path: str = "./faiss_normative_db"):
    """
    Загружает FAISS-индекс нормативной базы.

    Если индекс отсутствует, создается минимальный пустой индекс с dummy-документом.
    Это позволяет разработчику проверить pipeline без готовой нормативной базы.
    """
    index_dir = Path(index_path)

    if (index_dir / "index.faiss").exists():
        logger.info("Загрузка FAISS-индекса нормативной базы: %s", index_dir)
        return FAISS.load_local(
            str(index_dir),
            embeddings,
            allow_dangerous_deserialization=True,
        )

    logger.warning("FAISS-индекс нормативной базы не найден. Создается временный dummy-индекс: %s", index_dir)
    index_dir.mkdir(parents=True, exist_ok=True)

    dummy_doc = Document(
        page_content="Инициализационный документ. Замените индекс реальной нормативной базой.",
        metadata={"source": "dummy"},
    )
    vectorstore = FAISS.from_documents([dummy_doc], embeddings)
    vectorstore.save_local(str(index_dir))
    return vectorstore


### Историческая память риска контрагентов: динамическая модель

Хранение — SQLite (`risk_memory.db`): нулевая инфраструктура для прототипа, транзакции,
честные UNIQUE-ограничения; схема без изменений переносится в PostgreSQL при
масштабировании.

**Источник истины — журнал событий, а не накопленное число.** Таблица `risk_events`
хранит подтвержденные риск-события (ИНН, операция, risk_level 2–3, даты, обоснование,
метки времени) с ключом `(inn, court_filing_date, operation_idx)`: повторный анализ той же
операции обновляет запись, а не дублирует ее. Агрегированный риск — **функция от событий,
вычисляемая на момент чтения**: иначе риск физически не мог бы угасать без новых записей.

**Модель агрегации.** Вклад события: `s = severity × proximity × recency`, итог —
noisy-OR: `score = 1 − Π(1 − sᵢ)`. Каждый множитель отвечает за свой временной фактор
из постановки задачи:

- `severity` — risk 3 качественно сильнее risk 2 (1.0 против 0.55), не линейная шкала;
- `proximity` — экспоненциальное затухание по |дней между операцией и датой заявления её
  дела| (вес 0.4 на горизонте года): операция накануне банкротства весит больше операции
  за годы до него; окно симметрично — операции в ходе процедуры так же значимы.
  Банкротство при этом лишь один из множителей, а не единственный источник риска;
- `recency` — экспоненциальное затухание по давности **последнего подтверждения** события
  (полураспад 180 дней от `updated_at`): долго не подтверждаемый риск постепенно теряет
  влияние; повторное подтверждение той же операции при новом анализе освежает вес;
- noisy-OR вместо среднего: регулярно появляющиеся новые подозрительные операции
  **наращивают** риск (повторяемость усиливает), при этом десяток слабых угасших событий
  не «размывает» одно сильное свежее, как это делало бы средневзвешенное.

**Использование агентом.** `get_conterparty_risk` возвращает не число, а объяснение:
score и грейд + количество событий, повторяемость за 180 дней, давность последней
рискованной операции, три сильнейших вклада с обоснованиями (`reason`). По контракту
промптов история — дополнительный сигнал гипотезы `H_COUNTERPARTY_RISK`, а не
автоматический определитель risk_level новой операции.

**Обновление** — детерминированный шаг пайплайна после завершения анализа (не LLM-tool):
модель не решает, что попадает в долговременную память. Снимок агрегата
(`counterparty_risk_snapshot`) поддерживается для выгрузок; актуальное значение всегда
пересчитывается на дату чтения.


In [ ]:
class RiskMemoryDB:
    """
    Динамическая историческая память риска контрагентов.

    Источник истины — журнал событий risk_events; агрегированный риск — функция
    от событий, вычисляемая на момент чтения (as_of), а не статичное накопленное
    число. Модель агрегации (обоснование — в markdown выше):

        s_i   = severity(risk_level) * proximity(операция↔банкротство) * recency(давность подтверждения)
        score = 1 - Π(1 - s_i)          # noisy-OR по событиям

    - severity: risk 3 качественно сильнее risk 2 (1.0 против 0.55), а не линейная шкала;
    - proximity: экспоненциальное затухание по |дней между операцией и датой заявления|
      его дела — операция накануне банкротства весит больше операции за годы до него,
      банкротство остается лишь одним из множителей, а не единственным источником;
    - recency: экспоненциальное затухание по давности последнего подтверждения события
      (полураспад RECENCY_HALF_LIFE_DAYS) — не подтверждаемый риск постепенно угасает;
    - noisy-OR: каждое новое независимое событие увеличивает риск (повторяемость
      усиливает), при этом множество слабых старых событий не «размывает» сильное
      свежее, как это делало бы среднее.
    """

    SEVERITY = {2: 0.55, 3: 1.0}
    PROXIMITY_WEIGHT_AT_ONE_YEAR = 0.4     # вес события в году от даты заявления
    RECENCY_HALF_LIFE_DAYS = 180           # полураспад влияния неподтверждаемого события

    def __init__(self, db_path: str = "risk_memory.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.row_factory = sqlite3.Row
        self.cursor = self.conn.cursor()
        self._create_tables()

    def _create_tables(self):
        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS risk_events (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            inn TEXT NOT NULL,
            operation_idx TEXT NOT NULL DEFAULT '',
            risk_level INTEGER NOT NULL CHECK (risk_level IN (2, 3)),
            operation_date TEXT NOT NULL,
            court_filing_date TEXT NOT NULL,
            reason TEXT,
            created_at TEXT NOT NULL DEFAULT (datetime('now')),
            updated_at TEXT NOT NULL DEFAULT (datetime('now')),
            UNIQUE (inn, court_filing_date, operation_idx)
        )
        """)

        # Снимок агрегата — для выгрузок и инспекции; актуальный риск всегда
        # пересчитывается на момент чтения через compute_current_risk.
        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS counterparty_risk_snapshot (
            inn TEXT PRIMARY KEY,
            as_of TEXT NOT NULL,
            risk_score REAL NOT NULL,
            risk_grade TEXT NOT NULL,
            events_count INTEGER NOT NULL,
            last_event_date TEXT,
            reasoning TEXT,
            updated_at TEXT NOT NULL DEFAULT (datetime('now'))
        )
        """)
        self.conn.commit()
        self._check_schema()

    def _check_schema(self):
        """Ловит устаревшую схему от прошлых запусков и дает явную инструкцию."""
        cols = {row["name"] for row in self.cursor.execute("PRAGMA table_info(risk_events)")}
        if "operation_idx" not in cols or "updated_at" not in cols:
            raise RuntimeError(
                "risk_memory.db создан старой версией схемы. "
                "Для прототипа удалите файл БД и запустите ячейку заново."
            )

    # ------------------------------------------------------
    # Запись событий
    # ------------------------------------------------------

    def add_risk_event(
        self,
        inn: str,
        risk_level: int,
        operation_date: str,
        court_filing_date: str | None = None,
        reason: str | None = None,
        operation_idx: str = "",
    ) -> None:
        """
        Записывает (или обновляет при повторном анализе) риск-событие.

        Ключ (inn, court_filing_date, operation_idx): повторный запуск агента по
        тем же данным обновляет risk_level/reason/updated_at существующей записи,
        а не создает дубль. Обновление updated_at одновременно «освежает» recency
        события — повторное подтверждение того же риска поддерживает его влияние.
        """
        if risk_level not in (2, 3):
            raise ValueError("risk_level должен быть 2 или 3")
        inn = str(inn).strip()
        if not inn:
            raise ValueError("inn не должен быть пустым")

        self._validate_date(operation_date)
        if court_filing_date is None:
            court_filing_date = date.today().isoformat()
        else:
            self._validate_date(court_filing_date)

        if not operation_idx:
            logger.warning(
                "add_risk_event без operation_idx (inn=%s): дедупликация невозможна.", inn
            )
            self.cursor.execute(
                """INSERT INTO risk_events
                   (inn, operation_idx, risk_level, operation_date, court_filing_date, reason)
                   VALUES (?, '', ?, ?, ?, ?)""",
                (inn, risk_level, operation_date, court_filing_date, reason),
            )
        else:
            self.cursor.execute(
                """INSERT INTO risk_events
                   (inn, operation_idx, risk_level, operation_date, court_filing_date, reason)
                   VALUES (?, ?, ?, ?, ?, ?)
                   ON CONFLICT (inn, court_filing_date, operation_idx) DO UPDATE SET
                       risk_level = excluded.risk_level,
                       operation_date = excluded.operation_date,
                       reason = excluded.reason,
                       updated_at = datetime('now')""",
                (inn, str(operation_idx), risk_level, operation_date, court_filing_date, reason),
            )

        self._refresh_snapshot(inn)
        self.conn.commit()

    # ------------------------------------------------------
    # Динамический расчет риска
    # ------------------------------------------------------

    def compute_current_risk(self, inn: str, as_of: str | None = None) -> dict:
        """
        Считает текущий агрегированный риск контрагента на дату as_of
        (по умолчанию — сегодня) по всем делам, в которых он встречался.

        Возвращает не просто число, а объяснение формирования риска: вклад
        сильнейших событий с причинами, повторяемость (события за последние
        180 дней), давность последнего подтверждения и примечание об
        интерпретации.
        """
        inn = str(inn).strip()
        as_of_date = date.today() if as_of is None else datetime.fromisoformat(as_of).date()

        rows = self.cursor.execute(
            """SELECT risk_level, operation_date, court_filing_date, reason, updated_at
               FROM risk_events WHERE inn = ?""",
            (inn,),
        ).fetchall()

        if not rows:
            return {
                "inn": inn,
                "as_of": as_of_date.isoformat(),
                "risk_score": 0.0,
                "risk_grade": "NO_HISTORY",
                "events_count": 0,
                "explanation": "История риск-событий по контрагенту отсутствует.",
            }

        lambda_prox = -math.log(self.PROXIMITY_WEIGHT_AT_ONE_YEAR) / 365
        lambda_rec = math.log(2) / self.RECENCY_HALF_LIFE_DAYS

        contributions = []
        recent_180d = 0
        last_event_date = None

        for row in rows:
            op_dt = datetime.fromisoformat(row["operation_date"]).date()
            case_dt = datetime.fromisoformat(row["court_filing_date"]).date()
            confirmed_dt = datetime.fromisoformat(row["updated_at"]).date()

            severity = self.SEVERITY[row["risk_level"]]
            # Близость к процедуре банкротства: чувствительное окно симметрично —
            # операции незадолго до заявления и в ходе процедуры одинаково значимы.
            proximity = math.exp(-lambda_prox * abs((case_dt - op_dt).days))
            # Давность подтверждения: событие, не подтверждавшееся as_of - updated_at
            # дней, постепенно теряет влияние (будущие подтверждения не усиливаются).
            days_since_confirmed = max((as_of_date - confirmed_dt).days, 0)
            recency = math.exp(-lambda_rec * days_since_confirmed)

            s_i = min(max(severity * proximity * recency, 0.0), 0.99)
            contributions.append({
                "s_i": round(s_i, 4),
                "risk_level": row["risk_level"],
                "operation_date": row["operation_date"],
                "court_filing_date": row["court_filing_date"],
                "reason": row["reason"],
                "proximity": round(proximity, 3),
                "recency": round(recency, 3),
            })

            if days_since_confirmed <= 180:
                recent_180d += 1
            if last_event_date is None or row["operation_date"] > last_event_date:
                last_event_date = row["operation_date"]

        # noisy-OR: повторяющиеся независимые события усиливают риск,
        # а не усредняются.
        score = 1.0
        for c in contributions:
            score *= (1.0 - c["s_i"])
        score = 1.0 - score

        top = sorted(contributions, key=lambda c: c["s_i"], reverse=True)[:3]
        days_since_last = (as_of_date - datetime.fromisoformat(last_event_date).date()).days

        explanation = (
            f"Событий в истории: {len(contributions)} (за последние 180 дней подтверждено: {recent_180d}). "
            f"Последняя рискованная операция: {last_event_date} ({days_since_last} дн. назад). "
            "Сильнейшие вклады: "
            + "; ".join(
                f"[{c['operation_date']}, risk {c['risk_level']}, вклад {c['s_i']:.2f}] {c['reason'] or 'без обоснования'}"
                for c in top
            )
        )

        return {
            "inn": inn,
            "as_of": as_of_date.isoformat(),
            "risk_score": round(score, 4),
            "risk_grade": self._get_risk_grade(score),
            "events_count": len(contributions),
            "recent_events_180d": recent_180d,
            "last_event_date": last_event_date,
            "days_since_last_event": days_since_last,
            "top_contributions": top,
            "explanation": explanation,
        }

    @staticmethod
    def _get_risk_grade(risk_score: float) -> str:
        # Пороги откалиброваны под noisy-OR: одно свежее событие risk 2 рядом с
        # банкротством дает ~0.5 (MEDIUM), одно свежее risk 3 — ~0.95 (HIGH);
        # CRITICAL достижим только повторяемостью (несколько независимых событий),
        # угасшая без подтверждений история опускается в LOW.
        if risk_score <= 0:
            return "NO_HISTORY"
        if risk_score < 0.35:
            return "LOW"
        if risk_score < 0.65:
            return "MEDIUM"
        if risk_score < 0.95:
            return "HIGH"
        return "CRITICAL"

    def _refresh_snapshot(self, inn: str):
        current = self.compute_current_risk(inn)
        self.cursor.execute(
            """INSERT INTO counterparty_risk_snapshot
               (inn, as_of, risk_score, risk_grade, events_count, last_event_date, reasoning, updated_at)
               VALUES (?, ?, ?, ?, ?, ?, ?, datetime('now'))
               ON CONFLICT (inn) DO UPDATE SET
                   as_of = excluded.as_of,
                   risk_score = excluded.risk_score,
                   risk_grade = excluded.risk_grade,
                   events_count = excluded.events_count,
                   last_event_date = excluded.last_event_date,
                   reasoning = excluded.reasoning,
                   updated_at = datetime('now')""",
            (
                current["inn"], current["as_of"], current["risk_score"], current["risk_grade"],
                current["events_count"], current.get("last_event_date"),
                current.get("explanation"),
            ),
        )

    def get_risk_events(
        self, inn: str, court_filing_date: str | None = None, limit: int = 5
    ) -> list[dict]:
        """Последние события по контрагенту (с обоснованиями)."""
        inn = str(inn).strip()
        if court_filing_date is not None:
            self._validate_date(court_filing_date)
            rows = self.cursor.execute(
                """SELECT operation_idx, risk_level, operation_date, court_filing_date, reason
                   FROM risk_events
                   WHERE inn = ? AND court_filing_date = ?
                   ORDER BY operation_date DESC LIMIT ?""",
                (inn, court_filing_date, int(limit)),
            ).fetchall()
        else:
            rows = self.cursor.execute(
                """SELECT operation_idx, risk_level, operation_date, court_filing_date, reason
                   FROM risk_events
                   WHERE inn = ?
                   ORDER BY operation_date DESC LIMIT ?""",
                (inn, int(limit)),
            ).fetchall()
        return [dict(row) for row in rows]

    @staticmethod
    def _validate_date(value: str):
        try:
            datetime.fromisoformat(value).date()
        except (ValueError, TypeError):
            raise ValueError("Дата должна быть строкой в формате YYYY-MM-DD")

    def close(self):
        self.conn.close()


risk_db = RiskMemoryDB()


In [ ]:
try:
    import faiss
except ImportError as e:
    raise ImportError("FAISS не установлен.") from e

BASE_DIR_COURT = Path("court_practice_storage")
BASE_DIR_COURT.mkdir(exist_ok=True)

DB_PATH_COURT = BASE_DIR_COURT / "court_practice.sqlite"
FAISS_INDEX_PATH_COURT = BASE_DIR_COURT / "court_practice.faiss"
FAISS_META_PATH_COURT = BASE_DIR_COURT / "faiss_metadata.pkl"


def listify(value: Any) -> list:
    """Приводит скаляр к списку из одного элемента, список оставляет как есть."""
    if value is None:
        return []
    if isinstance(value, (list, tuple, set)):
        return list(value)
    return [value]


def load_court_faiss_index() -> tuple["faiss.Index", list[dict]]:
    """Загружает FAISS-индекс судебной практики и его метаданные с диска."""
    if not FAISS_INDEX_PATH_COURT.exists() or not FAISS_META_PATH_COURT.exists():
        raise FileNotFoundError(
            f"Индекс судебной практики не найден: {FAISS_INDEX_PATH_COURT} / {FAISS_META_PATH_COURT}"
        )
    index = faiss.read_index(str(FAISS_INDEX_PATH_COURT))
    with open(FAISS_META_PATH_COURT, "rb") as f:
        metadata = pickle.load(f)
    return index, metadata


def embed_texts(texts: List[str], embeddings_model: Any | None = None, normalize: bool = True) -> np.ndarray:
    """Возвращает эмбеддинги float32. Для cosine similarity нормализуем векторы."""
    model = embeddings_model or require_runtime_object("embeddings")
    vectors = np.array(model.embed_documents(texts), dtype="float32")

    if vectors.ndim == 1:
        vectors = vectors.reshape(1, -1)
    if normalize:
        faiss.normalize_L2(vectors)
    return vectors


def fetch_cases_by_ids(conn: sqlite3.Connection, case_ids: List[str]) -> Dict[str, Dict[str, Any]]:
    if not case_ids:
        return {}
    placeholders = ",".join("?" for _ in case_ids)
    rows = conn.execute(
        f"SELECT case_id, full_json FROM cases WHERE case_id IN ({placeholders})",
        case_ids,
    ).fetchall()
    return {case_id: json.loads(full_json) for case_id, full_json in rows}


def case_matches_filters(conn: sqlite3.Connection, case_id: str, filters: Optional[Dict[str, Any]]) -> bool:
    if not filters:
        return True

    row = conn.execute(
        """
        SELECT court_level, act_type, case_type, operation_type, timing_pattern,
               counterparty_relation, real_performance, transaction_regular,
               court_result
        FROM cases WHERE case_id = ?
        """,
        (case_id,),
    ).fetchone()
    if not row:
        return False

    direct_filters = {
        "court_level": row[0],
        "act_type": row[1],
        "case_type": row[2],
        "operation_type": row[3],
        "timing_pattern": row[4],
        "counterparty_relation": row[5],
        "real_performance": row[6],
        "transaction_regular": row[7],
        "court_result": row[8],
    }
    for key, actual in direct_filters.items():
        wanted = filters.get(key)
        if wanted and actual not in set(listify(wanted)):
            return False

    tag_filter_keys = [
        "hypotheses",
        "legal_sources",
        "important_documents",
        "missing_documents",
        "documents_to_request",
    ]
    for tag_type in tag_filter_keys:
        wanted = filters.get(tag_type)
        if not wanted:
            continue
        wanted_set = set(map(str, listify(wanted)))
        rows = conn.execute(
            "SELECT tag_value FROM case_tags WHERE case_id = ? AND tag_type = ?",
            (case_id, tag_type),
        ).fetchall()
        actual_set = {r[0] for r in rows}
        if actual_set.isdisjoint(wanted_set):
            return False
    return True


def compact_case_for_agent(card: Dict[str, Any], score: float) -> Dict[str, Any]:
    """Возвращаем агенту только полезный сжатый слой, без сырого акта и лишнего шума."""
    return {
        "score": round(float(score), 4),
        "case_summary": card.get("case_summary"),
        "risk_factors": card.get("risk_factors"),
        "legal_logic": card.get("legal_logic"),
        "evidence": card.get("evidence"),
    }


def search_court_practice(
    statement: str,
    filters: Optional[Dict[str, Any]] = None,
    top_k: int = 5,
    faiss_multiplier: int = 5,
    conn: Optional[sqlite3.Connection] = None,
    index: Optional["faiss.Index"] = None,
    metadata: Optional[List[Dict[str, Any]]] = None,
) -> Dict[str, Any]:
    """Поиск судебной практики: statement -> FAISS -> case_id -> SQLite -> compact result."""
    if not statement or not statement.strip():
        raise ValueError("statement не должен быть пустым")

    own_conn = False
    if conn is None:
        conn = sqlite3.connect(DB_PATH_COURT)
        own_conn = True

    try:
        if index is None or metadata is None:
            index, metadata = load_court_faiss_index()

        query_vec = embed_texts([statement], normalize=True)
        n_candidates = min(max(top_k * faiss_multiplier, top_k), index.ntotal)
        scores, ids = index.search(query_vec, n_candidates)

        results = []
        seen_case_ids = set()

        for score, vector_id in zip(scores[0], ids[0]):
            if vector_id < 0:
                continue
            case_id = metadata[int(vector_id)]["case_id"]
            if case_id in seen_case_ids:
                continue
            seen_case_ids.add(case_id)

            if not case_matches_filters(conn, case_id, filters):
                continue

            card = fetch_cases_by_ids(conn, [case_id]).get(case_id)
            if not card:
                continue

            results.append(compact_case_for_agent(card, float(score)))
            if len(results) >= top_k:
                break
    finally:
        if own_conn:
            conn.close()

    return {
        "statement": statement,
        "filters": filters or {},
        "top_k": top_k,
        "results_count": len(results),
        "results": results,
    }


class CourtPracticeSearchTool:
    """
    Обертка над индексом судебной практики.

    Если файлы индекса отсутствуют, инструмент переходит в деградированный режим и
    возвращает пустые результаты с предупреждением — это позволяет прогонять pipeline
    без готового индекса (аналогично dummy-индексу нормативной базы выше).
    """

    def __init__(
        self,
        db_path: Path = DB_PATH_COURT,
        faiss_index_path: Path = FAISS_INDEX_PATH_COURT,
        faiss_meta_path: Path = FAISS_META_PATH_COURT,
    ):
        self.db_path = Path(db_path)
        self.faiss_index_path = Path(faiss_index_path)
        self.faiss_meta_path = Path(faiss_meta_path)
        self.available = (
            self.db_path.exists()
            and self.faiss_index_path.exists()
            and self.faiss_meta_path.exists()
        )

        if not self.available:
            logger.warning(
                "Индекс судебной практики не найден (%s). Поиск практики будет возвращать пустой результат.",
                self.faiss_index_path,
            )
            self.conn = None
            self.index = None
            self.metadata = None
            return

        self.conn = sqlite3.connect(self.db_path)
        self.index = faiss.read_index(str(self.faiss_index_path))
        with open(self.faiss_meta_path, "rb") as f:
            self.metadata = pickle.load(f)

    def search(self, statement: str, filters: Optional[Dict[str, Any]] = None, top_k: int = 5) -> Dict[str, Any]:
        if not self.available:
            return {
                "statement": statement,
                "filters": filters or {},
                "top_k": top_k,
                "results_count": 0,
                "results": [],
                "warning": "Индекс судебной практики не подключен.",
            }
        return search_court_practice(
            statement=statement,
            filters=filters,
            top_k=top_k,
            conn=self.conn,
            index=self.index,
            metadata=self.metadata,
        )

    def close(self):
        if self.conn is not None:
            self.conn.close()


practice_tool = CourtPracticeSearchTool()


In [ ]:
def search_case_law_practice(statement: str, filters: dict | None = None, top_k: int = 1) -> dict:
    """
    Ищет похожую судебную практику по описанию подозрительной банковской операции.

    statement — развернутое описание операции и гипотезы.
    Возвращает найденные карточки судебной практики.
    """
    logger.info("search_case_law_practice: statement=%s", statement[:300])
    return require_runtime_object("practice_tool").search(
        statement=statement,
        filters=filters,
        top_k=top_k,
    )


def search_normative_base(query: str, k: int = 2) -> str:
    """
    Ищет релевантные фрагменты в базе нормативных документов.

    В query лучше писать развернуто: тема + термин, например
    "оспаривание сделок должника", "мнимая притворная сделка".
    """
    try:
        docs = require_runtime_object("retriever").invoke(query)
        logger.info("search_normative_base: получено документов: %s", len(docs) if docs else 0)

        if not docs:
            return "Ничего не найдено в нормативной базе"

        chunks = []
        for i, doc in enumerate(docs[:k], start=1):
            meta = doc.metadata or {}
            block = "\n".join([
                f"[Фрагмент {i}]",
                f"Документ: {meta.get('law_name') or 'Неизвестный документ'}",
                f"Глава: {meta.get('chapter_title') or 'Неизвестная глава'}",
                f"Статья: {meta.get('article_title') or 'Неизвестная статья'}",
                f"Путь: {meta.get('path') or 'Нет пути до статьи'}",
                f"Текст: {doc.page_content.strip()}",
            ])
            chunks.append(block)

        result = "\n" + "\n---\n".join(chunks)
        logger.info("search_normative_base: длина результата: %s", len(result))
        return result

    except Exception as e:
        logger.exception("Критическая ошибка в search_normative_base")
        return f"Ошибка при поиске в нормативной базе: {type(e).__name__} — {str(e)}"


@tool
def get_conterparty_risk(inn: str, court_filing_date: str | None = None) -> dict:
    """
    Получить исторический риск контрагента: текущий агрегированный риск с объяснением,
    почему он сформировался, и последние 5 риск-событий с обоснованиями.

    Использовать:
    - при анализе подозрительной операции;
    - если есть ИНН контрагента;
    - если нужно понять историческую подозрительность контрагента.

    Исторический риск — дополнительный сигнал для reasoning, а не готовый risk_level
    новой операции. court_filing_date опциональна: с ней дополнительно возвращаются
    события в разрезе указанного дела.
    """
    logger.info("get_conterparty_risk: inn=%s, court_filing_date=%s", inn, court_filing_date)
    db = require_runtime_object("risk_db")

    # Риск динамический: считается по всем событиям контрагента на текущую дату
    # (recency-затухание идет от момента анализа, а не от даты дела).
    current = db.compute_current_risk(inn=inn)
    events = db.get_risk_events(inn=inn, court_filing_date=court_filing_date, limit=5)
    logger.debug("historical_risk=%s, events=%s", current, events)

    return {
        "counterparty": inn,
        "historical_risk": current,
        "last_events": events,
        "usage_note": (
            "Использовать как дополнительный сигнал (H_COUNTERPARTY_RISK), "
            "а не как автоматическое основание risk_level текущей операции."
        ),
    }


def _normalize_to_list(value: Any) -> List[str]:
    """
    Нормализует значение в список строк: список, JSON-строку списка,
    одиночную строку или None.
    """
    if value is None:
        return []
    if isinstance(value, list):
        return [str(item) for item in value if item]
    if isinstance(value, str):
        text = value.strip()
        if text.startswith("["):
            try:
                parsed = json.loads(text)
                if isinstance(parsed, list):
                    return [str(item) for item in parsed if item]
            except json.JSONDecodeError:
                pass
        if text:
            return [text]
    return []


# Сколько поисковых запросов каждого типа реально выполняется на один вызов tool.
# Ограничение сохраняет контроль стоимости, но не выбрасывает все запросы кроме первого.
MAX_QUERIES_PER_TYPE = 2


@tool
def search_practice_and_normative(
    search_task: str,
    hypotheses: List[str] | None = None,
    operation_context: List[dict] | None = None,
) -> str:
    """
    Ищет релевантную нормативную базу и судебную практику по правовой задаче,
    сформулированной оркестратором.

    Главный вход — search_task: словесная постановка задачи (какие операции
    анализируются, какая гипотеза проверяется, почему возник вопрос, какие правовые
    критерии и практику нужно найти, каких фактов или документов не хватает).

    Пример хорошего search_task:
    "Нужно найти нормативные критерии и судебную практику по ситуации, где должник
    до подачи заявления о банкротстве перечислил деньги контрагенту как возврат долга.
    Проверяется, может ли операция иметь признаки предпочтительного удовлетворения
    отдельного кредитора. Не установлены дата возникновения долга, наличие других
    кредиторов, обычность платежа и документы, подтверждающие основание задолженности."

    hypotheses — не более двух проверяемых гипотез оркестратора; каждая содержит
    code, statement, basis, needs_to_check, evidence_gaps.

    operation_context — структурированный контекст операций для уточнения поиска:
    transaction_category, direction, interval, counterparty_category, confirmed_facts,
    unknown_facts, profile_signals, affiliation_signals.
    """
    logger.info("TOOL search_practice_and_normative: старт")
    logger.info("search_task: %s%s", search_task[:200], "..." if len(search_task) > 200 else "")

    hypotheses = hypotheses or []
    operation_context = operation_context or []
    logger.info("Гипотез: %d, элементов контекста: %d", len(hypotheses), len(operation_context))

    context = {
        "search_task": search_task,
        "hypotheses": hypotheses,
        "operations_context": operation_context,
    }

    messages = [
        SystemMessage(content=LEGAL_MODULE_SYSTEM_PROMPT),
        HumanMessage(content=json.dumps(context, ensure_ascii=False, indent=2)),
    ]

    try:
        response = require_runtime_object("llm_helper").invoke(messages)
    except Exception as e:
        logger.error("Ошибка при вызове LLM правового модуля: %s", e)
        return json.dumps({"error": "Ошибка при вызове LLM", "details": str(e)}, ensure_ascii=False)

    response_text = response.content if hasattr(response, "content") else str(response)
    logger.debug("Сырой ответ правового модуля: %s", response_text)

    clean_text = re.sub(r"^```json\s*|\s*```$", "", response_text.strip(), flags=re.MULTILINE)
    clean_text = clean_text.replace("{{", "{").replace("}}", "}")

    try:
        response_dict = json.loads(clean_text)
    except json.JSONDecodeError as e:
        logger.error("Правовой модуль вернул невалидный JSON: %s", e)
        return json.dumps(
            {"error": "LLM вернула невалидный JSON", "details": str(e), "raw_response": response_text},
            ensure_ascii=False,
        )

    # Нормализация: поддерживаем и исторические варианты ключей от LLM.
    law_queries = _normalize_to_list(
        response_dict.get("case_law_queries", response_dict.get("case_law_practice", []))
    )[:MAX_QUERIES_PER_TYPE]
    norm_queries = _normalize_to_list(
        response_dict.get("normative_queries", response_dict.get("normaive_queries", []))
    )[:MAX_QUERIES_PER_TYPE]

    logger.info("Запросы практики (%d): %s", len(law_queries), law_queries)
    logger.info("Запросы нормативки (%d): %s", len(norm_queries), norm_queries)

    if not law_queries and not norm_queries:
        logger.warning("Правовой модуль не сформировал ни одного поискового запроса")
        return json.dumps(
            {"error": "LLM не сформировала поисковые запросы", "case_law_practice": [], "normative_base": []},
            ensure_ascii=False,
        )

    # Каждый тип поиска выполняется независимо: отсутствие запросов одного типа
    # не блокирует другой (раньше вызов целиком падал в ошибку).
    law_practice = []
    for q in law_queries:
        try:
            law_practice.append(search_case_law_practice(q))
        except Exception as e:
            logger.error("Ошибка поиска судебной практики по запросу %r: %s", q[:100], e)

    normative_base = []
    for q in norm_queries:
        try:
            normative_base.append(search_normative_base(q))
        except Exception as e:
            logger.error("Ошибка поиска нормативной базы по запросу %r: %s", q[:100], e)

    result = {
        "case_law_practice": law_practice,
        "normative_base": normative_base,
    }
    logger.info(
        "TOOL search_practice_and_normative: завершено (практика=%d, нормативка=%d)",
        len(law_practice), len(normative_base),
    )
    return json.dumps(result, ensure_ascii=False)


def update_counterparty_risk_memory(
    inn: str,
    risk_level: int,
    operation_date: str,
    reason: str,
    court_filing_date: str | None = None,
    operation_idx: str = "",
) -> dict:
    """
    Записать подтвержденное риск-событие в долгосрочную память контрагентов.

    Вызывается пайплайном ПОСЛЕ финального анализа (см. секцию обновления памяти
    в конце ноутбука), только для итогового risk_level 2 или 3. Намеренно не
    оформлена как LLM-tool: запись в память — детерминированный шаг пайплайна,
    LLM не должна решать, что попадает в историю.
    """
    if risk_level not in (2, 3):
        raise ValueError("В память можно записывать только risk_level 2 или 3")

    risk_db.add_risk_event(
        inn=inn,
        risk_level=risk_level,
        operation_date=operation_date,
        court_filing_date=court_filing_date,
        reason=reason,
        operation_idx=operation_idx,
    )
    return {"status": "Память обновлена", "inn": inn, "operation_idx": operation_idx}


In [ ]:
orchestrator_tools = [
    search_practice_and_normative,
    get_conterparty_risk,
]

llm = require_runtime_object("llm")

# bind_tools передает модели корректные схемы инструментов (имена, параметры,
# описания) — это соответствует рекомендациям GigaChain и повышает точность
# аргументов даже при том, что запросы инструментов дублируются в JSON-поле
# requested_tools. Узел оркестратора поддерживает оба канала (см. ниже).
orchestrator_llm = llm.bind_tools(orchestrator_tools)

# Анализатор не вызывает инструменты, поэтому инструменты к нему не привязываются.
analyzer_llm = llm


## Оркестратор и анализатор

Секция собирает два LLM-цепочных компонента:

- оркестратор готовит кластер к анализу, формирует гипотезы и запрашивает инструменты;
- анализатор не вызывает инструменты и возвращает финальный структурированный вывод по каждой операции.



In [ ]:
'''
Разделяем задачу по выбору необходимых инструментов, выдвижение гипотез подозрительности и
финальный анализ операций на два разных модуля LLM
'''

# ============================================================
# Промпты
# ============================================================

orchestrator_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        ORCHESTRATOR_SYSTEM_PROMPT
        + """

ДОПОЛНИТЕЛЬНЫЕ ПРАВИЛА ВЫЗОВА TOOLS

Ты можешь вызывать tools только если они реально уменьшают неопределенность.

Если нужен нормативный или судебный поиск, вызывай search_practice_and_normative.
В этот tool нужно передавать:
- search_task: словесную постановку задачи;
- hypotheses: проверяемые гипотезы;
- operation_context: структурированный контекст операции;
- tool_reason: зачем нужен tool.
search_practice_and_normative можно вызвать максимум 2 раза на один кластер.

Если нужна история риска контрагента, вызывай get_conterparty_risk.
Не вызывай get_conterparty_risk без валидного ИНН или наименования контрагента.

Если tools не нужны или все гипотезы уже проверены, не вызывай tools.
Тогда следующим узлом будет analyzer.

Не формируй финальный risk_level. Это делает analyzer.
"""
    ),
    MessagesPlaceholder(variable_name="messages", optional=True),
    (
        "human",
        """
Текущее состояние анализа:

Количество операций в кластере: {operations_count}

Операции кластера:
{operations}

Результаты предыдущих вызовов инструментов:
{tool_results}

Оставшиеся шаги:
- до вызова: {remaining_steps_before_call}
- после вызова: {remaining_steps_after_call}

Твоя задача:
1. Проанализировать ВСЕ операции кластера.
2. Собрать всю доступную информацию по кластеру.
3. Сформулировать гипотезы.
4. При необходимости вызвать tools.
5. Передать в анализатор полную информацию по кластеру.

Верни СТРОГО JSON без markdown.
"""
    ),
])

orchestrator_prompt = orchestrator_prompt.partial(messages=[])
orchestrator_chain = orchestrator_prompt | orchestrator_llm


analyzer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        ANALYZER_SYSTEM_PROMPT
        + """

ДОПОЛНИТЕЛЬНЫЕ ПРАВИЛА

Ты — финальный анализатор. Ты работаешь с КЛАСТЕРОМ операций.

Ты получаешь от оркестратора:
- OrchestratorClusterResult — всю информацию, которую оркестратор смог собрать по кластеру
- tool_results — результаты вызовов инструментов по кластеру
- operations — сами операции кластера

Ты должен:
1. Проанализировать КАЖДУЮ операцию кластера.
2. Использовать всю информацию от оркестратора как контекст для каждой операции.
3. Для каждой операции определить risk_level, legal_qualification, recommendation.
4. Сформулировать общую оценку кластера (overall_risk_assessment).
5. Вернуть результат по ВСЕМУ кластеру с разбивкой по операциям.

Ты НЕ вызываешь tools.
Ты НЕ строишь новые гипотезы — ты используешь гипотезы оркестратора.
Ты принимаешь решение на основе того, что передал оркестратор.

В текстовых значениях итогового JSON запрещены английские слова; ключи JSON остаются
на английском.

Формат ответа — обертка кластера, где каждый элемент operations строго соответствует
объекту операции из раздела «ФОРМАТ ОТВЕТА» выше:
{{
  "cluster_summary": "",
  "operations": [ ...объекты операций в формате из раздела «ФОРМАТ ОТВЕТА»... ],
  "overall_risk_assessment": "",
  "used_tools": []
}}

Количество объектов в operations должно строго совпадать с количеством операций в кластере.
"""
    ),
    MessagesPlaceholder(variable_name="messages", optional=True),
    (
        "human",
        """
Сформируй финальный анализ по кластеру.

Количество операций: {operations_count}

Операции кластера:
{operations}

Результат оркестратора (вся информация по кластеру):
{orchestrator_result}

Результаты инструментов:
{tool_results}

Предупреждения:
{warnings}

Достигнут лимит инструментов: {tool_limit_reached}
Невыполненные запрошенные инструменты: {unexecuted_requested_tools}

Твоя задача:
1. Проанализировать КАЖДУЮ операцию кластера.
2. Использовать всю информацию от оркестратора как контекст.
3. Для каждой операции определить risk_level, legal_qualification, recommendation.
4. Сформировать общую оценку кластера.
5. Вернуть результат по ВСЕМУ кластеру.

Верни СТРОГО JSON без markdown.
"""
    ),
])

analyzer_prompt = analyzer_prompt.partial(messages=[])
analyzer_chain = analyzer_prompt | analyzer_llm


# ============================================================
# Утилиты
# ============================================================

def _extract_json_from_ai_message(message: Any) -> Dict[str, Any]:
    """Извлекает JSON из ответа LLM"""
    if isinstance(message, dict):
        return message

    if isinstance(message, AIMessage):
        content = message.content
    elif isinstance(message, str):
        content = message
    else:
        raise ValueError(f"Неожиданный тип ответа LLM: {type(message).__name__}")

    if not content or not content.strip():
        return {}

    text = content.strip()
    if text.startswith("```json"):
        text = text[7:]
    elif text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]
    text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                pass
        return {"_parse_error": f"Не удалось распарсить JSON: {text[:200]}"}


def _tool_request_key(req: Dict[str, Any]) -> str:
    """Каноничный ключ запроса инструмента для дедупликации."""
    name = req.get("tool_name") or req.get("name") or req.get("tool") or "unknown"
    args = req.get("tool_args") or req.get("arguments") or req.get("args") or {}
    try:
        args_key = json.dumps(args, ensure_ascii=False, sort_keys=True, default=str)
    except TypeError:
        args_key = str(args)
    return f"{name}|{args_key}"


def _collect_requested_tools(raw_response: Any, orchestrator_result: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Собирает запросы инструментов из двух каналов:

    1. native tool calling (AIMessage.tool_calls) — основной канал, соответствующий
       рекомендациям GigaChain/LangChain: модель формирует структурированный вызов
       по схеме инструмента, привязанной через bind_tools;
    2. JSON-поле requested_tools в ответе оркестратора — исторический канал прототипа.

    Раньше native-вызовы молча игнорировались: инструменты были привязаны через
    bind_tools, но узел читал только JSON-поле. Дубликаты (одинаковый инструмент с
    одинаковыми аргументами) объединяются.
    """
    requested: List[Dict[str, Any]] = []
    seen_keys: set[str] = set()

    native_calls = getattr(raw_response, "tool_calls", None) or []
    for call in native_calls:
        req = {
            "tool_call_id": call.get("id") or f"native_{len(requested)}",
            "name": call.get("name"),
            "args": call.get("args") or {},
            "reason": "native tool call",
        }
        key = _tool_request_key(req)
        if key not in seen_keys:
            seen_keys.add(key)
            requested.append(req)

    json_requests = orchestrator_result.get("requested_tools", []) or []
    for i, req in enumerate(json_requests):
        if not isinstance(req, dict):
            continue
        key = _tool_request_key(req)
        if key in seen_keys:
            continue
        seen_keys.add(key)
        if "tool_call_id" not in req:
            name = req.get("name", "unknown")
            req["tool_call_id"] = f"{name}_{i}"
        requested.append(req)

    return requested


# ============================================================
# Узлы графа
# ============================================================

def orchestrator_node(state: AgentState) -> Dict[str, Any]:
    """
    Оркестратор работает с ВСЕМ кластером операций.
    Собирает информацию по всему кластеру и передаёт её в анализатор.
    """
    logger.info("%s", "=" * 80)
    logger.info("ORCHESTRATOR NODE")
    logger.info("%s", "=" * 80)

    operations = state.get("operations", [])
    logger.info("Кластер: %d операций, idx=%s", len(operations), [op.get("idx") for op in operations])

    remaining_steps_before = state.get("remaining_steps", 0)
    remaining_steps_after = max(remaining_steps_before - 1, 0)
    logger.info("Remaining steps: %s -> %s", remaining_steps_before, remaining_steps_after)

    tool_results = state.get("tool_results", [])
    logger.info("Результаты tools: %d шт.", len(tool_results))

    orchestrator_input = {
        # json.dumps вместо неявного str(list[dict]): раньше операции попадали в
        # промпт в виде Python-repr с одинарными кавычками, а не в виде JSON.
        "operations": json.dumps(operations, ensure_ascii=False, indent=2, default=str),
        "operations_count": len(operations),
        "tool_results": json.dumps(tool_results, ensure_ascii=False, default=str),
        "remaining_steps_before_call": remaining_steps_before,
        "remaining_steps_after_call": remaining_steps_after,
    }

    logger.info("Вызов orchestrator_chain...")
    try:
        raw_response = orchestrator_chain.invoke(orchestrator_input)
    except Exception as e:
        logger.error("Ошибка orchestrator_chain: %s", e)
        raise

    orchestrator_result = _extract_json_from_ai_message(raw_response)
    if not isinstance(orchestrator_result, dict):
        logger.error("Неожиданный тип orchestrator_result: %s", type(orchestrator_result))
        orchestrator_result = {"requested_tools": []}

    requested_tools = _collect_requested_tools(raw_response, orchestrator_result)
    logger.info("Запрошенные tools: %d шт. %s", len(requested_tools),
                [req.get("name") for req in requested_tools])

    warnings = list(state.get("warnings", []))
    if remaining_steps_after == 0 and requested_tools:
        warning_msg = "Лимит итераций оркестратора исчерпан: часть запрошенных tools не будет выполнена."
        warnings.append(warning_msg)
        logger.warning(warning_msg)

    result = {
        # AIMessage сохраняется в историю, чтобы ToolMessage не оставались «сиротами».
        "messages": [raw_response] if isinstance(raw_response, AIMessage) else [],
        "remaining_steps": remaining_steps_after,
        "orchestrator_result": orchestrator_result,
        "tool_to_check": requested_tools,
        "warnings": warnings,
    }

    logger.info("ORCHESTRATOR NODE: завершение. Следующий шаг: %s",
                "analyzer" if not requested_tools or remaining_steps_after == 0 else "tools")
    return result


def analyzer_node(state: AgentState) -> Dict[str, Any]:
    """
    Анализатор работает с ВСЕМ кластером операций.
    Получает от оркестратора всю собранную информацию по кластеру,
    tool_results, сами операции — и выдаёт финальный анализ по всем операциям кластера.
    """
    logger.info("%s", "=" * 80)
    logger.info("ANALYZER NODE")
    logger.info("%s", "=" * 80)

    operations = state.get("operations", [])
    logger.info("Кластер: %d операций, idx=%s", len(operations), [op.get("idx") for op in operations])

    orchestrator_result = state.get("orchestrator_result", {})
    tool_results = state.get("tool_results", [])
    logger.info("Результаты tools: %d шт.", len(tool_results))

    requested_tools = state.get("tool_to_check", [])
    remaining_steps = state.get("remaining_steps", 0)
    tool_limit_reached = remaining_steps <= 0 and bool(requested_tools)
    if tool_limit_reached:
        logger.warning("Достигнут лимит инструментов. Невыполненные tools: %s", requested_tools)

    analyzer_input = {
        "operations": json.dumps(operations, ensure_ascii=False, indent=2, default=str),
        "operations_count": len(operations),
        "orchestrator_result": json.dumps(orchestrator_result, ensure_ascii=False, default=str),
        "tool_results": json.dumps(tool_results, ensure_ascii=False, default=str),
        "tool_limit_reached": tool_limit_reached,
        "unexecuted_requested_tools": requested_tools if tool_limit_reached else [],
        "warnings": state.get("warnings", []),
    }

    logger.info("Вызов analyzer_chain...")
    try:
        raw_response = analyzer_chain.invoke(analyzer_input)
    except Exception as e:
        logger.error("Ошибка analyzer_chain: %s", e)
        raise

    analyzer_result = _extract_json_from_ai_message(raw_response)
    if not isinstance(analyzer_result, dict):
        logger.error("Неожиданный тип analyzer_result: %s", type(analyzer_result))
        analyzer_result = {"operations": []}

    operations_analysis = analyzer_result.get("operations", [])
    logger.info("Проанализировано операций: %d", len(operations_analysis))
    for op_analysis in operations_analysis[:3]:
        logger.info("   idx=%s: risk_level=%s",
                    op_analysis.get("idx", "N/A"), op_analysis.get("risk_level", "N/A"))

    logger.info("ANALYZER NODE: завершение")
    return {
        "analyzer_result": analyzer_result,
        "final_json": analyzer_result,
    }


## Tool node и граф выполнения

Tool node выполняет только те инструменты, которые оркестратор явно записал в `tool_to_check`. Результаты не перезаписываются, а накапливаются в `tool_results`, чтобы последующие проходы оркестратора видели уже собранный контекст.

In [ ]:
# ==========================================================
# Tool node и маршрутизация графа
# ==========================================================
# Узел выполняет только те инструменты, которые оркестратор явно положил
# в state["tool_to_check"]. Это сохраняет кластерную логику: tools вызываются
# по потребности кластера, а не автоматически для каждой операции.

# Справочник доступных инструментов. Ключ должен совпадать с именем tool,
# которое оркестратор возвращает в запросе.
tools_by_name = {tool.name: tool for tool in orchestrator_tools}


def tools_node(state: AgentState) -> Dict[str, Any]:
    """
    Выполняет инструменты, запрошенные оркестратором.

    Вход:
    - state["tool_to_check"] — список ToolRequest-подобных словарей.

    Выход:
    - messages — ToolMessage для истории LangGraph;
    - tool_results — накопленный список результатов;
    - tool_to_check — очищается после выполнения, чтобы исключить повторный вызов.

    Запрос, идентичный уже выполненному в этом кластере (тот же инструмент с теми же
    аргументами), пропускается: его результат уже лежит в tool_results, и повторный
    вызов только тратит бюджет remaining_steps. Это программно закрепляет правило
    дедупликации из промпта оркестратора.
    """
    logger.info("%s", "=" * 80)
    logger.info("TOOLS NODE: начало работы")
    logger.info("%s", "=" * 80)

    tool_calls = state.get("tool_to_check", [])
    existing_tool_results = list(state.get("tool_results", []))

    if not tool_calls:
        logger.warning("tool_to_check пуст. Выполнение инструментов пропущено.")
        return {
            "messages": [],
            "tool_results": existing_tool_results,
        }

    executed_keys = {_tool_request_key(item) for item in existing_tool_results}

    logger.info("Запрошено инструментов: %d", len(tool_calls))

    new_tool_messages: list[ToolMessage] = []
    new_tool_results: list[dict[str, Any]] = []

    for i, tool_call in enumerate(tool_calls, start=1):
        # Поддерживаются несколько вариантов названий полей, потому что LLM может
        # вернуть tool request в разных JSON-формах. Это не меняет контракт, а
        # делает узел устойчивее к формату ответа оркестратора.
        name = (
            tool_call.get("tool_name")
            or tool_call.get("name")
            or tool_call.get("tool")
        )
        args = (
            tool_call.get("tool_args")
            or tool_call.get("arguments")
            or tool_call.get("args")
            or {}
        )
        tool_call_id = tool_call.get("tool_call_id") or f"{name}_{i}"

        logger.info("%s", "-" * 60)
        logger.info("Вызов инструмента %d/%d: %s (id=%s)", i, len(tool_calls), name, tool_call_id)
        logger.debug("Аргументы инструмента: %s", args)

        request_key = _tool_request_key({"name": name, "args": args})
        if request_key in executed_keys:
            logger.info("Пропуск: идентичный вызов %s уже выполнен в этом кластере.", name)
            continue

        if not name:
            result = None
            error = "name отсутствует в tool_call"
            logger.error(error)
        elif name not in tools_by_name:
            result = None
            error = f"Tool '{name}' не найден в tools_by_name"
            logger.error(error)
        else:
            selected_tool = tools_by_name[name]
            try:
                logger.info("Выполнение инструмента: %s", name)
                result = selected_tool.invoke(args)
                error = None
                logger.info("Инструмент выполнен успешно: %s", name)
                logger.debug("Результат инструмента %s: %s", name, result)
            except Exception:
                result = None
                error = "Ошибка выполнения инструмента"
                logger.exception("Ошибка при выполнении инструмента %s", name)

        executed_keys.add(request_key)

        tool_result_record = {
            "tool_call_id": tool_call_id,
            "name": name,
            "args": args,
            "success": error is None,
            "result": result if error is None else {},
            "error": error,
            "limitations": [],
        }
        new_tool_results.append(tool_result_record)

        new_tool_messages.append(
            ToolMessage(
                content=json.dumps(
                    {
                        "name": name,
                        "tool_result": result,
                        "error": error,
                        "success": error is None,
                    },
                    ensure_ascii=False,
                    default=str,
                ),
                tool_call_id=tool_call_id,
                name=name,
            )
        )

        logger.info("Статус инструмента %s: %s", name, "успех" if error is None else "ошибка")

    success_count = sum(1 for item in new_tool_results if item["success"])
    fail_count = len(new_tool_results) - success_count

    logger.info("%s", "-" * 60)
    logger.info("Итог tools_node: успешных вызовов=%d, ошибок=%d, пропущено дублей=%d",
                success_count, fail_count, len(tool_calls) - len(new_tool_results))

    return {
        "messages": new_tool_messages,
        "tool_results": existing_tool_results + new_tool_results,
        "tool_to_check": [],
    }


def should_continue(state: AgentState) -> Literal["tools", "analyzer"]:
    """
    Определяет следующий узел после оркестратора.

    Правила маршрутизации:
    - если лимит remaining_steps исчерпан, управление передается анализатору;
    - если оркестратор запросил инструменты, управление передается tools_node;
    - если инструменты не нужны, управление передается анализатору.
    """
    remaining_steps = state.get("remaining_steps", 0)
    tool_to_check = state.get("tool_to_check", [])

    logger.info("SHOULD_CONTINUE: remaining_steps=%s, tool_to_check=%d",
                remaining_steps, len(tool_to_check))

    if remaining_steps <= 0:
        logger.info("Маршрут: analyzer, причина: исчерпан лимит remaining_steps")
        return "analyzer"

    if tool_to_check:
        names = [item.get("name", "unknown") for item in tool_to_check]
        logger.info("Маршрут: tools, запрошены: %s", ", ".join(names))
        return "tools"

    logger.info("Маршрут: analyzer, причина: инструменты не запрошены")
    return "analyzer"


# ==========================================================
# Граф выполнения
# ==========================================================
# Сохраняется исходная схема: orchestrator -> tools/analyzer -> orchestrator/analyzer -> END.

graph_builder = StateGraph(AgentState)

graph_builder.add_node("orchestrator", orchestrator_node)
graph_builder.add_node("orchestrator_tools", tools_node)
graph_builder.add_node("analyzer", analyzer_node)

graph_builder.add_edge(START, "orchestrator")

graph_builder.add_conditional_edges(
    "orchestrator",
    should_continue,
    {
        "tools": "orchestrator_tools",
        "analyzer": "analyzer",
    },
)

graph_builder.add_edge("orchestrator_tools", "orchestrator")
graph_builder.add_edge("analyzer", END)

graph = graph_builder.compile()


## Временные признаки и адаптивная стратегия анализа кластеров

Перед запуском LLM рассчитывается период операции относительно даты заявления о
банкротстве, затем каждый кластер классифицируется как **компактный** или
**разреженный**, и от этого зависит способ передачи в LLM.

**Единое метрическое пространство.** Плотность, выбор представителей и последующее
распространение объяснений считаются одной и той же гибридной метрикой — иначе решение
«кластер достаточно однороден для распространения» принималось бы в одном пространстве,
а само распространение выполнялось в другом. Состав метрики (шаг 6 постановки):

- **текст назначения (вес 0.45)** — TF-IDF символьных n-грамм (3–5): главный носитель
  смысла операции; символьные n-граммы устойчивы к морфологии и номерам договоров.
  Embedding-модель сознательно не используется: она задается вручную вне ноутбука,
  а TF-IDF детерминирован, бесплатен и не создает зависимости от ее подключения;
- **числовые признаки (0.25)** — `anomaly_score`, `anomaly_model_score`, `interval_days`
  и `log1p(|amount|)` в min-max шкале (лог-сжатие не дает сумме доминировать);
- **контрагенты и связи (0.30)** — совпадение ИНН контрагента + сходство картины
  графовых/аффилированных связей (`collect_all_graph_connections.description`). Это
  прямой ответ на требование: операции с одинаковым назначением и суммой, но разными
  группами контрагентов, не получат автоматически одинаковое объяснение — контекстный
  блок опустит их сходство ниже порогов уверенности.

**Определение плотности.** Из кандидатов (silhouette, максимальное расстояние,
дисперсия, расстояние до центроида) выбрана пара показателей:

- средняя попарная похожесть (аналог среднего внутрикластерного расстояния) — против
  silhouette, который измеряет отделимость от чужих кластеров, а не внутреннюю
  однородность, и против максимума, чувствительного к единичному выбросу;
- доля «изолированных» операций (лучший сосед ниже порога уверенности) — такие операции
  некому представлять, сколько представителей ни выбирай.

Порог не фиксированный «с потолка»: по умолчанию он равен нижнему порогу уверенности
распространения (0.65) — если средняя пара операций кластера ниже него, перенос
объяснений был бы преимущественно низкоуверенным. Распределение однородности по
кластерам выгружается в `cluster_density_diagnostics.xlsx` для осознанной корректировки
порога под данные.

**Разреженный кластер** передается в LLM целиком (крупные — частями, чтобы не раздувать
контекст). **Компактный кластер** получает представителей через **Farthest Point
Sampling с остановкой по покрытию**: FPS — жадная 2-аппроксимация задачи k-center
(минимизировать худшее расстояние до ближайшего представителя), что ровно соответствует
цели «у каждой операции должен быть похожий представитель»; k-medoids отвергнут, так как
требует фиксированного k и оптимизирует среднее, жертвуя экстремальными операциями —
а именно аномалии нельзя «представлять» типовыми платежами. Размер выборки не
фиксирован: отбор идет, пока худшая операция кластера не покрыта представителем со
сходством ≥ 0.65 (с верхней страховкой по стоимости) — он автоматически зависит от
размера, компактности и разнообразия кластера.


In [ ]:
def days_to_period(series: pd.Series) -> pd.Series:
    """
    Преобразует разницу в днях относительно даты заявления о банкротстве
    в человекочитаемый период для LLM.
    """
    s = pd.to_numeric(series, errors="coerce")
    sign = np.where(s >= 0, "+ ", "- ")
    abs_days = s.abs()

    conditions = [
        abs_days < 30,
        (abs_days >= 30) & (abs_days < 90),
        (abs_days >= 90) & (abs_days < 180),
        (abs_days >= 180) & (abs_days < 365),
        (abs_days >= 365) & (abs_days < 1095),
        abs_days >= 1095,
    ]

    choices = [
        "<1 месяца",
        "1-3 месяца",
        "3-6 месяцев",
        "6-12 месяцев",
        "1-3 года",
        "более 3 лет",
    ]

    period = np.select(conditions, choices, default="неизвестно")
    return pd.Series(np.where(s.isna(), "неизвестно", sign + period), index=series.index)


In [ ]:
df["court_filing_date"] = pd.to_datetime(df["court_filing_date"], errors="coerce")
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Отрицательное значение означает, что операция была до даты заявления.
df["interval_days"] = (df["date"] - df["court_filing_date"]).dt.days


In [ ]:
df["interval"] = days_to_period(df["interval_days"])
df["interval"].value_counts(dropna=False)


## Определение типа операции: классификатор + анализатор-верификатор

Перед адаптивной кластерной стратегией добавлен отдельный этап: по каждой операции выписки
(включая шумовой кластер `-1`, который адаптивная стратегия ниже полностью исключает из LLM-
анализа) определяется предварительный **тип операции** и список **документов к запросу**.

Пайплайн двухэтапный, зеркалит связку "оркестратор → анализатор" основного агента:

1. **Классификатор** — по закрытому каталогу `subject_id`/`action_id`/`flags` (~90 типов)
   определяет предполагаемый тип операции (`resolve_operation_type`) и базовый список
   документов (`resolve_documents`). Каталог передается модели как данные, а не как список
   keyword-правил в тексте промпта — это снижает ложные срабатывания по подстрокам.
2. **Анализатор-верификатор** — получает предложение классификатора вместе с исходным
   `purpose`/`direction` и независимо сверяет, соответствует ли предложенный тип смыслу
   назначения платежа. Если нет — сам выбирает более подходящий `subject_id`/`action_id` из
   того же каталога (`decision="correct"`), с кратким обоснованием; документы пересчитываются
   под скорректированный тип. Если да — тип подтверждается (`decision="confirm"`).

Результат (`operation_type`, `requested_documents`, `classifier_operation_type`,
`analyzer_decision`, `analyzer_correction_reason`, `operation_type_need_review`) добавляется
колонками в `df` и течет дальше вместе с операцией:

- `operation_type` попадает в `agent_columns` и передается основному анализатору как
  **подсказка** для `transaction_category` — анализатор обязан сверить её с `purpose`
  самостоятельно и может отвергнуть (см. обновленный `SHARED_AGENT_CONTRACT`, ШАГ 2);
- `requested_documents` передается тем же путем как базовый список для `documents_to_request`
  в `recommendation` (ШАГ 13) — анализатор дополняет его документами, специфичными для
  выявленного риска, а не строит список с нуля.


In [ ]:
CATALOG = {
    "actions": {
        "payment":          "оплата/перечисление за предмет (обычно outflow)",
        "receipt":          "поступление оплаты за предмет (обычно inflow)",
        "provision":        "предоставление/выдача (заём, наличные)",
        "return":           "возврат (заём, переплата, товар)",
        "repayment":        "погашение (кредит, основной долг)",
        "interest_payment": "уплата процентов (заём/кредит)",
        "tax_payment":      "уплата налога/взноса в бюджет",
        "tax_refund":       "возврат налога из бюджета",
        "purchase":         "приобретение (ЦБ, доля, пай, имущество)",
        "sale":             "продажа (ЦБ, доля, имущество)",
        "contribution":     "внесение/взнос (наличные, УК, товарищество)",
        "withdrawal":       "снятие наличных",
    },
    "flags": {
        "gov_procurement": "44-ФЗ / 223-ФЗ / госконтракт",
        "related_party":   "явно аффилированное / связанное / взаимозависимое лицо",
        "advance":         "аванс / предоплата / задаток явно указаны",
        "interest_free":   "беспроцентный заём явно указан",
        "price_suspicion": "в назначении явные признаки нерыночной цены",
    },
    "subjects": {
        "goods_material": {
            "goods_supply":       {"triggers": ["поставка", "поставки", "поставленный товар", "поставленной продукции", "отгрузка", "отгружен", "отгруженный товар", "товар", "товары", "за товар", "оплата товара", "предоплата за товар", "продукция", "материалы", "товарно-материальные ценности", "оборудование", "ТМЦ", "договор поставки", "ТОРГ-12"], "allowed_actions": ["payment", "receipt", "return"]},
            "retail_sale":        {"triggers": ["розничная покупка", "розничная продажа", "кассовый чек", "чек за товар"], "allowed_actions": ["payment", "receipt"]},
            "agri_contract":      {"triggers": ["контрактация", "сельхозпродукция", "сельскохозяйственная продукция"], "allowed_actions": ["payment", "receipt"]},
            "energy_supply":      {"triggers": ["энергоснабжение", "электроэнергия", "электрическая энергия"], "allowed_actions": ["payment", "receipt"]},
            "real_estate":        {"triggers": ["недвижимость", "объект недвижимости", "здание", "помещение", "земельный участок", "ЕГРН"], "allowed_actions": ["purchase", "sale", "payment", "receipt"]},
            "enterprise_complex": {"triggers": ["продажа предприятия", "имущественный комплекс"], "allowed_actions": ["purchase", "sale"]},
            "barter":             {"triggers": ["мена", "бартер", "обмен имуществом"], "allowed_actions": ["payment", "receipt"]},
            "exclusive_rights":   {"triggers": ["исключительное право", "патент", "свидетельство", "товарный знак"], "allowed_actions": ["purchase", "sale", "payment"]},
        },
        "works": {
            "construction_works":  {"triggers": ["строительные работы", "СМР", "КС-2", "КС-3", "смета", "подряд на строительство"], "allowed_actions": ["payment", "receipt"]},
            "household_works":     {"triggers": ["бытовой подряд", "заказ-наряд", "бытовые работы"], "allowed_actions": ["payment", "receipt"]},
            "design_survey_works": {"triggers": ["проектные работы", "изыскательские работы", "проектирование"], "allowed_actions": ["payment", "receipt"]},
            "rnd_research":        {"triggers": ["НИР", "научно-исследовательские работы"], "allowed_actions": ["payment", "receipt"]},
            "rnd_development":     {"triggers": ["ОКР", "опытно-конструкторские работы"], "allowed_actions": ["payment", "receipt"]},
            "maintenance_repair":  {"triggers": ["ремонт", "техническое обслуживание", "ТО"], "allowed_actions": ["payment", "receipt"]},
        },
        "services": {
            "consulting":    {"triggers": ["консультационные услуги", "консалтинг"], "allowed_actions": ["payment", "receipt"]},
            "audit":         {"triggers": ["аудит", "аудиторские услуги"], "allowed_actions": ["payment", "receipt"]},
            "marketing":     {"triggers": ["маркетинг", "маркетинговые услуги"], "allowed_actions": ["payment", "receipt"]},
            "it_services":   {"triggers": ["IT-услуги", "ИТ-услуги", "разработка ПО", "сопровождение ПО"], "allowed_actions": ["payment", "receipt"]},
            "legal":         {"triggers": ["юридические услуги", "правовое сопровождение"], "allowed_actions": ["payment", "receipt"]},
            "accounting":    {"triggers": ["бухгалтерские услуги", "ведение бухгалтерского учёта"], "allowed_actions": ["payment", "receipt"]},
            "valuation":     {"triggers": ["оценочные услуги", "оценка имущества", "отчёт об оценке"], "allowed_actions": ["payment", "receipt"]},
            "education":     {"triggers": ["образовательные услуги", "обучение", "курсы"], "allowed_actions": ["payment", "receipt"]},
            "medical":       {"triggers": ["медицинские услуги", "лечение"], "allowed_actions": ["payment", "receipt"]},
            "transport":     {"triggers": ["транспортные услуги", "перевозка", "доставка", "логистика"], "allowed_actions": ["payment", "receipt"]},
            "communication": {"triggers": ["услуги связи", "интернет", "телефония"], "allowed_actions": ["payment", "receipt"]},
            "utilities":     {"triggers": ["коммунальные услуги", "ЖКХ", "водоснабжение"], "allowed_actions": ["payment", "receipt"]},
            "storage":       {"triggers": ["хранение", "складское хранение"], "allowed_actions": ["payment", "receipt"]},
            "insurance":     {"triggers": ["страхование", "страховая премия", "страховой полис"], "allowed_actions": ["payment", "receipt"]},
            "tourism":       {"triggers": ["туристические услуги", "тур", "путёвка"], "allowed_actions": ["payment", "receipt"]},
            "realtor":       {"triggers": ["риэлторские услуги", "комиссия риэлтора"], "allowed_actions": ["payment", "receipt"]},
            "rent":          {"triggers": ["аренда", "арендная плата", "договор аренды"], "allowed_actions": ["payment", "receipt"]},
            "leasing":       {"triggers": ["лизинг", "лизинговый платёж", "договор лизинга"], "allowed_actions": ["payment", "receipt"]},
        },
        "finance": {
            "loan":               {"triggers": ["заём", "займ", "предоставление займа", "возврат займа", "погашение займа"], "allowed_actions": ["provision", "return", "receipt", "interest_payment"]},
            "credit":             {"triggers": ["кредит", "погашение кредита", "платёж по кредиту"], "allowed_actions": ["repayment", "interest_payment"]},
            "financial_aid":      {"triggers": ["финансовая помощь", "безвозмездная финансовая помощь"], "allowed_actions": ["provision", "receipt"]},
            "charter_capital":    {"triggers": ["вклад в уставный капитал", "взнос в УК"], "allowed_actions": ["contribution"]},
            "dividends":          {"triggers": ["дивиденды", "выплата дивидендов"], "allowed_actions": ["payment", "receipt"]},
            "share_buyout":       {"triggers": ["выкуп доли", "купля-продажа доли"], "allowed_actions": ["purchase", "sale"]},
            "deposit":            {"triggers": ["депозит", "банковский вклад"], "allowed_actions": ["contribution", "return"]},
            "securities":         {"triggers": ["ценные бумаги", "акции", "облигации"], "allowed_actions": ["purchase", "sale"]},
            "simple_partnership": {"triggers": ["простое товарищество", "вклад в товарищество"], "allowed_actions": ["contribution"]},
            "mutual_fund_unit":   {"triggers": ["пай ПИФ", "инвестиционный пай"], "allowed_actions": ["purchase"]},
            "factoring":          {"triggers": ["факторинг", "уступка денежного требования"], "allowed_actions": ["payment", "receipt"]},
            "debt_transfer":      {"triggers": ["перевод долга"], "allowed_actions": ["payment", "receipt"]},
        },
        "labor": {
            "salary":        {"triggers": ["заработная плата", "зарплата"], "allowed_actions": ["payment"]},
            "bonus":         {"triggers": ["премия", "премирование"], "allowed_actions": ["payment"]},
            "vacation_pay":  {"triggers": ["отпускные", "оплата отпуска"], "allowed_actions": ["payment"]},
            "severance":     {"triggers": ["выходное пособие"], "allowed_actions": ["payment"]},
            "material_aid":  {"triggers": ["материальная помощь", "матпомощь"], "allowed_actions": ["payment"]},
            "sick_leave":    {"triggers": ["больничный", "пособие по нетрудоспособности"], "allowed_actions": ["payment"]},
            "comp_payments": {"triggers": ["компенсация работнику", "компенсация за отпуск"], "allowed_actions": ["payment"]},
        },
        "budget": {
            "vat":                     {"triggers": ["уплата НДС", "НДС за период"], "allowed_actions": ["tax_payment", "tax_refund"]},
            "profit_tax":              {"triggers": ["налог на прибыль"], "allowed_actions": ["tax_payment"]},
            "ndfl":                    {"triggers": ["НДФЛ"], "allowed_actions": ["tax_payment"]},
            "insurance_contributions": {"triggers": ["страховые взносы", "ПФР", "ФСС", "ФОМС"], "allowed_actions": ["tax_payment"]},
            "transport_tax":           {"triggers": ["транспортный налог"], "allowed_actions": ["tax_payment"]},
            "property_tax":            {"triggers": ["налог на имущество"], "allowed_actions": ["tax_payment"]},
            "land_tax":                {"triggers": ["земельный налог"], "allowed_actions": ["tax_payment"]},
            "excise":                  {"triggers": ["акциз", "акцизы"], "allowed_actions": ["tax_payment"]},
            "tax_penalty":             {"triggers": ["пени", "штраф", "налоговый штраф"], "allowed_actions": ["tax_payment"]},
            "tax_refund_from_budget":  {"triggers": ["возврат налога из бюджета", "возврат НДС из бюджета"], "allowed_actions": ["tax_refund"]},
        },
        "intragroup_cash": {
            "own_accounts_transfer": {"triggers": ["перевод между своими счетами"], "allowed_actions": ["payment"]},
            "affiliated_transfer":   {"triggers": ["перевод аффилированному лицу", "связанному лицу"], "allowed_actions": ["payment"]},
            "intragroup_loan":       {"triggers": ["заём внутри группы", "займ внутри группы"], "allowed_actions": ["provision", "return"]},
            "account_topup":         {"triggers": ["пополнение расчётного счёта"], "allowed_actions": ["contribution"]},
            "cash_deposit":          {"triggers": ["внесение наличных", "взнос наличными"], "allowed_actions": ["contribution"]},
            "cash_withdrawal":       {"triggers": ["снятие наличных", "выдача наличных"], "allowed_actions": ["withdrawal"]},
            "card_transfer":         {"triggers": ["перевод на карту", "на банковскую карту"], "allowed_actions": ["payment"]},
            "tech_site":             {"triggers": ["технологический участок", "ТУ"], "allowed_actions": ["payment", "receipt"]},
        },
        "unilateral_enforcement": {
            "writ_execution":           {"triggers": ["исполнительный лист", "исполнительное производство", "постановление пристава"], "allowed_actions": ["payment"]},
            "fns_demand":               {"triggers": ["требование ФНС", "требование налогового органа"], "allowed_actions": ["tax_payment"]},
            "damage_compensation":      {"triggers": ["возмещение ущерба", "возмещение вреда"], "allowed_actions": ["payment"]},
            "state_duty":               {"triggers": ["госпошлина", "государственная пошлина"], "allowed_actions": ["payment"]},
            "third_party_performance":  {"triggers": ["исполнение обязательства третьим лицом", "оплата за третье лицо"], "allowed_actions": ["payment"]},
        },
        "bank_services": {
            "rko_fee":                 {"triggers": ["комиссия банка за РКО", "РКО"], "allowed_actions": ["payment"]},
            "cash_issue_fee":          {"triggers": ["плата за выдачу наличных"], "allowed_actions": ["payment"]},
            "transfer_fee":            {"triggers": ["плата за перевод", "комиссия за перевод"], "allowed_actions": ["payment"]},
            "account_open_maintain":   {"triggers": ["открытие счёта", "ведение счёта", "обслуживание счёта"], "allowed_actions": ["payment"]},
            "arbitration_manager_fee": {"triggers": ["вознаграждение арбитражного управляющего"], "allowed_actions": ["payment"]},
        },
        "other": {
            "gratuitous_transfer":         {"triggers": ["безвозмездная передача", "дарение"], "allowed_actions": ["payment"]},
            "charity":                     {"triggers": ["благотворительность", "пожертвование"], "allowed_actions": ["payment"]},
            "gift":                        {"triggers": ["подарок"], "allowed_actions": ["payment"]},
            "pledge_security":             {"triggers": ["залог", "обеспечительный платёж"], "allowed_actions": ["payment", "return"]},
            "deposit_advance":             {"triggers": ["задаток", "аванс", "предоплата"], "allowed_actions": ["payment"]},
            "settlement_composition":      {"triggers": ["отступное"], "allowed_actions": ["payment"]},
            "novation":                    {"triggers": ["новация"], "allowed_actions": ["payment"]},
            "debt_forgiveness":            {"triggers": ["прощение долга"], "allowed_actions": []},
            "technical":                   {"triggers": ["сторно", "корректировка"], "allowed_actions": []},
            "current_payment_bankruptcy":  {"triggers": ["текущий платёж в процедуре банкротства", "текущие обязательства"], "allowed_actions": ["payment"]},
            "registry_payment":            {"triggers": ["реестровый платёж"], "allowed_actions": ["payment"]},
            "registry_inclusion":          {"triggers": ["включение в реестр"], "allowed_actions": ["payment"]},
            "bankruptcy_expenses":         {"triggers": ["расходы на процедуру банкротства"], "allowed_actions": ["payment"]},
            "unknown":                     {"triggers": ["оплата по договору", "оплата счёта", "оплата услуг", "перечисление", "возврат", "компенсация", "прочие расчёты"], "allowed_actions": []},
        },
    },
}

# плоский индекс subject_id -> {group, allowed_actions} для валидации
CATALOG_ACTIONS = {}
for group_id, subjects in CATALOG["subjects"].items():
    for subj_id, spec in subjects.items():
        CATALOG_ACTIONS[subj_id] = spec["allowed_actions"]

ALL_FLAGS = set(CATALOG["flags"].keys())

print(f"Загружено subject_id: {len(CATALOG_ACTIONS)}")


# Плоский индекс subject_id -> triggers, нужен детерминированному классификатору/верификатору.
SUBJECT_TRIGGERS: dict[str, list[str]] = {}
for _group_id, _subjects in CATALOG["subjects"].items():
    for _subj_id, _spec in _subjects.items():
        SUBJECT_TRIGGERS[_subj_id] = list(_spec["triggers"])



In [ ]:
OPERATION_TYPE_BY_SUBJECT = {
    "goods_supply": "Поставка товаров", "retail_sale": "Розничная купля-продажа",
    "agri_contract": "Контрактация (сельхозпродукция)", "energy_supply": "Энергоснабжение",
    "real_estate": "Продажа недвижимости", "enterprise_complex": "Продажа предприятия как имущественного комплекса",
    "barter": "Мена (бартер)", "exclusive_rights": "Приобретение исключительных прав",

    "construction_works": "Строительный подряд", "household_works": "Бытовой подряд",
    "design_survey_works": "Проектные и изыскательские работы", "rnd_research": "Научно-исследовательские работы (НИР)",
    "rnd_development": "Опытно-конструкторские работы (ОКР)", "maintenance_repair": "Техническое обслуживание и ремонт",

    "consulting": "Консультационные услуги", "audit": "Аудиторские услуги", "marketing": "Маркетинговые услуги",
    "it_services": "IT-услуги", "legal": "Юридические услуги", "accounting": "Бухгалтерские услуги",
    "valuation": "Оценочные услуги", "education": "Образовательные услуги", "medical": "Медицинские услуги",
    "transport": "Транспортные услуги", "communication": "Услуги связи", "utilities": "Коммунальные услуги",
    "storage": "Услуги хранения", "insurance": "Страховые услуги", "tourism": "Туристические услуги",
    "realtor": "Риэлторские услуги", "rent": "Аренда", "leasing": "Лизинг",

    "financial_aid": "Финансовая помощь", "charter_capital": "Вклад в уставный капитал",
    "dividends": "Выплата дивидендов", "share_buyout": "Выкуп доли у участника",
    "deposit": "Депозит / вклад в банке", "simple_partnership": "Вклад в простое товарищество",
    "mutual_fund_unit": "Приобретение пая в ПИФ", "factoring": "Факторинг", "debt_transfer": "Перевод долга",

    "salary": "Заработная плата", "bonus": "Премия", "vacation_pay": "Отпускные",
    "severance": "Выходное пособие", "material_aid": "Материальная помощь",
    "sick_leave": "Оплата больничного листа", "comp_payments": "Компенсационные выплаты",

    "profit_tax": "Уплата налога на прибыль", "ndfl": "Уплата НДФЛ",
    "insurance_contributions": "Уплата страховых взносов", "transport_tax": "Уплата транспортного налога",
    "property_tax": "Уплата налога на имущество", "land_tax": "Уплата земельного налога",
    "excise": "Уплата акцизов", "tax_penalty": "Уплата пеней и штрафов",
    "tax_refund_from_budget": "Возврат налога из бюджета",

    "own_accounts_transfer": "Перевод между своими счетами", "affiliated_transfer": "Перевод аффилированному лицу",
    "intragroup_loan": "Перевод по договору займа (внутри группы)", "account_topup": "Пополнение расчётного счёта",
    "cash_deposit": "Внесение наличных на счёт", "cash_withdrawal": "Снятие наличных",
    "card_transfer": "Перевод на банковскую карту", "tech_site": "Расчёты через технологический участок (ТУ)",

    "writ_execution": "Уплата по исполнительному листу", "fns_demand": "Уплата по требованию ФНС",
    "damage_compensation": "Возмещение ущерба", "state_duty": "Уплата государственной пошлины",
    "third_party_performance": "Исполнение обязательства третьим лицом",

    "rko_fee": "Комиссия банка за РКО", "cash_issue_fee": "Плата за выдачу наличных",
    "transfer_fee": "Плата за перевод", "account_open_maintain": "Открытие / ведение счёта",
    "arbitration_manager_fee": "Вознаграждение арбитражного управляющего",

    "gratuitous_transfer": "Безвозмездная передача / дарение", "charity": "Благотворительность",
    "gift": "Подарок", "pledge_security": "Залог / обеспечительный платёж",
    "deposit_advance": "Задаток / аванс", "settlement_composition": "Отступное", "novation": "Новация",
    "debt_forgiveness": "Прощение долга", "technical": "Техническая операция (корректировка, сторно)",
    "current_payment_bankruptcy": "Текущий платёж (в процедуре банкротства)", "registry_payment": "Реестровый платёж",
    "registry_inclusion": "Включение в реестр (взнос)", "bankruptcy_expenses": "Расходы на процедуру банкротства",
    "unknown": "Неопознанный платёж",
}

# переопределения там, где именно ДЕЙСТВИЕ меняет тип
OPERATION_TYPE_BY_ACTION = {
    ("loan", "provision"): "Предоставление займа", ("loan", "return"): "Возврат займа",
    ("loan", "receipt"): "Возврат займа", ("loan", "interest_payment"): "Проценты по займу",
    ("credit", "repayment"): "Погашение кредита", ("credit", "interest_payment"): "Проценты по кредиту",
    ("vat", "tax_payment"): "Уплата НДС", ("vat", "tax_refund"): "Возврат налога из бюджета",
    ("securities", "purchase"): "Приобретение ценных бумаг", ("securities", "sale"): "Продажа ценных бумаг",
}

def resolve_operation_type(subject_id, action_id, flags):
    if subject_id == "loan" and action_id == "provision" and "interest_free" in flags:
        return "Предоставление беспроцентного займа"
    key = (subject_id, action_id)
    if key in OPERATION_TYPE_BY_ACTION:
        return OPERATION_TYPE_BY_ACTION[key]
    return OPERATION_TYPE_BY_SUBJECT.get(subject_id, "Неопознанный платёж")


In [ ]:
# ---- Раздел 4: документы по типу операции (OPERATION_MATRIX) ----

OPERATION_MATRIX = {
    "Поставка товаров": {"requested_documents": [
        "Договор поставки", "Спецификация / приложение", "Товарная накладная (ТОРГ-12/УПД)",
        "Транспортная накладная (ТН/ТТН)", "Счёт-фактура", "Акт сверки расчётов",
        "Отчёт об оценке рыночной стоимости (при подозрении на завышение)"]},
    "Строительный подряд": {"requested_documents": [
        "Договор подряда", "Смета / локальный сметный расчёт", "Акт о приёмке выполненных работ (КС-2, КС-3)",
        "Журнал производства работ", "Исполнительная документация", "Акт скрытых работ",
        "Отчёт об оценке рыночной стоимости работ", "Разрешение на строительство"]},
    "Возврат займа": {"requested_documents": [
        "Договор займа", "Платёжное поручение о возврате", "График платежей (если был)",
        "Акт сверки расчётов", "Реестр требований кредиторов (для проверки других кредиторов)"]},
    "Неопознанный платёж": {"requested_documents": [
        "Запрос в банк о предоставлении расшифровки назначения платежа",
        "Договор (при обнаружении)", "Первичные документы (при обнаружении)"]},
    "Неопознанный платёж (ссылка на документ)": {"requested_documents": [
        "Запросить документ, указанный в назначении",
        "Первичные документы к нему (акты, накладные, счета-фактуры)",
        "Определить предмет платежа по содержанию документа и переклассифицировать"]},

    "Розничная купля-продажа": {"requested_documents": ["Кассовый чек", "Товарный чек", "Акт сверки расчётов"]},
    "Контрактация (сельхозпродукция)": {"requested_documents": ["Договор контрактации", "Товарная накладная", "Акт приёма-передачи продукции", "Счёт-фактура", "Акт сверки расчётов"]},
    "Энергоснабжение": {"requested_documents": ["Договор энергоснабжения", "Акт снятия показаний приборов учёта", "Счёт-фактура", "Акт сверки расчётов"]},
    "Продажа недвижимости": {"requested_documents": ["Договор купли-продажи недвижимости", "Выписка из ЕГРН", "Акт приёма-передачи объекта", "Отчёт об оценке рыночной стоимости", "Платёжное поручение"]},
    "Продажа предприятия как имущественного комплекса": {"requested_documents": ["Договор продажи предприятия", "Передаточный акт", "Акт инвентаризации имущества", "Бухгалтерский баланс предприятия на дату продажи", "Отчёт об оценке рыночной стоимости"]},
    "Мена (бартер)": {"requested_documents": ["Договор мены", "Акт приёма-передачи имущества (по каждой стороне)", "Отчёт об оценке рыночной стоимости обмениваемого имущества"]},
    "Приобретение исключительных прав": {"requested_documents": ["Договор об отчуждении исключительного права / лицензионный договор", "Свидетельство / патент", "Акт приёма-передачи прав", "Отчёт об оценке стоимости права"]},

    "Бытовой подряд": {"requested_documents": ["Договор бытового подряда", "Заказ-наряд", "Акт выполненных работ"]},
    "Проектные и изыскательские работы": {"requested_documents": ["Договор на выполнение проектных/изыскательских работ", "Техническое задание", "Проектная документация", "Акт сдачи-приёмки работ"]},
    "Научно-исследовательские работы (НИР)": {"requested_documents": ["Договор на выполнение НИР", "Техническое задание", "Отчёт о выполнении НИР", "Акт сдачи-приёмки работ"]},
    "Опытно-конструкторские работы (ОКР)": {"requested_documents": ["Договор на выполнение ОКР", "Техническое задание", "Конструкторская документация", "Акт сдачи-приёмки работ"]},
    "Техническое обслуживание и ремонт": {"requested_documents": ["Договор на техническое обслуживание/ремонт", "Дефектная ведомость", "Заказ-наряд", "Акт выполненных работ"]},

    "Консультационные услуги": {"requested_documents": ["Договор оказания консультационных услуг", "Техническое задание", "Акт оказанных услуг", "Отчёт консультанта / заключение"]},
    "Аудиторские услуги": {"requested_documents": ["Договор оказания аудиторских услуг", "Аудиторское заключение", "Акт оказанных услуг"]},
    "Маркетинговые услуги": {"requested_documents": ["Договор оказания маркетинговых услуг", "Маркетинговое исследование / отчёт", "Акт оказанных услуг"]},
    "IT-услуги": {"requested_documents": ["Договор на разработку/сопровождение ПО", "Техническое задание", "Акт сдачи-приёмки услуг", "Акт-приёмки ПО (при разработке)"]},
    "Юридические услуги": {"requested_documents": ["Договор оказания юридических услуг", "Акт оказанных услуг", "Правовое заключение / процессуальные документы (при наличии)"]},
    "Бухгалтерские услуги": {"requested_documents": ["Договор на бухгалтерское обслуживание", "Акт оказанных услуг"]},
    "Оценочные услуги": {"requested_documents": ["Договор на проведение оценки", "Отчёт об оценке", "Акт оказанных услуг"]},
    "Образовательные услуги": {"requested_documents": ["Договор оказания образовательных услуг", "Лицензия на образовательную деятельность", "Акт оказанных услуг / документ об обучении"]},
    "Медицинские услуги": {"requested_documents": ["Договор оказания медицинских услуг", "Лицензия на медицинскую деятельность", "Акт оказанных услуг"]},
    "Транспортные услуги": {"requested_documents": ["Договор перевозки / транспортной экспедиции", "Транспортная накладная (ТН/ТТН)", "Акт оказанных услуг"]},
    "Услуги связи": {"requested_documents": ["Договор оказания услуг связи", "Акт оказанных услуг", "Детализация счёта (при наличии)"]},
    "Коммунальные услуги": {"requested_documents": ["Договор на коммунальное обслуживание", "Акт снятия показаний приборов учёта", "Счёт-фактура / квитанция", "Акт сверки расчётов"]},
    "Услуги хранения": {"requested_documents": ["Договор хранения", "Акт приёма-передачи на хранение", "Акт возврата с хранения", "Акт оказанных услуг"]},
    "Страховые услуги": {"requested_documents": ["Договор страхования", "Страховой полис", "Платёжное поручение об уплате премии"]},
    "Туристические услуги": {"requested_documents": ["Договор о реализации туристского продукта", "Ваучер / путёвка", "Акт оказанных услуг"]},
    "Риэлторские услуги": {"requested_documents": ["Договор на оказание риэлторских услуг", "Акт оказанных услуг", "Отчёт об оценке рыночной стоимости вознаграждения"]},
    "Аренда": {"requested_documents": ["Договор аренды", "Акт приёма-передачи объекта аренды", "Акт оказанных услуг (арендная плата)", "Акт сверки расчётов"]},
    "Лизинг": {"requested_documents": ["Договор лизинга", "Акт приёма-передачи предмета лизинга", "График лизинговых платежей", "Акт сверки расчётов"]},

    "Предоставление займа": {"requested_documents": ["Договор займа", "Платёжное поручение о выдаче займа", "Доказательства наличия средств у займодавца", "График платежей (если есть)"]},
    "Предоставление беспроцентного займа": {"requested_documents": ["Договор беспроцентного займа", "Платёжное поручение о выдаче займа", "Доказательства наличия средств у займодавца"]},
    "Проценты по займу": {"requested_documents": ["Договор займа", "Расчёт процентов", "Платёжное поручение"]},
    "Погашение кредита": {"requested_documents": ["Кредитный договор", "График погашения", "Платёжное поручение", "Справка банка-кредитора об остатке задолженности"]},
    "Проценты по кредиту": {"requested_documents": ["Кредитный договор", "Расчёт процентов", "Платёжное поручение"]},
    "Финансовая помощь": {"requested_documents": ["Решение / соглашение об оказании финансовой помощи", "Платёжное поручение", "Доказательства наличия средств у стороны, оказывающей помощь"]},
    "Вклад в уставный капитал": {"requested_documents": ["Решение об увеличении уставного капитала / устав", "Платёжное поручение", "Выписка из ЕГРЮЛ"]},
    "Выплата дивидендов": {"requested_documents": ["Решение о выплате дивидендов", "Расчёт суммы дивидендов", "Платёжное поручение"]},
    "Выкуп доли у участника": {"requested_documents": ["Договор купли-продажи доли", "Выписка из ЕГРЮЛ", "Отчёт об оценке стоимости доли", "Платёжное поручение"]},
    "Депозит / вклад в банке": {"requested_documents": ["Договор банковского вклада", "Платёжное поручение", "Выписка по депозитному счёту"]},
    "Приобретение ценных бумаг": {"requested_documents": ["Договор купли-продажи ценных бумаг", "Выписка по счёту депо", "Отчёт брокера", "Платёжное поручение"]},
    "Продажа ценных бумаг": {"requested_documents": ["Договор купли-продажи ценных бумаг", "Выписка по счёту депо", "Отчёт брокера", "Платёжное поручение"]},
    "Вклад в простое товарищество": {"requested_documents": ["Договор простого товарищества", "Платёжное поручение", "Акт оценки вклада (если вклад имуществом)"]},
    "Приобретение пая в ПИФ": {"requested_documents": ["Заявка на приобретение пая", "Правила доверительного управления ПИФ", "Выписка из реестра владельцев инвестиционных паёв"]},
    "Факторинг": {"requested_documents": ["Договор факторинга", "Реестр уступленных денежных требований", "Уведомление об уступке", "Платёжное поручение"]},
    "Перевод долга": {"requested_documents": ["Договор о переводе долга", "Согласие кредитора на перевод долга", "Акт сверки расчётов"]},

    "Заработная плата": {"requested_documents": ["Трудовой договор", "Расчётный лист / расчётная ведомость", "Штатное расписание", "Платёжная ведомость"]},
    "Премия": {"requested_documents": ["Положение о премировании", "Приказ о премировании", "Платёжная ведомость"]},
    "Отпускные": {"requested_documents": ["Приказ о предоставлении отпуска", "Расчёт отпускных", "Платёжная ведомость"]},
    "Выходное пособие": {"requested_documents": ["Приказ об увольнении", "Расчёт выходного пособия", "Платёжная ведомость"]},
    "Материальная помощь": {"requested_documents": ["Заявление работника", "Приказ о выплате материальной помощи", "Платёжная ведомость"]},
    "Оплата больничного листа": {"requested_documents": ["Листок нетрудоспособности", "Расчёт пособия", "Платёжная ведомость"]},
    "Компенсационные выплаты": {"requested_documents": ["Приказ / соглашение о компенсационной выплате", "Расчёт компенсации", "Платёжная ведомость"]},

    "Уплата НДС": {"requested_documents": ["Налоговая декларация по НДС", "Платёжное поручение на уплату налога", "Требование/уведомление налогового органа (при наличии)"]},
    "Уплата налога на прибыль": {"requested_documents": ["Налоговая декларация по налогу на прибыль", "Платёжное поручение на уплату налога"]},
    "Уплата НДФЛ": {"requested_documents": ["Расчёт 6-НДФЛ", "Платёжное поручение на уплату налога"]},
    "Уплата страховых взносов": {"requested_documents": ["Расчёт по страховым взносам (РСВ)", "Платёжное поручение на уплату взносов"]},
    "Уплата транспортного налога": {"requested_documents": ["Налоговое уведомление / декларация по транспортному налогу", "Платёжное поручение на уплату налога"]},
    "Уплата налога на имущество": {"requested_documents": ["Налоговая декларация по налогу на имущество", "Платёжное поручение на уплату налога"]},
    "Уплата земельного налога": {"requested_documents": ["Налоговая декларация по земельному налогу", "Платёжное поручение на уплату налога"]},
    "Уплата акцизов": {"requested_documents": ["Налоговая декларация по акцизам", "Платёжное поручение на уплату налога"]},
    "Уплата пеней и штрафов": {"requested_documents": ["Требование налогового органа / постановление о штрафе", "Расчёт суммы пени/штрафа", "Платёжное поручение"]},
    "Возврат налога из бюджета": {"requested_documents": ["Заявление о возврате налога", "Решение налогового органа", "Платёжное поручение (входящее)"]},
    "Уплата по требованию ФНС": {"requested_documents": ["Требование налогового органа", "Расчёт задолженности", "Платёжное поручение"]},

    "Перевод между своими счетами": {"requested_documents": ["Выписки по обоим счетам", "Платёжное поручение"]},
    "Перевод аффилированному лицу": {"requested_documents": ["Документы о структуре владения/связях сторон", "Основание перевода (договор/иное)", "Платёжное поручение", "Реестр требований кредиторов (для проверки других кредиторов)"]},
    "Перевод по договору займа (внутри группы)": {"requested_documents": ["Договор займа", "Документы о структуре владения/связях сторон", "Платёжное поручение", "Акт сверки расчётов"]},
    "Пополнение расчётного счёта": {"requested_documents": ["Платёжное поручение", "Основание пополнения (договор/решение)"]},
    "Внесение наличных на счёт": {"requested_documents": ["Объявление на взнос наличными", "Приходный кассовый ордер"]},
    "Снятие наличных": {"requested_documents": ["Расходный кассовый ордер", "Чек банкомата / расходный ордер банка", "Авансовый отчёт (при наличии)"]},
    "Перевод на банковскую карту": {"requested_documents": ["Основание перевода (договор/приказ/иное)", "Платёжное поручение"]},
    "Расчёты через технологический участок (ТУ)": {"requested_documents": ["Договор / регламент расчётов через ТУ", "Реестр операций по ТУ", "Акт сверки расчётов"]},

    "Уплата по исполнительному листу": {"requested_documents": ["Исполнительный лист", "Судебный акт", "Постановление судебного пристава", "Платёжное поручение"]},
    "Возмещение ущерба": {"requested_documents": ["Акт о причинении ущерба", "Расчёт суммы ущерба", "Соглашение / судебный акт о возмещении", "Платёжное поручение"]},
    "Уплата государственной пошлины": {"requested_documents": ["Основание уплаты пошлины (заявление, иск и т.п.)", "Платёжное поручение"]},
    "Исполнение обязательства третьим лицом": {"requested_documents": ["Договор / основание обязательства", "Уведомление о возложении исполнения на третье лицо", "Платёжное поручение"]},

    "Комиссия банка за РКО": {"requested_documents": ["Договор банковского счёта / тарифы банка", "Выписка по счёту"]},
    "Плата за выдачу наличных": {"requested_documents": ["Тарифы банка", "Выписка по счёту"]},
    "Плата за перевод": {"requested_documents": ["Тарифы банка", "Платёжное поручение"]},
    "Открытие / ведение счёта": {"requested_documents": ["Договор банковского счёта", "Тарифы банка"]},
    "Вознаграждение арбитражного управляющего": {"requested_documents": ["Определение арбитражного суда об утверждении управляющего", "Расчёт вознаграждения", "Платёжное поручение"]},

    "Безвозмездная передача / дарение": {"requested_documents": ["Договор дарения", "Акт приёма-передачи"]},
    "Благотворительность": {"requested_documents": ["Договор пожертвования / основание благотворительного взноса", "Платёжное поручение"]},
    "Подарок": {"requested_documents": ["Основание передачи подарка", "Акт приёма-передачи (при наличии)"]},
    "Залог / обеспечительный платёж": {"requested_documents": ["Договор залога / соглашение об обеспечительном платеже", "Платёжное поручение"]},
    "Задаток / аванс": {"requested_documents": ["Договор / соглашение о задатке (авансе)", "Платёжное поручение"]},
    "Отступное": {"requested_documents": ["Соглашение об отступном", "Акт приёма-передачи (при передаче имущества)"]},
    "Новация": {"requested_documents": ["Соглашение о новации", "Первоначальный договор"]},
    "Прощение долга": {"requested_documents": ["Соглашение о прощении долга", "Акт сверки расчётов на дату прощения"]},
    "Техническая операция (корректировка, сторно)": {"requested_documents": ["Бухгалтерская справка", "Обоснование корректировки"]},
    "Текущий платёж (в процедуре банкротства)": {"requested_documents": ["Основание текущего платежа (договор/счёт)", "Платёжное поручение"]},
    "Реестровый платёж": {"requested_documents": ["Реестр требований кредиторов", "Определение арбитражного суда о включении в реестр", "Платёжное поручение"]},
    "Включение в реестр (взнос)": {"requested_documents": ["Заявление о включении в реестр требований кредиторов", "Определение арбитражного суда"]},
    "Расходы на процедуру банкротства": {"requested_documents": ["Смета расходов на процедуру банкротства", "Подтверждающие первичные документы", "Платёжное поручение"]},
}

ACTION_DOCS = {
    "return": ["Платёжное поручение о возврате"], "repayment": ["Кредитный договор", "График погашения"],
    "interest_payment": ["Расчёт процентов"], "tax_refund": ["Заявление о возврате налога", "Решение налогового органа"],
    "provision": ["Доказательства наличия средств у займодавца"],
    "purchase": ["Отчёт об оценке рыночной стоимости"], "sale": ["Отчёт об оценке рыночной стоимости"],
}
FLAG_DOCS = {
    "gov_procurement": ["Государственный (муниципальный) контракт", "Документация о закупке"],
    "related_party": ["Документы о структуре владения/связях сторон"],
    "advance": ["Документы о встречном предоставлении по авансу"],
    "price_suspicion": ["Отчёт об оценке рыночной стоимости"],
}
OUTFLOW_OVERLAY_DOC = "Реестр требований кредиторов (для проверки других кредиторов)"

def resolve_documents(operation_type, action_id, flags, direction):
    docs = list(OPERATION_MATRIX.get(operation_type, OPERATION_MATRIX["Неопознанный платёж"])["requested_documents"])
    def add(items):
        for d in items:
            if d not in docs:
                docs.append(d)
    if action_id:
        add(ACTION_DOCS.get(action_id, []))
    for f in (flags or []):
        add(FLAG_DOCS.get(f, []))
    if direction == "outflow" and operation_type not in ("Неопознанный платёж", "Неопознанный платёж (ссылка на документ)"):
        add([OUTFLOW_OVERLAY_DOC])
    return docs


In [ ]:
# ---- Раздел 5: уточнение "Неопознанного платежа" ----

_DOC_KEYWORDS = (
    r'\bдоговор\w*|\bконтракт\w*|\bсоглашени\w*|\bсч[её]т[-\s]?фактур\w*|'
    r'\bсч[её]т\w*|\bакт\b|\bакта\b|\bакту\b|\bупд\b|\bнакладн\w*|'
    r'\bторг[-\s]?12\b|\bспецификац\w*|\bприложени\w*|\bинвойс\w*|\bордер\w*'
)
DOC_REF_RE = re.compile(_DOC_KEYWORDS, re.IGNORECASE)
DOC_NUMBER_RE = re.compile(r'(?:№|N|#|номер\w*)\s*[-–]?\s*([0-9][0-9\-\/]*)', re.IGNORECASE)
DOC_NUMBER_AFTER_RE = re.compile(
    r'(?:договор\w*|контракт\w*|соглашени\w*|сч[её]т\w*|акт\w*|упд|накладн\w*|'
    r'торг[-\s]?12|спецификац\w*|приложени\w*|инвойс\w*|ордер\w*)'
    r'[\s№#nN-]*([0-9][0-9\-\/]*)', re.IGNORECASE)
DOC_DATE_RE = re.compile(r'от\s*([0-9]{1,2}[.,\-/][0-9]{1,2}[.,\-/][0-9]{2,4})')

_DOC_LABEL = {
    "договор": "договор", "контракт": "контракт", "соглашени": "соглашение",
    "счётфактур": "счёт-фактура", "счетфактур": "счёт-фактура", "счёт": "счёт", "счет": "счёт",
    "акт": "акт", "упд": "УПД", "накладн": "накладная", "торг": "ТОРГ-12",
    "спецификац": "спецификация", "приложени": "приложение", "инвойс": "инвойс", "ордер": "ордер",
}

def _doc_label(raw):
    key = re.sub(r'[-\s]', '', raw.lower())
    for stem, label in _DOC_LABEL.items():
        if key.startswith(stem):
            return label
    return raw.lower()

def refine_unknown(purpose):
    """Уточняет 'Неопознанный платёж' по тексту назначения. Возвращает operation_type,
    requested_documents и document_reference (структура для сопоставления с реестром договоров)."""
    m = DOC_REF_RE.search(purpose or "")
    if not m:
        return {
            "operation_type": "Неопознанный платёж",
            "requested_documents": list(OPERATION_MATRIX["Неопознанный платёж"]["requested_documents"]),
            "document_reference": None,
        }
    doc_type = _doc_label(m.group(0))
    number_m = DOC_NUMBER_RE.search(purpose) or DOC_NUMBER_AFTER_RE.search(purpose)
    date_m = DOC_DATE_RE.search(purpose)
    number = number_m.group(1) if number_m else None
    date = date_m.group(1) if date_m else None
    ref_str = doc_type
    if number:
        ref_str += f" №{number}"
    if date:
        ref_str += f" от {date}"
    return {
        "operation_type": "Неопознанный платёж (ссылка на документ)",
        "requested_documents": [f"Запросить документ, указанный в назначении: {ref_str}"]
            + OPERATION_MATRIX["Неопознанный платёж (ссылка на документ)"]["requested_documents"][1:],
        "document_reference": {"raw": purpose, "doc_type": doc_type, "number": number, "date": date},
    }


In [ ]:
def validate_op(subject_id: str | None, action_id: str | None, flags: list[str] | None) -> dict[str, Any]:
    """Проверяет subject_id/action_id/flags против закрытого каталога. Несуществующий
    subject_id -> forced unknown; action_id вне allowed_actions -> обнуляется."""
    flags = flags or []
    need_review = False

    if subject_id not in CATALOG_ACTIONS:
        subject_id, action_id = "unknown", None
        need_review = True
    else:
        allowed = CATALOG_ACTIONS[subject_id]
        if action_id is not None and action_id not in allowed:
            action_id = None
            need_review = True

    flags = [f for f in flags if f in ALL_FLAGS]
    return {"subject_id": subject_id, "action_id": action_id, "flags": flags, "need_review": need_review}


def build_operation_row(subject_id: str, action_id: str | None, flags: list[str], purpose: str, direction: str) -> dict[str, Any]:
    """Собирает предварительный (до проверки анализатором) тип + документы -- direction
    здесь уже в конвенции тетрадки ("outflow"/"inflow"), см. _to_catalog_direction."""
    op_type = resolve_operation_type(subject_id, action_id, flags)
    if op_type == "Неопознанный платёж":
        refined = refine_unknown(purpose)
        op_type = refined["operation_type"]
        docs = refined["requested_documents"]
    else:
        docs = resolve_documents(op_type, action_id, flags, direction)
    return {"operation_type": op_type, "requested_documents": docs}


In [ ]:
CLASSIFIER_PROMPT_TEMPLATE = '''Ты — модуль классификации банковских операций для проверки в процедуре банкротства и работе с проблемными активами.

ЗАДАЧА
По каждому назначению платежа и направлению средств определи:
- subject_id — предмет операции (за что фактически идут деньги);
- action_id — действие над предметом;
- direction_consistent — согласуется ли смысл назначения с фактическим направлением средств;
- flags — дополнительные признаки из закрытого списка (если явно есть в тексте);
- confidence — уверенность классификации.

Документы ты НЕ выдаёшь. Их определяет система по subject_id/action_id. Не упоминай документы.

ВХОД
JSON:
{{ "operations": [ {{ "idx": 0, "purpose": "", "direction": "outflow" }} ] }}
- operations — батч из одного кластера, используется только как техническая группировка.
  Кластер НЕ означает единый тип: каждый idx классифицируется отдельно по своему purpose и direction.
- idx — идентификатор операции, вернуть без изменений.
- purpose — назначение платежа, главный источник классификации.
- direction — "outflow" (списание со счёта должника, выбытие средств) или "inflow" (зачисление, поступление).

КАТАЛОГ (закрытый)
Ниже передаётся CATALOG. Разрешено использовать ТОЛЬКО id из него:
- subjects — предметы, сгруппированы по group_id; у каждого subject_id есть triggers (подсказки) и allowed_actions;
- actions — действия;
- flags — признаки.
Запрещено: придумывать новые id, менять id, использовать id не из CATALOG.

{catalog_json}

АЛГОРИТМ ПРИНЯТИЯ РЕШЕНИЯ (выполняй строго по шагам, не пропускай)

Шаг 1. Убери из текста служебный шум, который НИКОГДА не влияет на выбор subject_id:
   номер договора/счёта/акта/УПД, дату, "с учётом НДС"/"без НДС", "по счёту", "за период",
   "частичная/окончательная оплата". После этого шума может остаться либо (а) конкретное
   предметное слово, либо (б) названный ТИП договора, либо (в) ничего содержательного.

Шаг 2. Ищи ДВА типа сигналов, каждого из них по отдельности достаточно:
   (a) предметное слово/словосочетание — то, ЧТО покупается/оплачивается: товар, работа,
       конкретная услуга, аренда, заём и т.п. — сверяй с triggers subject_id в CATALOG;
   (b) НАЗВАННЫЙ ТИП договора — если в тексте указан именно вид договора по названию
       ("договор поставки", "договор подряда", "договор аренды", "договор займа",
       "договор лизинга", "договор хранения" и т.п.), это САМОДОСТАТОЧНЫЙ сигнал сам по себе,
       даже если больше никаких предметных слов в тексте нет. Название типа договора —
       это и есть предмет, потому что тип договора юридически определяет предмет.

Шаг 3. Если после шагов 1–2 не нашлось НИ (a), НИ (b) — то есть в тексте остался только
   родовой, неспецифичный "договор" БЕЗ названного типа ("договор №234", "по договору",
   "оплата по договору"), либо только "счёт"/"расчёты"/"перечисление"/"задолженность" без
   конкретики — subject_id="unknown". Это ожидаемый и частый корректный результат, а не
   провал классификации. НЕ пытайся угадать предмет по вероятности, статистике или по тому,
   какой тип "обычно" стоит за такими платежами в банковской практике.

Шаг 4. Голое слово "договор" САМО ПО СЕБЕ, без названного типа (шаг 2b) и без предметных
   слов (шаг 2a), НИКОГДА не даёт основание выбрать конкретный subject_id, включая
   "goods_supply". Наличие номера или даты договора никак не повышает уверенность в
   конкретном типе — это нейтральные метаданные, а не признак предмета.

Шаг 4.1. Упоминание ФИНАНСОВОГО ДОКУМЕНТА — "счёт", "счёт-фактура", "накладная", "УПД",
   "акт", "спецификация" — САМО ПО СЕБЕ тоже НЕ является признаком конкретного предмета,
   включая "goods_supply". Эти документы сопровождают операции ЛЮБОГО типа: товары, работы,
   услуги, — счёт-фактура выставляется при любой облагаемой НДС операции, а не только при
   поставке товара. Выбирай subject_id только если рядом ЕСТЬ отдельное предметное слово
   (шаг 2a) или названный тип договора (шаг 2b). Просто "оплата по счёту-фактуре №45" без
   слова о товаре/работе/услуге — unknown, а не "goods_supply".

Шаг 5. Не выбирай subject_id потому что он "стандартный", "самый частый" или встретился
   первым в списке. Каждая классификация должна опираться на конкретный текстовый признак
   из purpose (предметное слово или названный тип договора), который ты можешь явно указать
   сам себе. Если такого признака нет — это unknown, а не наиболее вероятный тип.

Шаг 6. Не занижай классификацию, если предметный признак есть, но сформулирован не
   дословно как в triggers. triggers — это примеры формулировок, а не исчерпывающий список.
   Явные синонимы и глагольные формы ОДНОГО И ТОГО ЖЕ понятия ("отгружен товар" = "поставка
   товара", "оказаны услуги" = "услуга оказана") считаются тем же предметом.

   ВАЖНАЯ ОГОВОРКА: общий КОРЕНЬ слова — это НЕ то же самое, что общий ПРЕДМЕТ. Слово
   "поставщик" (сторона договора, роль контрагента) и слово "поставка"/"поставки" (тип
   сделки) однокоренные, но означают разное: упоминание "поставщика" говорит только о том,
   КОМУ или ОТ КОГО платёж, а не о том, ЗА ЧТО он. "Задолженность перед поставщиком" или
   "оплата поставщику" БЕЗ отдельного упоминания товара или типа договора поставки —
   это unknown, а не "goods_supply". Всегда проверяй: слово описывает САМУ СДЕЛКУ
   (что покупается) или РОЛЬ СТОРОНЫ (кто участвует)? Только первое — валидный признак.

Шаг 7. Не поднимай общий термин до узкого типа, когда сам термин действительно общий:
   "оплата услуг" ≠ конкретная услуга (нет вида услуги — unknown);
   "оплата работ" ≠ строительный подряд (нет признаков стройки — unknown);
   "возврат долга" ≠ возврат займа (нет слова заём/займ — unknown);
   "компенсация" сама по себе ≠ трудовая выплата (нет признаков трудовых отношений — unknown).
   Это правило НЕ противоречит шагу 2b: "договор подряда" — назван тип, это подряд;
   просто "оплата работ" без типа договора и без слов о характере работ — unknown.

Шаг 8. Действие (action_id). Для большинства предметов: direction=outflow → "payment",
   direction=inflow → "receipt". Специальные действия — для семейств заём/кредит/налог/ЦБ.
   Если действие не определимо и не требуется — action_id=null.

Шаг 9. Направление как дизамбигуатор. Используй direction, когда purpose неоднозначен.
   Если смысл назначения подразумевает одно направление, а фактическое direction другое —
   direction_consistent=false (сигнал для проверки, не ошибка).

Шаг 10. Обрабатывай каждую операцию изолированно, как если бы она была единственной во
   входе. Не переноси тип с одного idx на другой и не используй соседние операции батча
   как доказательство типа текущей операции — batch есть только техническая группировка.

Шаг 11. Запрещено: оценивать риск, юридическая квалификация, вывод о подозрительности,
   возврат документов, добавление полей вне схемы.

ПРИМЕРЫ (обрати особое внимание на пары 1-2 — они внешне похожи, но результат разный;
и на примеры 3-5 — это частые ошибочные срабатывания, которых нужно избежать)

Пример 1 (родовой договор без типа и без предмета → unknown):
Вход:  {{"idx": 1, "purpose": "Оплата по договору №234 от 05.10.2008", "direction": "outflow"}}
Выход: {{"idx": 1, "group": "other", "subject_id": "unknown", "action_id": null, "direction_consistent": true, "flags": [], "confidence": "high"}}
(есть только номер и дата договора, тип договора не назван, предметных слов нет — это
корректный unknown, а не "Поставка товаров")

Пример 2 (тот же родовой шаблон, но назван ТИП договора → goods_supply):
Вход:  {{"idx": 2, "purpose": "Оплата по договору поставки №234 от 05.10.2008", "direction": "outflow"}}
Выход: {{"idx": 2, "group": "goods_material", "subject_id": "goods_supply", "action_id": "payment", "direction_consistent": true, "flags": [], "confidence": "high"}}
(единственное отличие от примера 1 — назван тип "поставки"; этого достаточно само по себе)

Пример 3 (упомянут только финансовый документ, без предмета → unknown, НЕ goods_supply):
Вход:  {{"idx": 3, "purpose": "Оплата по счёту-фактуре №45 от 12.02.2024", "direction": "outflow"}}
Выход: {{"idx": 3, "group": "other", "subject_id": "unknown", "action_id": null, "direction_consistent": true, "flags": [], "confidence": "high"}}
(счёт-фактура — документ по НДС, сопровождает любую операцию: товары, работы, услуги;
сам по себе он не говорит, что куплен именно товар — это unknown, а не "Поставка товаров")

Пример 4 ("поставщик" — роль контрагента, а не предмет сделки → unknown, НЕ goods_supply):
Вход:  {{"idx": 4, "purpose": "Погашение задолженности перед поставщиком", "direction": "outflow"}}
Выход: {{"idx": 4, "group": "other", "subject_id": "unknown", "action_id": null, "direction_consistent": true, "flags": [], "confidence": "high"}}
(слово "поставщиком" однокоренное с "поставка", но это роль стороны, а не указание, ЗА ЧТО
именно платёж; конкретный товар или тип договора поставки не назван — unknown)

Пример 5 (внесение наличных — своя, отдельная категория, НЕ goods_supply):
Вход:  {{"idx": 5, "purpose": "Взнос наличных денежных средств на расчётный счёт", "direction": "inflow"}}
Выход: {{"idx": 5, "group": "intragroup_cash", "subject_id": "cash_deposit", "action_id": "contribution", "direction_consistent": true, "flags": [], "confidence": "high"}}
(это кассовая операция внесения наличных, не имеет отношения к товарам)

Пример 6 (услуга без конкретного вида → unknown, а НЕ первая попавшаяся услуга):
Вход:  {{"idx": 6, "purpose": "Оплата услуг по договору №77", "direction": "outflow"}}
Выход: {{"idx": 6, "group": "other", "subject_id": "unknown", "action_id": null, "direction_consistent": true, "flags": [], "confidence": "high"}}
(вид услуги не назван, тип договора не назван — не консалтинг, не IT, не юридические, unknown)

Пример 7 (возврат займа — специальное действие внутри семейства):
Вход:  {{"idx": 7, "purpose": "Возврат займа по договору займа №5", "direction": "outflow"}}
Выход: {{"idx": 7, "group": "finance", "subject_id": "loan", "action_id": "return", "direction_consistent": true, "flags": [], "confidence": "high"}}

Пример 8 (НДС в цене услуги — второстепенное уточнение, не сам предмет и не налог в бюджет):
Вход:  {{"idx": 8, "purpose": "Оплата консультационных услуг с учётом НДС", "direction": "outflow"}}
Выход: {{"idx": 8, "group": "services", "subject_id": "consulting", "action_id": "payment", "direction_consistent": true, "flags": [], "confidence": "high"}}
(предмет — консультационные услуги, "с учётом НДС" уточняет цену, а не тип операции)

Пример 9 (предметное слово без буквального совпадения с "поставка" → всё же goods_supply):
Вход:  {{"idx": 9, "purpose": "Оплата за отгруженный товар согласно спецификации", "direction": "outflow"}}
Выход: {{"idx": 9, "group": "goods_material", "subject_id": "goods_supply", "action_id": "payment", "direction_consistent": true, "flags": [], "confidence": "high"}}
(нет слова "поставка", но есть "товар" и глагольная форма "отгруженный", описывающая саму
сделку, а не роль стороны — этого достаточно)

ФОРМАТ ВЫВОДА
Строго валидный JSON, ничего вне JSON:
{{
  "operations": [
    {{ "idx": 0, "group": "", "subject_id": "", "action_id": null, "direction_consistent": true, "flags": [], "confidence": "high" }}
  ]
}}

САМОПРОВЕРКА
1. Все входные idx возвращены ровно один раз, лишних нет.
2. group_id, subject_id, action_id, flags — только из CATALOG (action_id может быть null).
3. action_id входит в allowed_actions выбранного subject_id.
4. Для каждого subject_id ≠ "unknown" ты можешь явно назвать конкретное слово или названный
   тип договора из purpose, который стал основанием выбора. Если не можешь его назвать —
   исправь на "unknown".
5. Нет документов и полей вне схемы. JSON валиден.
'''


In [ ]:
ANALYZER_VERIFICATION_PROMPT_TEMPLATE = '''Ты — модуль финальной проверки классификации банковских операций для процедуры банкротства.

ЗАДАЧА
Классификатор уже определил по каждой операции предполагаемый тип: subject_id, action_id и
человекочитаемый operation_type. Твоя задача — независимо сверить этот предполагаемый тип с
текстом назначения платежа (purpose) и решить:
- "confirm" — предложенный тип соответствует смыслу назначения, менять не нужно;
- "correct" — предложенный тип НЕ соответствует смыслу назначения; выбери другой subject_id
  (и при необходимости action_id) из каталога, который подходит лучше, либо "unknown", если ни
  один конкретный тип не подтверждается текстом.

Ты не классифицируешь с нуля без основания — используй те же строгие правила, что и
классификатор:
- подтверждением типа может быть только конкретное предметное слово (что покупается/
  оплачивается) или НАЗВАННЫЙ тип договора в purpose — не логика "что обычно означает такой
  платёж";
- голый "договор"/"счёт"/"расчёты"/"перечисление" без конкретики — unknown, а не поставка;
- слово, описывающее РОЛЬ стороны ("поставщику", "арендодателю") — не то же самое, что предмет
  сделки; роль без предмета — unknown;
- упоминание финансового документа (счёт-фактура, УПД, накладная, акт) само по себе не признак
  конкретного предмета — эти документы сопровождают любые операции;
- не занижай тип, если признак выражен синонимом, а не дословно как в triggers каталога;
- не завышай общий термин ("оплата услуг", "оплата работ") до узкого типа без конкретики.

Если предложенный классификатором тип уже соответствует одному из этих правил и подтверждается
текстом purpose — confirm. Если предложенный тип явно основан на слове роли, родовом договоре
или документе без предмета (нарушение правил выше) — correct.

ВХОД
JSON:
{{
  "operations": [
    {{
      "idx": 0,
      "purpose": "",
      "direction": "outflow",
      "proposed_subject_id": "",
      "proposed_action_id": null,
      "proposed_operation_type": ""
    }}
  ]
}}
- operations — батч, техническая группировка, каждый idx проверяется независимо от соседних.

КАТАЛОГ (тот же закрытый каталог, что использовал классификатор)
Разрешено использовать ТОЛЬКО id из него для final_subject_id/final_action_id.

{catalog_json}

ФОРМАТ ВЫВОДА
Строго валидный JSON, ничего вне JSON:
{{
  "operations": [
    {{
      "idx": 0,
      "decision": "confirm",
      "final_subject_id": "",
      "final_action_id": null,
      "correction_reason": ""
    }}
  ]
}}
- Если decision="confirm": final_subject_id/final_action_id ДОЛЖНЫ совпадать с proposed_*,
  correction_reason — пустая строка "".
- Если decision="correct": final_subject_id обязателен и должен явно ОТЛИЧАТЬСЯ от
  proposed_subject_id (иначе это не коррекция, а confirm); correction_reason — краткое
  обоснование со ссылкой на конкретное слово из purpose.

САМОПРОВЕРКА
1. Все входные idx возвращены ровно один раз, лишних нет.
2. final_subject_id и final_action_id — только из CATALOG (action_id может быть null).
3. final_action_id входит в allowed_actions выбранного final_subject_id.
4. Если decision="correct", final_subject_id ≠ proposed_subject_id, и ты можешь явно назвать
   слово из purpose, на основании которого сделана коррекция.
5. Нет полей вне схемы. JSON валиден.
'''

CATALOG_JSON = json.dumps(CATALOG, ensure_ascii=False, indent=2)
CLASSIFIER_SYSTEM_PROMPT = CLASSIFIER_PROMPT_TEMPLATE.format(catalog_json=CATALOG_JSON)
ANALYZER_VERIFICATION_SYSTEM_PROMPT = ANALYZER_VERIFICATION_PROMPT_TEMPLATE.format(catalog_json=CATALOG_JSON)


In [ ]:
# Батч не более этого числа операций на один вызов LLM -- держит длину контекста
# предсказуемой независимо от размера кластера/выписки.
CLASSIFIER_MAX_OPS_PER_BATCH = 20


def _to_catalog_direction(direction_ru: Any) -> str:
    """Проект использует "убытие"/"прибытие" (см. колонку direction), классификатор -- "outflow"/"inflow"."""
    text = str(direction_ru or "").strip().lower()
    if text in ("прибытие", "inflow", "in"):
        return "inflow"
    return "outflow"


def _strip_code_fences(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\n", "", text)
        text = re.sub(r"\n```$", "", text)
    return text.strip()


def call_classifier_llm_for_batch(batch: list[dict], llm) -> list[dict]:
    """batch: [{idx, purpose, direction}]. Возвращает [{idx, subject_id, action_id, flags, ...}]."""
    payload = {"operations": batch}
    response = llm.invoke([
        SystemMessage(content=CLASSIFIER_SYSTEM_PROMPT),
        HumanMessage(content=json.dumps(payload, ensure_ascii=False)),
    ])
    parsed = json.loads(_strip_code_fences(response.content))
    ops = parsed.get("operations", [])
    returned_idx = {op.get("idx") for op in ops}
    expected_idx = {op["idx"] for op in batch}
    if returned_idx != expected_idx:
        raise ValueError(f"Классификатор вернул не все idx: ожидались {expected_idx}, получены {returned_idx}")
    return ops


def call_analyzer_verification_for_batch(batch: list[dict], llm) -> list[dict]:
    """batch: [{idx, purpose, direction, proposed_subject_id, proposed_action_id, proposed_operation_type}]."""
    payload = {"operations": batch}
    response = llm.invoke([
        SystemMessage(content=ANALYZER_VERIFICATION_SYSTEM_PROMPT),
        HumanMessage(content=json.dumps(payload, ensure_ascii=False)),
    ])
    parsed = json.loads(_strip_code_fences(response.content))
    ops = parsed.get("operations", [])
    returned_idx = {op.get("idx") for op in ops}
    expected_idx = {op["idx"] for op in batch}
    if returned_idx != expected_idx:
        raise ValueError(f"Анализатор вернул не все idx: ожидались {expected_idx}, получены {returned_idx}")
    return ops


In [ ]:
# ============================================================
# Запуск: классификатор -> анализатор-верификатор -> df
# ============================================================
# Работает на ВСЕЙ подготовленной выписке (df), включая шумовой кластер (-1) --
# на этом шаге кластер -1 еще не исключен (это происходит только в следующей
# секции, при построении df_sample/df_unique), поэтому "серая зона" тоже
# получает предварительный тип операции и базовый список документов.

_classifier_rows = df[["idx", "purpose", "direction"]].copy()
_classifier_rows["idx"] = _classifier_rows["idx"].astype(str)
_classifier_rows["_direction_catalog"] = _classifier_rows["direction"].apply(_to_catalog_direction)

_classifier_batches = [
    _classifier_rows.iloc[start:start + CLASSIFIER_MAX_OPS_PER_BATCH]
    for start in range(0, len(_classifier_rows), CLASSIFIER_MAX_OPS_PER_BATCH)
]
logger.info("Определение типа операции: %d операций, %d батчей", len(_classifier_rows), len(_classifier_batches))

_operation_type_results: list[dict] = []

for _batch_i, _batch_df in enumerate(tqdm(_classifier_batches, desc="Классификация типа операции"), start=1):
    _payload = [
        {"idx": row["idx"], "purpose": str(row["purpose"]), "direction": row["_direction_catalog"]}
        for _, row in _batch_df.iterrows()
    ]

    try:
        _raw_ops = call_classifier_llm_for_batch(_payload, llm)
    except Exception as exc:
        logger.warning("Батч классификатора %d/%d не разобран: %s", _batch_i, len(_classifier_batches), exc)
        _raw_ops = [{"idx": p["idx"], "subject_id": "unknown", "action_id": None, "flags": []} for p in _payload]

    _by_idx = {str(op.get("idx")): op for op in _raw_ops}
    _built = []
    for row in _payload:
        _raw_op = _by_idx.get(row["idx"]) or {"subject_id": "unknown", "action_id": None, "flags": []}
        _validated = validate_op(_raw_op.get("subject_id"), _raw_op.get("action_id"), _raw_op.get("flags"))
        _row_built = build_operation_row(
            _validated["subject_id"], _validated["action_id"], _validated["flags"],
            row["purpose"], row["direction"],
        )
        _built.append({
            **_row_built,
            "idx": row["idx"], "purpose": row["purpose"], "direction": row["direction"],
            "subject_id": _validated["subject_id"], "action_id": _validated["action_id"],
            "need_review": _validated["need_review"],
        })

    _verify_payload = [
        {"idx": b["idx"], "purpose": b["purpose"], "direction": b["direction"],
         "proposed_subject_id": b["subject_id"], "proposed_action_id": b["action_id"],
         "proposed_operation_type": b["operation_type"]}
        for b in _built
    ]
    try:
        _verification_ops = call_analyzer_verification_for_batch(_verify_payload, llm)
        _verification_by_idx = {str(op.get("idx")): op for op in _verification_ops}
    except Exception as exc:
        logger.warning("Батч анализатора %d/%d не разобран, тип подтверждается без проверки: %s", _batch_i, len(_classifier_batches), exc)
        _verification_by_idx = {}

    for b in _built:
        _v_op = _verification_by_idx.get(b["idx"])
        if _v_op is None:
            _operation_type_results.append({
                "idx": b["idx"], "operation_type": b["operation_type"],
                "requested_documents": b["requested_documents"],
                "classifier_operation_type": b["operation_type"],
                "analyzer_decision": "confirm", "analyzer_correction_reason": "",
                "operation_type_need_review": b["need_review"],
            })
            continue

        _decision = _v_op.get("decision", "confirm")
        if _decision != "correct":
            _operation_type_results.append({
                "idx": b["idx"], "operation_type": b["operation_type"],
                "requested_documents": b["requested_documents"],
                "classifier_operation_type": b["operation_type"],
                "analyzer_decision": "confirm", "analyzer_correction_reason": "",
                "operation_type_need_review": b["need_review"],
            })
            continue

        _final_subject_id = _v_op.get("final_subject_id") or b["subject_id"]
        _final_action_id = _v_op.get("final_action_id")
        _final_row = build_operation_row(_final_subject_id, _final_action_id, [], str(b["purpose"]), b["direction"])
        _operation_type_results.append({
            "idx": b["idx"],
            "operation_type": _final_row["operation_type"],
            "requested_documents": _final_row["requested_documents"],
            "classifier_operation_type": b["operation_type"],
            "analyzer_decision": "correct",
            "analyzer_correction_reason": _v_op.get("correction_reason") or "",
            "operation_type_need_review": True,
        })

operation_types_df = pd.DataFrame(_operation_type_results)
operation_types_df["idx"] = operation_types_df["idx"].astype(str)

df["idx"] = df["idx"].astype(str)
df = df.merge(operation_types_df, on="idx", how="left")

_n_corrected = int((operation_types_df["analyzer_decision"] == "correct").sum())
_n_unknown = int(operation_types_df["operation_type"].str.startswith("Неопознанный").sum())
logger.info(
    "Тип операции определен: %d операций, исправлений анализатора: %d, неопознанных: %d",
    len(operation_types_df), _n_corrected, _n_unknown,
)
operation_types_df["analyzer_decision"].value_counts()


In [ ]:
# ==========================================================
# Гибридная метрика сходства, плотность кластеров и
# адаптивный выбор представителей
# ==========================================================
# Единое метрическое пространство для трех задач: (1) определение плотности
# кластера, (2) выбор представителей, (3) распространение объяснений.
# Обоснование выбора признаков, весов и алгоритмов — в markdown выше.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------- Признаки и веса гибридного сходства ----------
# Текст назначения — главный носитель смысла операции (категория, аргументация).
# Числовые признаки — контекст аномальности/периода/материальности.
# Контрагентско-графовый блок — защита от переноса объяснений между операциями
# с одинаковым назначением, но разными группами контрагентов/связей.
SIMILARITY_TEXT_WEIGHT = 0.45
SIMILARITY_NUMERIC_WEIGHT = 0.25
SIMILARITY_CONTEXT_WEIGHT = 0.30

SIMILARITY_NUMERIC_FEATURES = ["anomaly_score", "anomaly_model_score", "interval_days", "amount_log"]
GRAPH_CONNECTIONS_COLUMN = "collect_all_graph_connections.description"

# ---------- Пороги уверенности сопоставления ----------
# high — объяснение распространяется автоматически;
# medium — распространяется с пометкой уровня уверенности;
# low — операция уходит в повторный LLM-анализ (см. секцию второго прохода).
PROPAGATION_CONFIDENCE_HIGH = 0.85
PROPAGATION_CONFIDENCE_LOW = 0.65


def resolve_counterparty_inn(row: pd.Series | dict) -> str | None:
    """
    Определяет ИНН контрагента операции.

    Допущение прототипа: знак суммы кодирует направление (отрицательная сумма —
    убытие средств у должника, контрагент — credit-сторона; положительная —
    прибытие, контрагент — debit-сторона). Если знак не информативен, приоритет
    отдается credit_inn. При появлении явной колонки direction правило нужно
    заменить на нее.
    """
    get = row.get if hasattr(row, "get") else row.__getitem__
    credit_inn = str(get("credit_inn") or "").strip()
    debit_inn = str(get("debit_inn") or "").strip()
    amount = get("amount")

    def _valid(inn: str) -> bool:
        return inn.isdigit() and len(inn) in (10, 12)

    if amount is not None and pd.notna(amount):
        if amount < 0 and _valid(credit_inn):
            return credit_inn
        if amount > 0 and _valid(debit_inn):
            return debit_inn

    if _valid(credit_inn):
        return credit_inn
    if _valid(debit_inn):
        return debit_inn
    return None


def _prepare_numeric_block(frame: pd.DataFrame) -> pd.DataFrame:
    """Готовит числовой блок признаков (включая лог-сжатую сумму)."""
    block = pd.DataFrame(index=frame.index)
    for col in ("anomaly_score", "anomaly_model_score", "interval_days"):
        block[col] = pd.to_numeric(frame.get(col), errors="coerce") if col in frame.columns else np.nan
    # Логарифм суммы: материальность важна, но линейная шкала дала бы сумме
    # доминировать над остальными признаками из-за тяжелых хвостов.
    block["amount_log"] = np.log1p(pd.to_numeric(frame.get("amount"), errors="coerce").abs())
    return block


def _text_similarity(A_texts: pd.Series, B_texts: pd.Series) -> np.ndarray:
    """
    Косинусное сходство текстов по TF-IDF символьных n-грамм (3–5).
    Символьные n-граммы устойчивы к морфологии, номерам договоров и вариациям
    шаблонных формулировок; векторизатор обучается на текстах пары выборок.
    """
    a = A_texts.fillna("").astype(str).str.lower().tolist()
    b = B_texts.fillna("").astype(str).str.lower().tolist()
    if not any(t.strip() for t in a + b):
        return np.full((len(a), len(b)), 0.5)
    # use_idf=False: сравнение идет внутри кластера на малых корпусах, где IDF
    # обнуляет вес общих n-грамм и занижает сходство почти одинаковых шаблонных
    # назначений ("зарплата за март" vs "за апрель"). L2-нормированные частоты
    # n-грамм дают честный косинус для шаблонных текстов.
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1, use_idf=False)
    try:
        matrix = vectorizer.fit_transform(a + b)
    except ValueError:
        return np.full((len(a), len(b)), 0.5)
    return cosine_similarity(matrix[: len(a)], matrix[len(a):])


# Границы нормализации фиксируются один раз по всей выписке (робастные квантили
# 1%/99%). Нормализация внутри каждой пары выборок растягивала бы крошечные
# различия плотного кластера на весь единичный куб и занижала сходство реально
# похожих операций.
_NUMERIC_BOUNDS: dict[str, tuple[float, float]] | None = None


def fit_similarity_space(reference_df: pd.DataFrame) -> None:
    """Фиксирует глобальные границы числовых признаков по всей выписке."""
    global _NUMERIC_BOUNDS
    block = _prepare_numeric_block(reference_df)
    bounds = {}
    for col in block.columns:
        low, high = block[col].quantile(0.01), block[col].quantile(0.99)
        if pd.isna(low) or pd.isna(high) or high <= low:
            low, high = 0.0, 1.0
        bounds[col] = (float(low), float(high))
    _NUMERIC_BOUNDS = bounds
    logger.info("Границы нормализации признаков сходства зафиксированы: %s", bounds)


def _scale_numeric(frame: pd.DataFrame) -> np.ndarray:
    if _NUMERIC_BOUNDS is None:
        raise RuntimeError("Сначала вызовите fit_similarity_space(df) по всей выписке.")
    block = _prepare_numeric_block(frame)
    scaled = pd.DataFrame(index=block.index)
    for col, (low, high) in _NUMERIC_BOUNDS.items():
        scaled[col] = ((block[col] - low) / (high - low)).clip(0.0, 1.0)
    scaled = scaled.fillna(scaled.median()).fillna(0.5)
    return scaled.to_numpy(dtype=float)


def _numeric_similarity(A_df: pd.DataFrame, B_df: pd.DataFrame) -> np.ndarray:
    """1 - нормированное евклидово расстояние в глобальной min-max шкале."""
    X_a = _scale_numeric(A_df)
    X_b = _scale_numeric(B_df)
    dists = np.sqrt(((X_a[:, None, :] - X_b[None, :, :]) ** 2).sum(axis=2))
    return 1.0 - dists / math.sqrt(X_a.shape[1])


def _connections_text(frame: pd.DataFrame) -> pd.Series:
    """Нормализует описание графовых связей к строке ('' — связей нет)."""
    if GRAPH_CONNECTIONS_COLUMN not in frame.columns:
        return pd.Series([""] * len(frame), index=frame.index)

    def norm(value):
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return ""
        if isinstance(value, (list, tuple, set)):
            return " ; ".join(map(str, value))
        return str(value)

    return frame[GRAPH_CONNECTIONS_COLUMN].map(norm)


def _context_similarity(A_df: pd.DataFrame, B_df: pd.DataFrame) -> np.ndarray:
    """
    Контрагентско-графовый блок сходства (веса поровну между двумя компонентами):

    1. Совпадение контрагента: 1.0 — тот же ИНН контрагента; 0.0 — разные;
       0.5 — контрагент не определен хотя бы у одной стороны (нейтрально).
    2. Совпадение картины связей: обе операции без переданных связей — 1.0
       (одинаково «чистые»); связи есть только у одной — 0.0 (разные группы);
       связи есть у обеих — TF-IDF-сходство описаний связей.
    """
    a_cp = [resolve_counterparty_inn(row) for _, row in A_df.iterrows()]
    b_cp = [resolve_counterparty_inn(row) for _, row in B_df.iterrows()]
    cp = np.empty((len(a_cp), len(b_cp)))
    for i, x in enumerate(a_cp):
        for j, y in enumerate(b_cp):
            if x is None or y is None:
                cp[i, j] = 0.5
            else:
                cp[i, j] = 1.0 if x == y else 0.0

    a_conn = _connections_text(A_df)
    b_conn = _connections_text(B_df)
    a_has = a_conn.str.strip().astype(bool).to_numpy()
    b_has = b_conn.str.strip().astype(bool).to_numpy()

    conn = np.zeros((len(a_conn), len(b_conn)))
    both_empty = (~a_has[:, None]) & (~b_has[None, :])
    conn[both_empty] = 1.0
    both_have = a_has[:, None] & b_has[None, :]
    if both_have.any():
        conn_text_sim = _text_similarity(a_conn, b_conn)
        conn[both_have] = conn_text_sim[both_have]
    # Ровно одна сторона со связями остается 0.0 — разные группы контрагентов.

    return 0.5 * cp + 0.5 * conn


def hybrid_similarity_matrix(A_df: pd.DataFrame, B_df: pd.DataFrame | None = None) -> np.ndarray:
    """
    Итоговая гибридная матрица сходства A x B (B = A, если не передан):
    0.45 * текст назначения + 0.25 * числовые признаки + 0.30 * контрагенты/связи.
    """
    if B_df is None:
        B_df = A_df
    sim = (
        SIMILARITY_TEXT_WEIGHT * _text_similarity(A_df["purpose"], B_df["purpose"])
        + SIMILARITY_NUMERIC_WEIGHT * _numeric_similarity(A_df, B_df)
        + SIMILARITY_CONTEXT_WEIGHT * _context_similarity(A_df, B_df)
    )
    return np.clip(sim, 0.0, 1.0)


# ==========================================================
# Плотность кластера
# ==========================================================

def cluster_density_profile(sim_matrix: np.ndarray) -> dict:
    """
    Профиль компактности кластера по попарной матрице сходства:

    - homogeneity: средняя попарная похожесть (внедиагональная) — прямой аналог
      среднего внутрикластерного расстояния, но в той же метрике, в которой
      затем распространяются объяснения;
    - isolated_share: доля операций, у которых даже НАИБОЛЕЕ похожая операция
      кластера ниже порога уверенности, — такие операции в принципе некому
      представлять, сколько представителей ни выбирай.
    """
    n = len(sim_matrix)
    if n <= 1:
        return {"n_operations": n, "homogeneity": 1.0, "isolated_share": 0.0}

    mask = ~np.eye(n, dtype=bool)
    homogeneity = float(sim_matrix[mask].mean())
    masked = np.where(mask, sim_matrix, -np.inf)
    best_neighbor = masked.max(axis=1)
    isolated_share = float((best_neighbor < PROPAGATION_CONFIDENCE_LOW).mean())
    return {"n_operations": n, "homogeneity": homogeneity, "isolated_share": isolated_share}


def classify_cluster_density(profile: dict) -> str:
    """
    Кластер разреженный, если средняя попарная похожесть ниже порога уверенности
    распространения (перенос объяснений между «средней парой» операций был бы
    ненадежным) ЛИБО заметная доля операций изолирована.
    """
    threshold = CONFIG.get("cluster_homogeneity_threshold") or PROPAGATION_CONFIDENCE_LOW
    max_isolated = CONFIG.get("max_isolated_share", 0.30)
    if profile["homogeneity"] < threshold or profile["isolated_share"] > max_isolated:
        return "sparse"
    return "compact"


# ==========================================================
# Адаптивный выбор представителей (coverage-FPS)
# ==========================================================

def adaptive_select_representatives(
    sim_matrix: np.ndarray,
    coverage_threshold: float,
    k_max: int,
) -> list[int]:
    """
    Farthest Point Sampling с критерием остановки по покрытию.

    FPS — жадная 2-аппроксимация задачи k-center: минимизировать максимальное
    расстояние от операции до ближайшего представителя. Это ровно наша цель:
    у каждой операции должен быть достаточно похожий представитель. Вместо
    фиксированного k выбор продолжается, пока худшая операция не покрыта
    представителем со сходством >= coverage_threshold (или не достигнут k_max).
    Размер выборки при этом сам зависит от размера, компактности и разнообразия
    кластера: плотному кластеру хватает 1–2 представителей, разнообразному
    потребуется больше.

    Старт — с самой атипичной операции (минимальная средняя похожесть):
    экстремальные операции покрываются в первую очередь, что важно для
    антифрод-задачи (аномалии не должны «представляться» типовыми операциями).
    """
    n = len(sim_matrix)
    if n == 0:
        return []
    if n == 1:
        return [0]

    selected = [int(np.argmin(sim_matrix.mean(axis=1)))]
    k_max = max(1, min(k_max, n))

    while len(selected) < k_max:
        coverage = sim_matrix[:, selected].max(axis=1)
        coverage[selected] = 1.0
        worst = int(np.argmin(coverage))
        if coverage[worst] >= coverage_threshold:
            break
        selected.append(worst)

    return selected


In [ ]:
# ==========================================================
# Применение адаптивной стратегии к кластерам
# ==========================================================
# Для каждого кластера (среди LLM-релевантных операций, |amount| >= порога):
# 1. считается профиль плотности;
# 2. разреженный кластер уходит в LLM целиком;
# 3. компактный кластер получает адаптивную выборку представителей.

# Границы нормализации фиксируются по всей выписке, чтобы плотность, выбор
# представителей и распространение работали в одной абсолютной шкале.
fit_similarity_space(df)

amount_threshold = CONFIG["min_abs_amount_for_llm"]

df_sample = (
    df[df["amount"].abs() >= amount_threshold]
    .copy()
    .reset_index(drop=True)
)

cluster_strategy: dict[Any, dict] = {}
selected_parts = []

for cluster_id, group in df_sample.groupby("cluster", sort=False):
    if cluster_id == -1:
        # Шумовой кластер не имеет смысловой общности: стратегия к нему не
        # применяется, операции исключаются как и раньше (ячейка ниже).
        continue

    sim = hybrid_similarity_matrix(group)
    profile = cluster_density_profile(sim)
    mode = classify_cluster_density(profile)

    if mode == "sparse":
        # Разреженный кластер: операции значимо различаются, распространение
        # объяснений ненадежно — в LLM передается весь кластер.
        chosen = group.copy()
        chosen["selection_mode"] = "full_sparse"
        min_coverage = 1.0
    else:
        idx_positions = adaptive_select_representatives(
            sim,
            coverage_threshold=PROPAGATION_CONFIDENCE_LOW,
            k_max=CONFIG["max_representatives_per_cluster"],
        )
        chosen = group.iloc[idx_positions].copy()
        chosen["selection_mode"] = "representative"
        # Фактическое покрытие: минимальная похожесть операции кластера на
        # ближайшего выбранного представителя.
        min_coverage = float(sim[:, idx_positions].max(axis=1).min())

    cluster_strategy[cluster_id] = {
        "mode": mode,
        "n_operations_llm_eligible": profile["n_operations"],
        "homogeneity": round(profile["homogeneity"], 4),
        "isolated_share": round(profile["isolated_share"], 4),
        "n_selected": len(chosen),
        "min_coverage_after_selection": round(min_coverage, 4),
    }
    selected_parts.append(chosen)

df_unique = pd.concat(selected_parts, ignore_index=True) if selected_parts else df_sample.iloc[0:0].copy()

# Диагностика решений по кластерам: распределение homogeneity позволяет
# осознанно скорректировать CONFIG["cluster_homogeneity_threshold"], если
# дефолт (порог уверенности распространения) не подходит вашим данным.
strategy_df = (
    pd.DataFrame.from_dict(cluster_strategy, orient="index")
    .rename_axis("cluster")
    .reset_index()
    .sort_values("homogeneity")
)
strategy_df.to_excel(OUTPUT_DIR / "cluster_density_diagnostics.xlsx", index=False)

logger.info(
    "Стратегия кластеров: всего=%d, компактных=%d, разреженных=%d; операций в LLM: %d из %d релевантных",
    len(cluster_strategy),
    sum(1 for s in cluster_strategy.values() if s["mode"] == "compact"),
    sum(1 for s in cluster_strategy.values() if s["mode"] == "sparse"),
    len(df_unique),
    len(df_sample),
)
logger.info(
    "Распределение homogeneity по кластерам: min=%.3f, медиана=%.3f, max=%.3f (порог=%s)",
    strategy_df["homogeneity"].min() if not strategy_df.empty else float("nan"),
    strategy_df["homogeneity"].median() if not strategy_df.empty else float("nan"),
    strategy_df["homogeneity"].max() if not strategy_df.empty else float("nan"),
    CONFIG.get("cluster_homogeneity_threshold") or PROPAGATION_CONFIDENCE_LOW,
)

strategy_df


In [ ]:
df_unique.to_excel(OUTPUT_DIR / "unique_operations_for_llm.xlsx", index=False)


In [ ]:
# Убираем шумовой кластер HDBSCAN/KMeans, если он присутствует.
transactions = df_unique[df_unique["cluster"] != -1].copy()


In [ ]:
# Финальный набор колонок, который передается агенту.
agent_columns = [
    "date",
    "credit_inn",
    "debit_inn",
    "purpose",
    "anomaly_model_score",
    "amount",
    "interval",
    "cluster",
    "anomaly_score",
    "idx",
    "court_filing_date",
    "collect_all_graph_connections.description",
    "credit_registration_date",
    "credit_status",
    "credit_okved",
    "credit_num_court_cases",
    "credit_sum_court_cases",
    "credit_loss_profit_amount",
    "credit_tax_arrears",
    "operation_type",
    "requested_documents",
]

available_agent_columns = [col for col in agent_columns if col in transactions.columns]
missing_agent_columns = sorted(set(agent_columns) - set(available_agent_columns))

if missing_agent_columns:
    logger.warning("Некоторые агентные колонки отсутствуют и будут пропущены: %s", missing_agent_columns)

transactions = transactions[available_agent_columns].copy()
transactions.head()


In [ ]:
# Опциональный фильтр для отладки. Для полного запуска оставьте DEBUG_CLUSTERS = None.
DEBUG_CLUSTERS: list[int] | None = None

if DEBUG_CLUSTERS is not None:
    transactions = transactions[transactions["cluster"].isin(DEBUG_CLUSTERS)].copy()

len(transactions)


In [ ]:
# Опциональное ограничение объема для быстрой проверки pipeline.
DEBUG_SAMPLE_SIZE: int | None = None

if DEBUG_SAMPLE_SIZE is not None and len(transactions) > DEBUG_SAMPLE_SIZE:
    transactions = transactions.sample(DEBUG_SAMPLE_SIZE, random_state=CONFIG["random_state"]).copy()

len(transactions)


In [ ]:
transactions["cluster"].unique()


## Запуск агентного анализа

Взаимодействие с агентом не изменилось: батч операций передается в тот же граф тем же
состоянием. Батчи формируются по стратегии кластера:

- компактный кластер → батч из его представителей;
- разреженный кластер → все LLM-релевантные операции (крупные кластеры режутся на части
  по `max_operations_per_llm_batch`, при этом каждая часть остается группой похожих по
  признакам операций того же кластера).

В лог пишутся старт/завершение каждого батча, количество операций, вызванные
инструменты, ошибки парсинга JSON и финальное количество результатов.


In [ ]:
# ==========================================================
# Запуск анализа по кластерам и экспорт результатов
# ==========================================================
# Взаимодействие с агентом не меняется: каждый батч (кластер представителей,
# разреженный кластер целиком или его часть) передается в тот же граф тем же
# состоянием. Логика запуска вынесена в функцию, потому что она нужна дважды:
# для основного прохода и для повторного анализа операций с низкой уверенностью.

def run_agent_on_operations(batch_label: Any, operations: list[dict]) -> dict:
    """Прогоняет один батч операций через граф агента и разбирает результат."""
    n_ops = len(operations)
    logger.info("Обработка батча %s: операций=%d", batch_label, n_ops)

    # Важно: содержимое messages не попадает в промпты (обе цепочки собраны с
    # partial(messages=[])). Операции передаются в LLM через state["operations"].
    initial_state = {
        "messages": [HumanMessage(content=f"Анализ батча {batch_label}: операций={n_ops}.")],
        "operations": operations,
        "remaining_steps": 3,
        "tool_to_check": [],
        "tool_results": [],
        "orchestrator_result": None,
        "analyzer_result": None,
        "final_json": None,
        "errors": [],
        "warnings": [],
    }

    try:
        result = graph.invoke(initial_state, config={"recursion_limit": 20})
        logger.info("Батч %s обработан", batch_label)

        analyzer_result = result.get("analyzer_result")
        final_json = result.get("final_json")

        # Основной источник результата — analyzer_result. final_json используется
        # как fallback на случай, если конкретная версия узла сохранила ответ туда.
        parsed_operations = []

        if analyzer_result and isinstance(analyzer_result, dict):
            operations_list = analyzer_result.get("operations", [])
            if isinstance(operations_list, list):
                parsed_operations.extend(op for op in operations_list if isinstance(op, dict))

        if not parsed_operations and final_json and isinstance(final_json, dict):
            operations_list = final_json.get("operations", [])
            if isinstance(operations_list, list):
                parsed_operations.extend(op for op in operations_list if isinstance(op, dict))

        logger.info("Батч %s: распарсено операций=%d", batch_label, len(parsed_operations))

        if parsed_operations:
            return {
                "cluster_id": batch_label,
                "parsed_analysis": parsed_operations,
                "raw_response": json.dumps(parsed_operations, ensure_ascii=False, default=str),
                "status": "success",
                "operations_count": len(parsed_operations),
            }
        logger.warning("Батч %s: не удалось извлечь операции из ответа анализатора", batch_label)
        return {
            "cluster_id": batch_label,
            "parsed_analysis": None,
            "raw_response": json.dumps(result, ensure_ascii=False, default=str),
            "status": "Не удалось извлечь результаты анализа",
            "operations_count": 0,
        }

    except Exception as exc:
        logger.exception("Ошибка при обработке батча %s", batch_label)
        return {
            "cluster_id": batch_label,
            "parsed_analysis": None,
            "raw_response": None,
            "status": f"Error: {str(exc)}",
            "operations_count": 0,
        }


def build_analysis_batches(transactions_df: pd.DataFrame) -> list[tuple[Any, list[dict]]]:
    """
    Формирует батчи для агента:

    - компактный кластер -> один батч из его представителей;
    - разреженный кластер -> все операции; крупные разреженные кластеры режутся
      на части по max_operations_per_llm_batch, чтобы не раздувать контекст и не
      снижать качество разбора отдельных операций. Кластерная логика оркестратора
      при этом сохраняется: часть кластера остается группой похожих по признакам
      операций того же кластера.
    """
    max_batch = CONFIG["max_operations_per_llm_batch"]
    batches: list[tuple[Any, list[dict]]] = []

    for cluster_id, cluster_df in transactions_df.groupby("cluster", sort=False):
        records = cluster_df.to_dict(orient="records")
        mode = cluster_strategy.get(cluster_id, {}).get("mode", "compact")

        if mode == "sparse" and len(records) > max_batch:
            n_parts = math.ceil(len(records) / max_batch)
            for part in range(n_parts):
                chunk = records[part * max_batch: (part + 1) * max_batch]
                batches.append((f"{cluster_id}_part{part + 1}", chunk))
        else:
            batches.append((cluster_id, records))

    return batches


analysis_batches = build_analysis_batches(transactions)
logger.info("Сформировано батчей для агента: %d", len(analysis_batches))

cluster_results = []
for batch_label, batch_records in tqdm(analysis_batches, desc="Батчи"):
    cluster_results.append(run_agent_on_operations(batch_label, batch_records))


logger.info("%s", "=" * 80)
logger.info("Экспорт результатов в Excel")
logger.info("%s", "=" * 80)

flat_data = []

for item in cluster_results:
    cluster_id = item.get("cluster_id")
    parsed_analysis = item.get("parsed_analysis")
    status = item.get("status")

    logger.info(
        "Кластер %s: статус=%s, операций=%d",
        cluster_id,
        status,
        len(parsed_analysis) if parsed_analysis else 0,
    )

    if isinstance(parsed_analysis, list) and parsed_analysis:
        for tx in parsed_analysis:
            if isinstance(tx, dict):
                row = tx.copy()
                row["cluster_id"] = cluster_id
                row["status"] = status
                flat_data.append(row)
                logger.debug("Добавлена операция idx=%s", tx.get("idx", "N/A"))
            else:
                logger.warning("Неожиданный тип операции в parsed_analysis: %s", type(tx))
    else:
        raw_response = item.get("raw_response")
        flat_data.append(
            {
                "cluster_id": cluster_id,
                "status": status,
                "error": str(raw_response or "Нет распознанных данных")[:500],
            }
        )

logger.info("Всего строк для экспорта: %d", len(flat_data))

df_all = pd.DataFrame(flat_data)


def stringify_complex_values(value):
    """Преобразует списки и словари в JSON-строки для корректной записи в Excel."""
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False, default=str)
    return value


if not df_all.empty:
    df_all = df_all.map(stringify_complex_values)

    # Базовые поля ставятся в начало отчета. Остальные поля не теряются,
    # а добавляются после основного набора.
    cols_order = [
        "cluster_id",
        "idx",
        "transaction_category",
        "amount",
        "counterparty_category",
        "risk_level",
        "legal_qualification",
        "decision_argumentation",
        "risk_explanation",
        "recommendation",
        "legal_basis",
        "used_tools",
        "status",
        "error",
    ]

    existing_cols = [col for col in cols_order if col in df_all.columns]
    other_cols = [col for col in df_all.columns if col not in cols_order]
    df_all = df_all[existing_cols + other_cols]

    raw_report_path = OUTPUT_DIR / "cluster_analysis_raw.xlsx"
    df_all.to_excel(raw_report_path, index=False)
    logger.info("Сырой отчет сохранен: %s", raw_report_path)

    def color_row_by_risk(row):
        """Возвращает CSS-стиль строки на основе risk_level."""
        try:
            value = row.get("risk_level", -1)
            lvl = -1 if value is None or value == "" else int(float(value))
        except (ValueError, TypeError):
            lvl = -1

        if lvl in (0, 1):
            bg = "background-color: #d4edda"  # зеленый
        elif lvl == 2:
            bg = "background-color: #fff3cd"  # желтый
        elif lvl == 3:
            bg = "background-color: #f8d7da"  # красный
        else:
            bg = ""

        return [bg] * len(row)

    formatted_report_path = OUTPUT_DIR / "cluster_analysis_formatted.xlsx"
    styled_df = df_all.style.apply(color_row_by_risk, axis=1)
    styled_df.to_excel(formatted_report_path, index=False, engine="openpyxl")
    logger.info("Форматированный отчет сохранен: %s", formatted_report_path)

    summary = {
        "clusters_total": len(cluster_results),
        "clusters_success": sum(1 for item in cluster_results if item["status"] == "success"),
        "clusters_failed": sum(1 for item in cluster_results if item["status"] != "success"),
        "rows_exported": len(flat_data),
        "raw_report": str(raw_report_path),
        "formatted_report": str(formatted_report_path),
    }
    logger.info("Итог запуска агента: %s", summary)
    summary

else:
    logger.warning("Нет данных для экспорта")


## Распространение результатов и контроль качества

Финальный pipeline для каждого кластера:

```
Кластер → плотность → разреженный? → да: весь кластер в LLM
                                   → нет: представители → LLM
→ поиск наиболее похожего представителя (та же гибридная метрика)
→ распространение объяснений → confidence
→ confidence low? → да: автоматический повторный LLM-анализ
                  → нет: полностью заполненная выписка
```

Уровни уверенности сопоставления: **high** (≥ 0.85) — объяснение распространяется
автоматически; **medium** (0.65–0.85) — распространяется с пометкой уровня уверенности;
**low** (< 0.65) — операция **не получает** объяснение ближайшего представителя, а
автоматически уходит на повторный анализ LLM (следующая секция). Дополнительная страховка
поверх метрики: если контрагент операции не совпадает с контрагентом представителя, а
риск представителя ≥ 2, уверенность ограничивается уровнем medium — выводы о контрагенте
не переносятся на другого контрагента автоматически.

Технические поля результата: `analysis_source` (llm / llm_second_pass / propagated /
none), `matched_representative_idx`, `similarity_score`, `propagation_confidence`,
`propagation_status`, `propagation_note`, `needs_review`.


In [ ]:
# ==========================================================
# Распространение результатов анализа на весь кластер
# ==========================================================
# Использует ту же гибридную метрику, что определение плотности и выбор
# представителей (см. секцию адаптивной стратегии). Экспорт полной выписки
# выполняется ниже — после повторного анализа операций с низкой уверенностью.

# Поля анализа, которые переносятся с представителя на похожие операции.
PROPAGATED_ANALYSIS_FIELDS = [
    "risk_level",
    "transaction_category",
    "counterparty_category",
    "legal_qualification",
    "legal_basis",
    "court_basis",
    "decision_argumentation",
    "risk_explanation",
    "recommendation",
    "challenge_criteria",
    "connections_basis",
    "used_tools",
]


def _confidence_label(similarity: float) -> str:
    if similarity >= PROPAGATION_CONFIDENCE_HIGH:
        return "high"
    if similarity >= PROPAGATION_CONFIDENCE_LOW:
        return "medium"
    return "low"


def _analysis_row(op: pd.Series, analysis: dict | pd.Series, **tech_fields) -> dict:
    row = {**op.to_dict()}
    for field in PROPAGATED_ANALYSIS_FIELDS:
        if field in analysis:
            row[field] = analysis[field]
    row.update(tech_fields)
    return row


def propagate_cluster_analysis(full_df: pd.DataFrame, analyzed_df: pd.DataFrame) -> pd.DataFrame:
    """
    Заполняет полную выписку результатами анализа.

    Для каждой операции:
    - проанализирована LLM напрямую (представитель или операция разреженного
      кластера) -> результат берется как есть, analysis_source = "llm";
    - компактный кластер -> k-NN (k=1) к представителям кластера по гибридной
      метрике, перенос анализа + confidence;
    - разреженный кластер, операция не анализировалась (ниже порога суммы) ->
      распространение запрещено по построению стратегии, операция помечается;
    - шумовой кластер / кластер без представителей -> помечается на проверку.
    """
    analyzed_map = {
        str(row["idx"]): row
        for _, row in analyzed_df.iterrows()
        if pd.notna(row.get("idx"))
    }
    logger.info("Распространение: операций с готовым LLM-анализом: %d", len(analyzed_map))

    out_rows = []

    for cluster_id, ops in full_df.groupby("cluster", sort=False):
        ops = ops.copy()
        strategy = cluster_strategy.get(cluster_id, {})
        mode = strategy.get("mode")

        analyzed_mask = ops["idx"].astype(str).isin(analyzed_map)
        analyzed_ops = ops[analyzed_mask]

        # 1. Операции, проанализированные LLM напрямую.
        for _, op in analyzed_ops.iterrows():
            analysis = analyzed_map[str(op["idx"])]
            out_rows.append(_analysis_row(
                op, analysis,
                analysis_source="llm",
                matched_representative_idx=op["idx"],
                similarity_score=1.0,
                propagation_confidence="high",
                propagation_status="analyzed_by_llm",
                needs_review=False,
            ))

        rest = ops[~analyzed_mask]
        if rest.empty:
            continue

        # 2. Кластеры без базы для распространения.
        if cluster_id == -1 or analyzed_ops.empty:
            status = "noise_cluster" if cluster_id == -1 else "no_representative"
            for _, op in rest.iterrows():
                out_rows.append({
                    **op.to_dict(),
                    "analysis_source": "none",
                    "matched_representative_idx": None,
                    "similarity_score": None,
                    "propagation_confidence": None,
                    "propagation_status": status,
                    "needs_review": True,
                })
            continue

        # 3. Разреженный кластер: все LLM-релевантные операции уже проанализированы,
        # оставшиеся — ниже порога суммы. Распространять на них нечего и незачем:
        # внутри разреженного кластера операции значимо различаются.
        if mode == "sparse":
            for _, op in rest.iterrows():
                out_rows.append({
                    **op.to_dict(),
                    "analysis_source": "none",
                    "matched_representative_idx": None,
                    "similarity_score": None,
                    "propagation_confidence": None,
                    "propagation_status": "sparse_below_threshold",
                    "needs_review": True,
                })
            continue

        # 4. Компактный кластер: k-NN к представителям по гибридной метрике.
        combined = hybrid_similarity_matrix(rest, analyzed_ops)
        best_pos = combined.argmax(axis=1)
        best_sim = combined.max(axis=1)
        rep_records = analyzed_ops.to_dict(orient="records")

        for (_, op), pos, sim in zip(rest.iterrows(), best_pos, best_sim):
            rep_op = rep_records[int(pos)]
            rep_idx = str(rep_op["idx"])
            analysis = analyzed_map[rep_idx]
            confidence = _confidence_label(float(sim))

            # Страховка поверх метрики: несовпадение контрагента при материальном
            # риске представителя не позволяет уверенности быть high — выводы о
            # контрагенте не переносятся на другого контрагента автоматически.
            same_cp = resolve_counterparty_inn(op) == resolve_counterparty_inn(pd.Series(rep_op))
            try:
                rep_risk = int(float(analysis.get("risk_level")))
            except (TypeError, ValueError):
                rep_risk = None
            if not same_cp and rep_risk is not None and rep_risk >= 2 and confidence == "high":
                confidence = "medium"

            out_rows.append(_analysis_row(
                op, analysis,
                analysis_source="propagated",
                matched_representative_idx=rep_idx,
                similarity_score=round(float(sim), 4),
                propagation_confidence=confidence,
                propagation_status="propagated",
                propagation_note=(
                    f"Анализ перенесен с наиболее похожей операции-представителя idx={rep_idx} "
                    f"(сходство {float(sim):.2f}). Аргументация описывает представителя и требует "
                    "сверки с деталями текущей операции."
                ),
                needs_review=confidence == "low",
            ))

    result = pd.DataFrame(out_rows)
    logger.info(
        "Распространение завершено: всего=%d, llm=%d, propagated=%d, без анализа=%d, низкая уверенность=%d",
        len(result),
        int((result["analysis_source"] == "llm").sum()),
        int((result["analysis_source"] == "propagated").sum()),
        int((result["analysis_source"] == "none").sum()),
        int((result["propagation_confidence"] == "low").sum()),
    )
    return result


analyzed_representatives = df_all[df_all.get("status", pd.Series(dtype=str)) == "success"] \
    if "status" in df_all.columns else df_all
analyzed_representatives = analyzed_representatives.dropna(subset=["idx"]) \
    if "idx" in analyzed_representatives.columns else analyzed_representatives

full_statement = df.copy()
full_statement["is_below_llm_threshold"] = full_statement["amount"].abs() < CONFIG["min_abs_amount_for_llm"]

full_analysis = propagate_cluster_analysis(full_statement, analyzed_representatives)
full_analysis[["analysis_source", "propagation_confidence", "idx"]].groupby(
    ["analysis_source", "propagation_confidence"], dropna=False
).count()


## Повторный LLM-анализ операций с низкой уверенностью

Операции с `propagation_confidence = low` автоматически передаются агенту на второй
проход тем же механизмом запуска (та же функция, тот же граф, те же промпты). После
успешного разбора операция получает `analysis_source = "llm_second_pass"` и снимается с
ручной проверки. В `low_confidence_review.xlsx` остаются только операции, для которых
LLM-анализ невозможен или не удался: ниже порога материальности, шумовой кластер,
кластер без представителей, ошибки парсинга.


In [ ]:
# ==========================================================
# Повторный LLM-анализ операций с низкой уверенностью
# ==========================================================
# Операция с confidence = low НЕ получает объяснение ближайшего представителя:
# она автоматически уходит на второй проход агента. В повторный анализ попадают
# только LLM-релевантные операции (|amount| >= порога) — операции ниже порога
# материальности остаются помеченными на ручную проверку, чтобы фильтр стоимости
# не обходился через второй проход.

second_pass_mask = (
    (full_analysis["needs_review"] == True)  # noqa: E712
    & (full_analysis["analysis_source"] == "propagated")
    & (~full_analysis["is_below_llm_threshold"])
)
second_pass_ops = full_analysis[second_pass_mask].copy()
logger.info("Операций на повторный LLM-анализ: %d", len(second_pass_ops))

if not second_pass_ops.empty:
    # В агент передается тот же набор колонок, что и в основном проходе.
    second_pass_input = second_pass_ops[available_agent_columns].copy()

    second_pass_batches = []
    max_batch = CONFIG["max_operations_per_llm_batch"]
    for cluster_id, group in second_pass_input.groupby("cluster", sort=False):
        records = group.to_dict(orient="records")
        n_parts = math.ceil(len(records) / max_batch)
        for part in range(n_parts):
            chunk = records[part * max_batch: (part + 1) * max_batch]
            label = f"{cluster_id}_second_pass" + (f"_part{part + 1}" if n_parts > 1 else "")
            second_pass_batches.append((label, chunk))

    second_pass_results = []
    for batch_label, batch_records in tqdm(second_pass_batches, desc="Второй проход"):
        second_pass_results.append(run_agent_on_operations(batch_label, batch_records))

    # Слияние результатов второго прохода в полную выписку по idx.
    second_pass_map: dict[str, dict] = {}
    for item in second_pass_results:
        for op_analysis in item.get("parsed_analysis") or []:
            op_idx = str(op_analysis.get("idx", ""))
            if op_idx:
                second_pass_map[op_idx] = op_analysis

    # Страховка: колонки анализа могли не появиться, если основной проход не дал
    # успешных результатов — создаем их перед точечными присваиваниями.
    for field in PROPAGATED_ANALYSIS_FIELDS:
        if field not in full_analysis.columns:
            full_analysis[field] = None

    updated = 0
    for row_pos in full_analysis.index[second_pass_mask]:
        op_idx = str(full_analysis.at[row_pos, "idx"])
        analysis = second_pass_map.get(op_idx)
        if analysis is None:
            continue  # анализ не распарсился — операция остается в needs_review
        for field in PROPAGATED_ANALYSIS_FIELDS:
            if field in analysis:
                full_analysis.at[row_pos, field] = analysis[field] \
                    if not isinstance(analysis[field], (dict, list)) \
                    else json.dumps(analysis[field], ensure_ascii=False, default=str)
        full_analysis.at[row_pos, "analysis_source"] = "llm_second_pass"
        full_analysis.at[row_pos, "propagation_status"] = "reanalyzed_by_llm"
        full_analysis.at[row_pos, "propagation_confidence"] = "high"
        full_analysis.at[row_pos, "similarity_score"] = 1.0
        full_analysis.at[row_pos, "matched_representative_idx"] = op_idx
        full_analysis.at[row_pos, "needs_review"] = False
        updated += 1

    logger.info("Второй проход: обновлено операций=%d из %d", updated, int(second_pass_mask.sum()))


# ==========================================================
# Финальная сборка и экспорт полной выписки
# ==========================================================

full_analysis_export = full_analysis.map(stringify_complex_values)
full_report_path = OUTPUT_DIR / "full_statement_analysis.xlsx"
full_analysis_export.to_excel(full_report_path, index=False)
logger.info("Полная выписка с анализом сохранена: %s", full_report_path)

review_queue = full_analysis[full_analysis["needs_review"] == True].copy()  # noqa: E712
if not review_queue.empty:
    review_path = OUTPUT_DIR / "low_confidence_review.xlsx"
    review_queue.map(stringify_complex_values).to_excel(review_path, index=False)
    logger.warning("Операций, оставшихся на ручной проверке: %d. Сохранено: %s",
                   len(review_queue), review_path)

full_analysis.groupby(["analysis_source", "propagation_confidence"], dropna=False)["idx"].count()


## Обновление исторической памяти риска

Завершающий шаг пайплайна: подтвержденные результаты записываются в `RiskMemoryDB`,
чтобы следующий запуск агента видел накопленную историю через `get_conterparty_risk`.

Правила:
- запись — детерминированный шаг после анализа, а не LLM-tool;
- в память идут только операции с итоговым `risk_level` 2–3, проанализированные LLM
  напрямую (`analysis_source` = `llm` или `llm_second_pass`) — распространенные копии
  одного решения не должны умножать исторический риск через noisy-OR;
- повторный запуск по тем же данным обновляет записи (ключ
  `inn + court_filing_date + operation_idx`), одновременно освежая recency событий.


In [ ]:
# ==========================================================
# Автоматическое обновление исторической памяти риска
# ==========================================================

# resolve_counterparty_inn определен в секции гибридной метрики выше.

def update_risk_memory_from_results(
    results_df: pd.DataFrame,
    source_df: pd.DataFrame,
    court_filing_date: str,
) -> dict:
    """
    Обновляет историческую память риска по итогам завершенного анализа.

    Правила:
    - записываются только операции с итоговым risk_level 2 или 3;
    - записываются только операции, проанализированные LLM напрямую
      (analysis_source in {"llm", "llm_second_pass"}): распространенные результаты —
      производные, их запись задваивала бы одно и то же LLM-решение по числу похожих
      операций и искусственно раздувала исторический риск (для noisy-OR это критично);
    - обоснование (reason) берется из risk_explanation (fallback —
      legal_qualification);
    - дедупликация и обновление при повторном анализе обеспечиваются на уровне
      RiskMemoryDB через ключ (inn, court_filing_date, operation_idx).
    """
    stats = {"written": 0, "skipped_no_inn": 0, "skipped_low_risk": 0, "skipped_not_llm": 0, "errors": 0}

    date_by_idx = source_df.set_index(source_df["idx"].astype(str))["date"] if "idx" in source_df.columns else pd.Series(dtype=object)

    for _, row in results_df.iterrows():
        if row.get("analysis_source", "llm") not in ("llm", "llm_second_pass"):
            stats["skipped_not_llm"] += 1
            continue

        try:
            risk_level = int(float(row.get("risk_level")))
        except (TypeError, ValueError):
            stats["skipped_low_risk"] += 1
            continue
        if risk_level not in (2, 3):
            stats["skipped_low_risk"] += 1
            continue

        inn = resolve_counterparty_inn(row)
        if inn is None:
            stats["skipped_no_inn"] += 1
            continue

        op_idx = str(row.get("idx") or "")
        op_date = row.get("date")
        if (op_date is None or pd.isna(op_date)) and op_idx in date_by_idx.index:
            op_date = date_by_idx.loc[op_idx]
        if op_date is None or pd.isna(op_date):
            op_date = court_filing_date
        operation_date = pd.to_datetime(op_date).date().isoformat()

        reason = str(row.get("risk_explanation") or row.get("legal_qualification") or "")[:500]

        try:
            update_counterparty_risk_memory(
                inn=inn,
                risk_level=risk_level,
                operation_date=operation_date,
                reason=reason,
                court_filing_date=court_filing_date,
                operation_idx=op_idx,
            )
            stats["written"] += 1
        except Exception:
            logger.exception("Не удалось записать риск-событие: inn=%s, idx=%s", inn, op_idx)
            stats["errors"] += 1

    logger.info("Обновление памяти риска завершено: %s", stats)
    return stats


memory_update_stats = update_risk_memory_from_results(
    results_df=full_analysis,
    source_df=df,
    court_filing_date=CONFIG["court_filing_date"],
)
memory_update_stats


In [ ]:
# Служебная проверка итогового списка колонок.
transactions.columns
